# VIRCON-Classify — IJIES Continuation: Full Strengthening Run

This notebook **continues from** `VIRCON_IJIES_Reviewer_Revision_Artifacts.zip`. It does not replace the completed reviewer rerun.

It adds the remaining high-value evidence:

1. a **fine-tuned DistilBERT** contemporary baseline under the same fixed-split and event-disjoint protocols;
2. **external HumAID transfer validation** with exact-overlap removal and event-wise reporting;
3. **confidence/abstention (selective prediction)** analysis without claiming human-workflow validation;
4. **event-level domain-shift diagnostics** (lexical Jensen–Shannon divergence and OOV token mass versus event performance);
5. a **two-rater manual triage audit template** plus agreement/sensitivity analysis when annotations are completed;
6. reviewer-facing tables, predictions, checkpoints, hashes, and a final continuation artifact ZIP.

### Important scientific boundaries
- No experiment can guarantee acceptance.
- The existing CrisisBench event-disjoint results remain a **same-corpus event-shift** test.
- HumAID is used as a separate external dataset for transfer evaluation; exact normalized text overlap with CrisisBench is checked and removed.
- Human/practitioner validation is **not fabricated**. The audit analysis runs only after two raters fill the provided template.

### Runtime
For the full transformer run, use a Colab **GPU runtime**. T4/L4/A100 is suitable. The notebook checkpoints every fold, so rerunning can resume completed work.

In [ ]:
# 1. Mount Google Drive and enter the existing VIRCON folder.
from pathlib import Path
import os, sys, json, zipfile, hashlib, platform, subprocess, time, random, math, re, html, warnings, gc

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped or already available:', exc)

WORKDIR = Path('/content/drive/MyDrive/VirconPaper1')
if not WORKDIR.is_dir():
    raise FileNotFoundError(f'Expected Drive folder not found: {WORKDIR}')
os.chdir(WORKDIR)
print('Working directory:', Path.cwd())

TRAIN_PATH = WORKDIR / 'crisis_consolidated_humanitarian_filtered_lang_en_train.tsv'
TEST_PATH = WORKDIR / 'crisis_consolidated_humanitarian_filtered_lang_en_test.tsv'
PRIOR_ZIP = WORKDIR / 'VIRCON_IJIES_Reviewer_Revision_Artifacts.zip'
for p in (TRAIN_PATH, TEST_PATH, PRIOR_ZIP):
    if not p.is_file():
        raise FileNotFoundError(f'Missing required continuation input: {p}')
    print('FOUND:', p.name, p.stat().st_size, 'bytes')


In [ ]:
# 2. Colab-safe dependency check. Do not upgrade NumPy/Pandas/PyTorch.
import importlib.util

for mod, spec in [
    ('transformers', 'transformers>=4.45,<5'),
    ('requests', 'requests>=2.31'),
]:
    if importlib.util.find_spec(mod) is None:
        print('Installing missing package:', spec)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', spec])
    else:
        print(mod, 'already available')

import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import requests
import joblib
from scipy import sparse
from scipy.spatial.distance import jensenshannon
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support, log_loss
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Full continuation requires a GPU. In Colab choose Runtime -> Change runtime type -> GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 3. Configuration and reproducibility.
SEED = 20261001
MODEL_ID = 'distilbert-base-uncased'
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 128
FIXED_EPOCHS = 2
EVENT_EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0
RUN_EVENT_DIRECT_TRIAGE = True   # full reviewer-facing run
RUN_EXTERNAL_HUMAID = True
N_BOOTSTRAP = 2000

OUT_ROOT = WORKDIR / 'artifacts_ijies_continuation_strengthening'
TABLE_DIR = OUT_ROOT / 'tables'
REPORT_DIR = OUT_ROOT / 'reports'
MODEL_DIR = OUT_ROOT / 'models'
CHECKPOINT_DIR = OUT_ROOT / 'checkpoints'
for d in (OUT_ROOT, TABLE_DIR, REPORT_DIR, MODEL_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

CONFIG = {
    'seed': SEED, 'model_id': MODEL_ID, 'max_length': MAX_LENGTH,
    'train_batch_size': TRAIN_BATCH_SIZE, 'eval_batch_size': EVAL_BATCH_SIZE,
    'fixed_epochs': FIXED_EPOCHS, 'event_epochs': EVENT_EPOCHS,
    'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO, 'gradient_clip': GRAD_CLIP,
    'event_direct_triage': RUN_EVENT_DIRECT_TRIAGE,
    'external_humaid': RUN_EXTERNAL_HUMAID, 'n_bootstrap': N_BOOTSTRAP,
    'device_training': 'cuda', 'deployment_benchmark_device': 'cpu',
}
(REPORT_DIR / 'continuation_config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print(json.dumps(CONFIG, indent=2))


In [ ]:
# 4. Restore the completed reviewer-rerun artifact bundle.
PRIOR_EXTRACT = WORKDIR / '_vircon_prior_reviewer_artifacts'
if not PRIOR_EXTRACT.exists():
    PRIOR_EXTRACT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PRIOR_ZIP, 'r') as z:
        bad = z.testzip()
        if bad is not None:
            raise zipfile.BadZipFile(f'Corrupt member in prior artifact ZIP: {bad}')
        z.extractall(PRIOR_EXTRACT)

PRIOR_ROOT = PRIOR_EXTRACT / 'artifacts_ijies_reviewer_revision'
PRIOR_TABLE = PRIOR_ROOT / 'tables'
required_prior = [
    'fixed_split_predictions.csv',
    'event_disjoint_oof_predictions.csv',
    'event_disjoint_fold_assignment.csv',
    'complete_sample_accounting.csv',
    'triage_manual_audit_template.csv',
]
for name in required_prior:
    p = PRIOR_TABLE / name
    if not p.is_file():
        raise FileNotFoundError(f'Prior artifact missing: {p}')
print('Prior reviewer artifacts restored:', PRIOR_ROOT)


In [ ]:
# 5. Reconstruct the exact prepared CrisisBench records and normalization.
URL_RE = re.compile(r'(?:https?://\S+|www\.\S+)', flags=re.IGNORECASE)
USER_RE = re.compile(r'@[A-Za-z0-9_]+')
HASHTAG_RE = re.compile(r'#([A-Za-z0-9_]+)')
NUMBER_RE = re.compile(r'\b\d+(?:[.,]\d+)?\b')
NON_TOKEN_RE = re.compile(r"[^a-z0-9_\s']+")
SPACE_RE = re.compile(r'\s+')

REQUEST_SET = {'requests_or_needs'}
RESPONSE_SET = {'response_efforts', 'donation_and_volunteering'}
TRIAGE_LABELS = ['other', 'request', 'response']

def normalize_text(value):
    if value is None:
        return ''
    try:
        if bool(pd.isna(value)):
            return ''
    except Exception:
        pass
    text = html.unescape(str(value)).lower()
    text = URL_RE.sub(' urltoken ', text)
    text = USER_RE.sub(' usertoken ', text)
    text = HASHTAG_RE.sub(r' \1 ', text)
    text = NUMBER_RE.sub(' numtoken ', text)
    text = NON_TOKEN_RE.sub(' ', text)
    return SPACE_RE.sub(' ', text).strip()

def triage_map(label):
    if label in REQUEST_SET: return 'request'
    if label in RESPONSE_SET: return 'response'
    return 'other'

def read_tsv(path, split_name):
    df = pd.read_csv(path, sep='\t')
    for col in ('text','class_label','event'):
        if col not in df.columns:
            raise KeyError(f'{split_name} missing required column {col}; columns={list(df.columns)}')
    df = df.copy()
    df['raw_text'] = df['text'].fillna('').astype(str)
    df['clean_text'] = df['raw_text'].map(normalize_text)
    df['class_label'] = df['class_label'].astype(str).str.strip()
    mask = df['class_label'].ne('') & df['clean_text'].ne('')
    df = df.loc[mask].reset_index(drop=True)
    df['source_split'] = split_name
    df['source_row'] = np.arange(len(df), dtype=np.int64)
    return df

train_df = read_tsv(TRAIN_PATH, 'train')
test_df = read_tsv(TEST_PATH, 'test')
combined = pd.concat([train_df, test_df], ignore_index=True, sort=False)
combined['global_row_id'] = np.arange(len(combined), dtype=np.int64)
combined['label_11'] = combined['class_label'].astype(str)
combined['label_triage'] = combined['label_11'].map(triage_map)
LABELS = sorted(train_df['class_label'].astype(str).unique().tolist())

prior_fixed = pd.read_csv(PRIOR_TABLE / 'fixed_split_predictions.csv')
prior_event = pd.read_csv(PRIOR_TABLE / 'event_disjoint_oof_predictions.csv')
prior_event_assign = pd.read_csv(PRIOR_TABLE / 'event_disjoint_fold_assignment.csv')

if len(train_df) != 61164 or len(test_df) != 17356:
    raise RuntimeError(f'Prepared row counts changed: train={len(train_df)}, test={len(test_df)}')
if len(combined) != 78520:
    raise RuntimeError('Prepared combined population must remain 78,520 records.')

strict_ids = prior_event['global_row_id'].astype(int).to_numpy()
strict_df = combined.set_index('global_row_id').loc[strict_ids].reset_index()
strict_df = strict_df.merge(
    prior_event_assign[['global_row_id','canonical_event','fold']],
    on='global_row_id', how='left', validate='one_to_one'
)
if strict_df['fold'].isna().any():
    raise RuntimeError('Strict population did not match saved event-disjoint fold assignments.')

print('Prepared train/test:', len(train_df), len(test_df))
print('Strict event population:', len(strict_df), 'events:', strict_df['canonical_event'].nunique())
print('Labels:', LABELS)


## Fine-tuned contemporary transformer

The previous rerun used a **frozen MiniLM embedding + logistic regression** baseline. This continuation adds **end-to-end supervised DistilBERT fine-tuning**. Hyperparameters are fixed before test evaluation; no official test labels are used for model selection.

In [ ]:
# 6. Transformer helpers: fixed epochs, class-weighted cross-entropy, checkpointable predictions.
DEVICE = torch.device('cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(map(str, texts))
        self.labels = np.asarray(labels, dtype=np.int64)
        self.tokenizer = tokenizer
        self.max_length = int(max_length)
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

def make_label_maps(label_names):
    label_names = list(label_names)
    return {x:i for i,x in enumerate(label_names)}, {i:x for i,x in enumerate(label_names)}

def class_weights_from_y(y_int, n_classes):
    present = np.unique(y_int)
    weights = np.ones(n_classes, dtype=np.float32)
    vals = compute_class_weight(class_weight='balanced', classes=present, y=y_int)
    for cls, w in zip(present, vals): weights[int(cls)] = float(w)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def train_transformer_fixed_epochs(train_texts, train_labels_str, label_names, epochs, run_name):
    label2id, id2label = make_label_maps(label_names)
    y = np.asarray([label2id[x] for x in train_labels_str], dtype=np.int64)
    ds = TextDataset(train_texts, y, tokenizer, MAX_LENGTH)
    gen = torch.Generator(); gen.manual_seed(SEED)
    loader = DataLoader(ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, generator=gen, num_workers=2, pin_memory=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(label_names), label2id=label2id, id2label=id2label
    ).to(DEVICE)
    weights = class_weights_from_y(y, len(label_names))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(loader) * int(epochs))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps
    )
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    history=[]
    for epoch in range(1, int(epochs)+1):
        model.train(); losses=[]; t0=time.time()
        for batch in loader:
            labels = batch.pop('labels').to(DEVICE, non_blocking=True)
            batch = {k:v.to(DEVICE, non_blocking=True) for k,v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=True):
                logits = model(**batch).logits
                loss = torch.nn.functional.cross_entropy(logits, labels, weight=weights)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            losses.append(float(loss.detach().cpu()))
        row={'run_name':run_name,'epoch':epoch,'mean_train_loss':float(np.mean(losses)),'seconds':float(time.time()-t0)}
        history.append(row); print(row)
    return model, pd.DataFrame(history), label2id, id2label

@torch.no_grad()
def predict_transformer(model, texts, id2label, return_prob=True, device=DEVICE):
    dummy = np.zeros(len(texts), dtype=np.int64)
    ds = TextDataset(texts, dummy, tokenizer, MAX_LENGTH)
    loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(device.type=='cuda'))
    model = model.to(device); model.eval()
    preds=[]; probs=[]
    for batch in loader:
        batch.pop('labels', None)
        batch = {k:v.to(device, non_blocking=(device.type=='cuda')) for k,v in batch.items()}
        logits = model(**batch).logits
        p = torch.softmax(logits, dim=-1)
        idx = torch.argmax(p, dim=-1).detach().cpu().numpy()
        preds.extend([id2label[int(i)] for i in idx])
        if return_prob: probs.append(p.detach().cpu().numpy())
    prob_arr = np.vstack(probs) if return_prob else None
    return np.asarray(preds, dtype=object), prob_arr

def metric_row(y_true, y_pred, labels):
    labels=list(labels)
    return {
        'n': int(len(y_true)),
        'accuracy': float(accuracy_score(y_true,y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true,y_pred)),
        'macro_f1': float(f1_score(y_true,y_pred,labels=labels,average='macro',zero_division=0)),
        'weighted_f1': float(f1_score(y_true,y_pred,labels=labels,average='weighted',zero_division=0)),
    }

def save_class_report(y_true, y_pred, labels, path):
    rep=classification_report(y_true,y_pred,labels=list(labels),output_dict=True,zero_division=0)
    pd.DataFrame(rep).T.reset_index().rename(columns={'index':'label'}).to_csv(path,index=False)


In [ ]:
# 7. Official fixed-split fine-tuned DistilBERT: 11-class and direct triage.
fixed_rows=[]
fixed_pred_path = TABLE_DIR / 'distilbert_fixed_split_predictions.csv'
fixed_model_11_dir = MODEL_DIR / 'distilbert_fixed_11class'
fixed_model_tri_dir = MODEL_DIR / 'distilbert_fixed_direct_triage'

if fixed_pred_path.is_file():
    print('Reusing completed fixed-split DistilBERT outputs.')
    distil_fixed_pred = pd.read_csv(fixed_pred_path)
else:
    # 11 class
    model11, hist11, l2i11, i2l11 = train_transformer_fixed_epochs(
        train_df['clean_text'].tolist(), train_df['class_label'].tolist(), LABELS, FIXED_EPOCHS, 'fixed_11class'
    )
    pred11, prob11 = predict_transformer(model11, test_df['clean_text'].tolist(), i2l11)
    hist11.to_csv(TABLE_DIR/'distilbert_fixed_11class_training_history.csv', index=False)
    save_class_report(test_df['class_label'].to_numpy(), pred11, LABELS, TABLE_DIR/'distilbert_fixed_11class_per_class.csv')
    fixed_model_11_dir.mkdir(parents=True, exist_ok=True)
    model11.save_pretrained(fixed_model_11_dir); tokenizer.save_pretrained(fixed_model_11_dir)
    np.save(TABLE_DIR/'distilbert_fixed_11class_probabilities.npy', prob11)
    del model11; gc.collect(); torch.cuda.empty_cache()

    # Direct triage
    modeltri, histtri, l2it, i2lt = train_transformer_fixed_epochs(
        train_df['clean_text'].tolist(), train_df['class_label'].map(triage_map).tolist(), TRIAGE_LABELS, FIXED_EPOCHS, 'fixed_direct_triage'
    )
    predtri, probtri = predict_transformer(modeltri, test_df['clean_text'].tolist(), i2lt)
    histtri.to_csv(TABLE_DIR/'distilbert_fixed_direct_triage_training_history.csv', index=False)
    save_class_report(test_df['class_label'].map(triage_map).to_numpy(), predtri, TRIAGE_LABELS, TABLE_DIR/'distilbert_fixed_direct_triage_per_class.csv')
    fixed_model_tri_dir.mkdir(parents=True, exist_ok=True)
    modeltri.save_pretrained(fixed_model_tri_dir); tokenizer.save_pretrained(fixed_model_tri_dir)
    np.save(TABLE_DIR/'distilbert_fixed_direct_triage_probabilities.npy', probtri)
    del modeltri; gc.collect(); torch.cuda.empty_cache()

    distil_fixed_pred = pd.DataFrame({
        'global_row_id': np.arange(len(train_df),len(train_df)+len(test_df),dtype=np.int64),
        'label_11': test_df['class_label'].to_numpy(),
        'label_triage': test_df['class_label'].map(triage_map).to_numpy(),
        'pred_11class__distilbert_ft': pred11,
        'pred_direct_triage__distilbert_ft': predtri,
    })
    distil_fixed_pred.to_csv(fixed_pred_path,index=False)

fixed_rows.append({'protocol':'official_fixed_split','task':'11class','model':'distilbert_ft',**metric_row(distil_fixed_pred['label_11'],distil_fixed_pred['pred_11class__distilbert_ft'],LABELS)})
fixed_rows.append({'protocol':'official_fixed_split','task':'direct_triage','model':'distilbert_ft',**metric_row(distil_fixed_pred['label_triage'],distil_fixed_pred['pred_direct_triage__distilbert_ft'],TRIAGE_LABELS)})
pd.DataFrame(fixed_rows).to_csv(TABLE_DIR/'distilbert_fixed_split_summary.csv',index=False)
print(pd.DataFrame(fixed_rows))


In [ ]:
# 8. Exact saved 5-fold event-disjoint protocol for fine-tuned DistilBERT.
event_fold_summaries=[]
event_oof = strict_df[['global_row_id','canonical_event','fold','label_11','label_triage']].copy()
event_oof['pred_11class__distilbert_ft'] = None
if RUN_EVENT_DIRECT_TRIAGE:
    event_oof['pred_direct_triage__distilbert_ft'] = None

for fold in sorted(strict_df['fold'].astype(int).unique()):
    fold_ckpt = CHECKPOINT_DIR / f'distilbert_event_fold_{fold}.csv'
    if fold_ckpt.is_file():
        print('Reusing fold checkpoint', fold)
        ck = pd.read_csv(fold_ckpt)
        pos = event_oof['fold'].astype(int).eq(fold)
        ck = ck.set_index('global_row_id')
        ids = event_oof.loc[pos,'global_row_id'].astype(int)
        event_oof.loc[pos,'pred_11class__distilbert_ft'] = ids.map(ck['pred_11class__distilbert_ft']).to_numpy()
        if RUN_EVENT_DIRECT_TRIAGE and 'pred_direct_triage__distilbert_ft' in ck.columns:
            event_oof.loc[pos,'pred_direct_triage__distilbert_ft'] = ids.map(ck['pred_direct_triage__distilbert_ft']).to_numpy()
        continue

    train_mask = strict_df['fold'].astype(int).ne(fold).to_numpy()
    test_mask = ~train_mask
    tr = strict_df.loc[train_mask]
    te = strict_df.loc[test_mask]
    print(f'\nEVENT FOLD {fold}: train={len(tr):,} test={len(te):,} held events={te.canonical_event.nunique()}')

    model11, hist, l2i, i2l = train_transformer_fixed_epochs(
        tr['clean_text'].tolist(), tr['label_11'].tolist(), LABELS, EVENT_EPOCHS, f'event_fold_{fold}_11class'
    )
    p11, _ = predict_transformer(model11, te['clean_text'].tolist(), i2l, return_prob=False)
    hist.to_csv(TABLE_DIR/f'distilbert_event_fold_{fold}_11class_history.csv',index=False)
    del model11; gc.collect(); torch.cuda.empty_cache()

    ck = te[['global_row_id','canonical_event','fold','label_11','label_triage']].copy()
    ck['pred_11class__distilbert_ft'] = p11

    if RUN_EVENT_DIRECT_TRIAGE:
        modelt, histt, l2it, i2lt = train_transformer_fixed_epochs(
            tr['clean_text'].tolist(), tr['label_triage'].tolist(), TRIAGE_LABELS, EVENT_EPOCHS, f'event_fold_{fold}_direct_triage'
        )
        pt, _ = predict_transformer(modelt, te['clean_text'].tolist(), i2lt, return_prob=False)
        histt.to_csv(TABLE_DIR/f'distilbert_event_fold_{fold}_triage_history.csv',index=False)
        ck['pred_direct_triage__distilbert_ft'] = pt
        del modelt; gc.collect(); torch.cuda.empty_cache()

    ck.to_csv(fold_ckpt,index=False)
    pos = event_oof['fold'].astype(int).eq(fold)
    mapper11 = ck.set_index('global_row_id')['pred_11class__distilbert_ft']
    event_oof.loc[pos,'pred_11class__distilbert_ft'] = event_oof.loc[pos,'global_row_id'].map(mapper11).to_numpy()
    if RUN_EVENT_DIRECT_TRIAGE:
        mappert = ck.set_index('global_row_id')['pred_direct_triage__distilbert_ft']
        event_oof.loc[pos,'pred_direct_triage__distilbert_ft'] = event_oof.loc[pos,'global_row_id'].map(mappert).to_numpy()

if event_oof['pred_11class__distilbert_ft'].isna().any():
    raise RuntimeError('Missing 11-class OOF predictions after event folds.')
if RUN_EVENT_DIRECT_TRIAGE and event_oof['pred_direct_triage__distilbert_ft'].isna().any():
    raise RuntimeError('Missing direct-triage OOF predictions after event folds.')

event_oof.to_csv(TABLE_DIR/'distilbert_event_disjoint_oof_predictions.csv',index=False)
rows=[{'protocol':'event_disjoint_same_corpus','task':'11class','model':'distilbert_ft',**metric_row(event_oof['label_11'],event_oof['pred_11class__distilbert_ft'],LABELS)}]
if RUN_EVENT_DIRECT_TRIAGE:
    rows.append({'protocol':'event_disjoint_same_corpus','task':'direct_triage','model':'distilbert_ft',**metric_row(event_oof['label_triage'],event_oof['pred_direct_triage__distilbert_ft'],TRIAGE_LABELS)})
pd.DataFrame(rows).to_csv(TABLE_DIR/'distilbert_event_disjoint_summary.csv',index=False)
save_class_report(event_oof['label_11'],event_oof['pred_11class__distilbert_ft'],LABELS,TABLE_DIR/'distilbert_event_disjoint_11class_per_class.csv')
if RUN_EVENT_DIRECT_TRIAGE:
    save_class_report(event_oof['label_triage'],event_oof['pred_direct_triage__distilbert_ft'],TRIAGE_LABELS,TABLE_DIR/'distilbert_event_disjoint_triage_per_class.csv')
print(pd.DataFrame(rows))


In [ ]:
# 9. Primary effect estimates: DistilBERT versus prior sparse and frozen-MiniLM models.
def confusion_fixed(y_true, y_pred, labels):
    return confusion_matrix(y_true,y_pred,labels=list(labels)).astype(float)

def macro_f1_from_cm(cm):
    tp=np.diag(cm); fp=cm.sum(0)-tp; fn=cm.sum(1)-tp
    den=2*tp+fp+fn
    vals=np.divide(2*tp,den,out=np.zeros_like(tp,dtype=float),where=den>0)
    return float(vals.mean())

def stratified_paired_bootstrap(y_true, pred_a, pred_b, labels, n_boot, seed):
    y_true=np.asarray(y_true,dtype=object); a=np.asarray(pred_a,dtype=object); b=np.asarray(pred_b,dtype=object)
    labels=list(labels); rng=np.random.default_rng(seed)
    idx_by=[np.flatnonzero(y_true==lab) for lab in labels]
    observed=macro_f1_from_cm(confusion_fixed(y_true,a,labels))-macro_f1_from_cm(confusion_fixed(y_true,b,labels))
    vals=[]
    for _ in range(n_boot):
        idx=np.concatenate([rng.choice(ix,size=len(ix),replace=True) for ix in idx_by if len(ix)])
        vals.append(macro_f1_from_cm(confusion_fixed(y_true[idx],a[idx],labels))-macro_f1_from_cm(confusion_fixed(y_true[idx],b[idx],labels)))
    lo,hi=np.quantile(vals,[.025,.975])
    return observed,float(lo),float(hi)

def event_cluster_bootstrap(y_true,pred_a,pred_b,groups,labels,n_boot,seed):
    y_true=np.asarray(y_true,dtype=object); a=np.asarray(pred_a,dtype=object); b=np.asarray(pred_b,dtype=object); g=np.asarray(groups,dtype=object)
    events=sorted(np.unique(g).tolist()); labels=list(labels)
    cms_a=[]; cms_b=[]
    for ev in events:
        m=g==ev; cms_a.append(confusion_fixed(y_true[m],a[m],labels)); cms_b.append(confusion_fixed(y_true[m],b[m],labels))
    cms_a=np.stack(cms_a); cms_b=np.stack(cms_b); rng=np.random.default_rng(seed)
    observed=macro_f1_from_cm(cms_a.sum(0))-macro_f1_from_cm(cms_b.sum(0)); vals=[]
    for _ in range(n_boot):
        ids=rng.integers(0,len(events),size=len(events))
        vals.append(macro_f1_from_cm(cms_a[ids].sum(0))-macro_f1_from_cm(cms_b[ids].sum(0)))
    lo,hi=np.quantile(vals,[.025,.975])
    return observed,float(lo),float(hi)

comparisons=[]
prior_fixed_idx = prior_fixed.set_index('global_row_id')
new_fixed_idx = distil_fixed_pred.set_index('global_row_id')
merged_fixed = prior_fixed_idx.join(new_fixed_idx[['pred_11class__distilbert_ft','pred_direct_triage__distilbert_ft']],how='inner')
for task,true_col,new_col,labels,prefix in [
    ('11class','label_11','pred_11class__distilbert_ft',LABELS,'pred_11class__'),
    ('direct_triage','label_triage','pred_direct_triage__distilbert_ft',TRIAGE_LABELS,'pred_direct_triage__')]:
    for base in ['tfidf_svc','tfidf_lr','hash_lr','minilm_lr']:
        old_col=prefix+base
        eff,lo,hi=stratified_paired_bootstrap(merged_fixed[true_col],merged_fixed[new_col],merged_fixed[old_col],labels,N_BOOTSTRAP,SEED+len(comparisons))
        comparisons.append({'protocol':'official_fixed_split','task':task,'model_a':'distilbert_ft','model_b':base,'effect_macro_f1_a_minus_b':eff,'ci_2_5':lo,'ci_97_5':hi,'resampling_unit':'message_within_true_class_strata','n_resamples':N_BOOTSTRAP})

prior_event_idx=prior_event.set_index('global_row_id')
new_event_idx=event_oof.set_index('global_row_id')
merged_event=prior_event_idx.join(new_event_idx[['pred_11class__distilbert_ft'] + (['pred_direct_triage__distilbert_ft'] if RUN_EVENT_DIRECT_TRIAGE else [])],how='inner')
for task,true_col,new_col,labels,prefix in [
    ('11class','label_11','pred_11class__distilbert_ft',LABELS,'pred_11class__'),
    ('direct_triage','label_triage','pred_direct_triage__distilbert_ft',TRIAGE_LABELS,'pred_direct_triage__')]:
    if task=='direct_triage' and not RUN_EVENT_DIRECT_TRIAGE: continue
    for base in ['tfidf_svc','tfidf_lr','hash_lr','minilm_lr']:
        old_col=prefix+base
        eff,lo,hi=event_cluster_bootstrap(merged_event[true_col],merged_event[new_col],merged_event[old_col],merged_event['canonical_event'],labels,N_BOOTSTRAP,SEED+100+len(comparisons))
        comparisons.append({'protocol':'event_disjoint_same_corpus','task':task,'model_a':'distilbert_ft','model_b':base,'effect_macro_f1_a_minus_b':eff,'ci_2_5':lo,'ci_97_5':hi,'resampling_unit':'canonical_event_cluster','n_resamples':N_BOOTSTRAP})

pd.DataFrame(comparisons).to_csv(TABLE_DIR/'distilbert_primary_effect_estimates_and_cis.csv',index=False)
print(pd.DataFrame(comparisons).to_string(index=False))


## External validation on HumAID

HumAID is a separately released, human-annotated disaster-event dataset with 19 events. The notebook downloads only the published **test split** for each event, checks for exact normalized-text overlap against CrisisBench, removes overlaps, and reports:

- external **three-way triage** transfer on the full retained HumAID test population;
- external **shared-ontology multi-class** transfer on labels that have an explicit one-to-one counterpart in the CrisisBench label set;
- pooled and event-wise metrics.

This is external validation, but claims should remain specific to the harmonized tasks and dataset.

In [ ]:
# 10. Download and harmonize HumAID test splits from the official QCRI Hugging Face dataset.
HUMAID_EVENTS = [
'hurricane_florence_2018','kaikoura_earthquake_2016','kerala_floods_2018','hurricane_harvey_2017',
'hurricane_maria_2017','midwestern_us_floods_2019','puebla_mexico_earthquake_2017','maryland_floods_2018',
'hurricane_irma_2017','ecuador_earthquake_2016','cyclone_idai_2019','canada_wildfires_2016',
'italy_earthquake_aug_2016','greece_wildfires_2018','hurricane_dorian_2019','california_wildfires_2018',
'pakistan_earthquake_2019','hurricane_matthew_2016','srilanka_floods_2017']

HUMAID_CACHE = WORKDIR / '_humaid_external_cache'; HUMAID_CACHE.mkdir(exist_ok=True)

def load_json_flexible(path):
    txt=Path(path).read_text(encoding='utf-8').strip()
    if not txt: return []
    try:
        obj=json.loads(txt)
        if isinstance(obj,list): return obj
        if isinstance(obj,dict):
            for key in ('data','records','examples'):
                if key in obj and isinstance(obj[key],list): return obj[key]
            return [obj]
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

def norm_humaid_label(x):
    x=str(x).strip().lower(); x=re.sub(r'[^a-z0-9]+','_',x).strip('_')
    aliases={
        'caution_and_advice':'caution_and_advice',
        'displaced_people_and_evacuations':'displaced_and_evacuations',
        'displaced_and_evacuations':'displaced_and_evacuations',
        'infrastructure_and_utility_damage':'infrastructure_and_utilities_damage',
        'infrastructure_and_utilities_damage':'infrastructure_and_utilities_damage',
        'injured_or_dead_people':'injured_or_dead_people',
        'missing_or_found_people':'missing_and_found_people',
        'missing_and_found_people':'missing_and_found_people',
        'not_humanitarian':'not_humanitarian',
        'requests_or_urgent_needs':'requests_or_needs',
        'requests_or_needs':'requests_or_needs',
        'rescue_volunteering_or_donation_effort':'donation_and_volunteering',
        'rescue_volunteering_or_donation_efforts':'donation_and_volunteering',
        'sympathy_and_support':'sympathy_and_support',
        'other_relevant_information':'other_relevant_information',
        'dont_know_cant_judge':'dont_know_cant_judge',
        'dont_know_cant_judge_':'dont_know_cant_judge',
    }
    return aliases.get(x,x)

def external_triage(label):
    lab=norm_humaid_label(label)
    if lab=='requests_or_needs': return 'request'
    if lab=='donation_and_volunteering': return 'response'
    return 'other'

if RUN_EXTERNAL_HUMAID:
    external_rows=[]
    for event in HUMAID_EVENTS:
        local=HUMAID_CACHE/f'{event}_test.json'
        if not local.is_file():
            url=f'https://huggingface.co/datasets/QCRI/HumAID-events/resolve/main/{event}/test.json'
            r=requests.get(url,timeout=120); r.raise_for_status(); local.write_bytes(r.content)
        records=load_json_flexible(local)
        for rec in records:
            text_val=rec.get('tweet_text', rec.get('text',''))
            label_val=rec.get('class_label', rec.get('label',''))
            external_rows.append({'external_event':event,'raw_text':str(text_val),'external_label_raw':str(label_val)})
    huma=pd.DataFrame(external_rows)
    huma['clean_text']=huma['raw_text'].map(normalize_text)
    huma['label_harmonized']=huma['external_label_raw'].map(norm_humaid_label)
    huma['label_triage']=huma['external_label_raw'].map(external_triage)
    huma=huma[huma['clean_text'].ne('')].reset_index(drop=True)

    crisis_all_text=set(combined['clean_text'].astype(str))
    huma['exact_overlap_crisisbench']=huma['clean_text'].isin(crisis_all_text)
    overlap_summary=huma.groupby('external_event',as_index=False)['exact_overlap_crisisbench'].agg(['sum','count']).reset_index()
    overlap_summary.to_csv(TABLE_DIR/'humaid_exact_overlap_by_event.csv',index=False)
    huma_clean=huma.loc[~huma['exact_overlap_crisisbench']].copy().reset_index(drop=True)
    huma.to_csv(TABLE_DIR/'humaid_external_test_all_with_overlap_flag.csv',index=False)
    huma_clean.to_csv(TABLE_DIR/'humaid_external_test_overlap_removed.csv',index=False)
    pd.DataFrame({'label':huma_clean['label_harmonized'].value_counts().index,'rows':huma_clean['label_harmonized'].value_counts().values}).to_csv(TABLE_DIR/'humaid_harmonized_label_inventory.csv',index=False)
    print('HumAID test rows downloaded:',len(huma),'after exact-overlap removal:',len(huma_clean),'exact overlaps:',int(huma['exact_overlap_crisisbench'].sum()))
    print(huma_clean['label_harmonized'].value_counts())


In [ ]:
# 11. External transfer: sparse baselines and the fixed-split fine-tuned DistilBERT models.
def build_tfidf():
    return TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=0.95,max_features=50000,lowercase=False,stop_words='english',token_pattern=r'(?u)\b[a-z][a-z0-9_]{1,}\b',norm='l2',dtype=np.float64)

if RUN_EXTERNAL_HUMAID:
    ext_rows=[]
    # Train sparse 11-class + triage on CrisisBench official training only.
    vec=build_tfidf(); Xtr=vec.fit_transform(train_df['clean_text']); Xext=vec.transform(huma_clean['clean_text'])
    svc11=LinearSVC(C=1.0,class_weight='balanced',max_iter=40000,random_state=SEED).fit(Xtr,train_df['class_label'])
    lr11=LogisticRegression(C=1.0,class_weight='balanced',solver='lbfgs',max_iter=30000,random_state=SEED).fit(Xtr,train_df['class_label'])
    svct=LinearSVC(C=1.0,class_weight='balanced',max_iter=40000,random_state=SEED).fit(Xtr,train_df['class_label'].map(triage_map))
    lrt=LogisticRegression(C=1.0,class_weight='balanced',solver='lbfgs',max_iter=30000,random_state=SEED).fit(Xtr,train_df['class_label'].map(triage_map))

    # Reload fine-tuned fixed models and predict external text.
    tok_ext=AutoTokenizer.from_pretrained(fixed_model_11_dir,use_fast=True)
    tokenizer = tok_ext
    d11=AutoModelForSequenceClassification.from_pretrained(fixed_model_11_dir).to(DEVICE)
    i2l11={int(k):v for k,v in d11.config.id2label.items()}
    p11_ext,_=predict_transformer(d11,huma_clean['clean_text'].tolist(),i2l11,return_prob=False)
    del d11; gc.collect(); torch.cuda.empty_cache()
    dt=AutoModelForSequenceClassification.from_pretrained(fixed_model_tri_dir).to(DEVICE)
    i2lt={int(k):v for k,v in dt.config.id2label.items()}
    pt_ext,pt_prob=predict_transformer(dt,huma_clean['clean_text'].tolist(),i2lt,return_prob=True)
    del dt; gc.collect(); torch.cuda.empty_cache()

    huma_pred=huma_clean[['external_event','external_label_raw','label_harmonized','label_triage','raw_text','clean_text']].copy()
    huma_pred['pred_11class__tfidf_svc']=svc11.predict(Xext)
    huma_pred['pred_11class__tfidf_lr']=lr11.predict(Xext)
    huma_pred['pred_11class__distilbert_ft']=p11_ext
    huma_pred['pred_triage__tfidf_svc']=svct.predict(Xext)
    huma_pred['pred_triage__tfidf_lr']=lrt.predict(Xext)
    huma_pred['pred_triage__distilbert_ft']=pt_ext
    huma_pred.to_csv(TABLE_DIR/'humaid_external_predictions.csv',index=False)
    np.save(TABLE_DIR/'humaid_external_distilbert_triage_probabilities.npy',pt_prob)

    # Full external triage evaluation.
    for model,col in [('tfidf_svc','pred_triage__tfidf_svc'),('tfidf_lr','pred_triage__tfidf_lr'),('distilbert_ft','pred_triage__distilbert_ft')]:
        ext_rows.append({'external_task':'triage_3class','model':model,**metric_row(huma_pred['label_triage'],huma_pred[col],TRIAGE_LABELS)})

    # Shared explicit ontology: only labels with one-to-one CrisisBench counterparts.
    shared_labels=['caution_and_advice','displaced_and_evacuations','infrastructure_and_utilities_damage','injured_or_dead_people','missing_and_found_people','not_humanitarian','requests_or_needs','donation_and_volunteering','sympathy_and_support']
    shared=huma_pred[huma_pred['label_harmonized'].isin(shared_labels)].copy()
    for model,col in [('tfidf_svc','pred_11class__tfidf_svc'),('tfidf_lr','pred_11class__tfidf_lr'),('distilbert_ft','pred_11class__distilbert_ft')]:
        ext_rows.append({'external_task':'shared_9class','model':model,**metric_row(shared['label_harmonized'],shared[col],shared_labels),'unsupported_prediction_rate':float((~shared[col].isin(shared_labels)).mean())})

    ext_summary=pd.DataFrame(ext_rows); ext_summary.to_csv(TABLE_DIR/'humaid_external_transfer_summary.csv',index=False)
    print(ext_summary.to_string(index=False))

    # Event-wise external metrics.
    evrows=[]
    for event,g in huma_pred.groupby('external_event'):
        for model,col in [('tfidf_svc','pred_triage__tfidf_svc'),('tfidf_lr','pred_triage__tfidf_lr'),('distilbert_ft','pred_triage__distilbert_ft')]:
            evrows.append({'external_event':event,'task':'triage_3class','model':model,**metric_row(g['label_triage'],g[col],TRIAGE_LABELS)})
        gs=g[g['label_harmonized'].isin(shared_labels)]
        if len(gs):
            for model,col in [('tfidf_svc','pred_11class__tfidf_svc'),('tfidf_lr','pred_11class__tfidf_lr'),('distilbert_ft','pred_11class__distilbert_ft')]:
                evrows.append({'external_event':event,'task':'shared_9class','model':model,**metric_row(gs['label_harmonized'],gs[col],shared_labels)})
    pd.DataFrame(evrows).to_csv(TABLE_DIR/'humaid_external_eventwise_metrics.csv',index=False)


In [ ]:
# 12. Selective prediction / abstention analysis on official test and external HumAID.
def selective_curve(y_true, pred, probs, classes, labels, model, dataset):
    conf=probs.max(axis=1)
    rows=[]
    for threshold in np.arange(0.34,0.951,0.05):
        accepted=conf>=threshold
        if accepted.sum()==0: continue
        m=metric_row(np.asarray(y_true)[accepted],np.asarray(pred)[accepted],labels)
        req_mask=np.asarray(y_true)[accepted]=='request'
        req_recall=float(np.mean(np.asarray(pred)[accepted][req_mask]=='request')) if req_mask.any() else np.nan
        rows.append({'dataset':dataset,'model':model,'threshold':float(threshold),'coverage':float(accepted.mean()),'abstention_rate':float(1-accepted.mean()),'accepted_n':int(accepted.sum()),'request_recall_within_accepted':req_recall,**m})
    return rows

sel_rows=[]
# DistilBERT official-test probabilities were saved in section 7.
fixed_tri_prob=np.load(TABLE_DIR/'distilbert_fixed_direct_triage_probabilities.npy')
# Class order is saved in the model config.
_tmp=AutoModelForSequenceClassification.from_pretrained(fixed_model_tri_dir)
tri_class_order=[_tmp.config.id2label[i] for i in range(_tmp.config.num_labels)]; del _tmp
sel_rows += selective_curve(distil_fixed_pred['label_triage'].to_numpy(),distil_fixed_pred['pred_direct_triage__distilbert_ft'].to_numpy(),fixed_tri_prob,tri_class_order,TRIAGE_LABELS,'distilbert_ft','crisisbench_official_test')

# Fresh TF-IDF LR probability baseline on official test.
vec_sel=build_tfidf(); Xtr_sel=vec_sel.fit_transform(train_df['clean_text']); Xte_sel=vec_sel.transform(test_df['clean_text'])
lr_sel=LogisticRegression(C=1.0,class_weight='balanced',solver='lbfgs',max_iter=30000,random_state=SEED).fit(Xtr_sel,train_df['class_label'].map(triage_map))
pred_lr_sel=lr_sel.predict(Xte_sel); prob_lr_sel=lr_sel.predict_proba(Xte_sel)
sel_rows += selective_curve(test_df['class_label'].map(triage_map).to_numpy(),pred_lr_sel,prob_lr_sel,lr_sel.classes_,TRIAGE_LABELS,'tfidf_lr','crisisbench_official_test')

if RUN_EXTERNAL_HUMAID:
    prob_ext=np.load(TABLE_DIR/'humaid_external_distilbert_triage_probabilities.npy')
    hp=pd.read_csv(TABLE_DIR/'humaid_external_predictions.csv')
    sel_rows += selective_curve(hp['label_triage'].to_numpy(),hp['pred_triage__distilbert_ft'].to_numpy(),prob_ext,tri_class_order,TRIAGE_LABELS,'distilbert_ft','humaid_external')
    Xext_sel=vec_sel.transform(hp['clean_text']); pred=lr_sel.predict(Xext_sel); prob=lr_sel.predict_proba(Xext_sel)
    sel_rows += selective_curve(hp['label_triage'].to_numpy(),pred,prob,lr_sel.classes_,TRIAGE_LABELS,'tfidf_lr','humaid_external')

pd.DataFrame(sel_rows).to_csv(TABLE_DIR/'selective_prediction_abstention_curves.csv',index=False)
print(pd.DataFrame(sel_rows).head(20).to_string(index=False))


In [ ]:
# 13. Event-level lexical shift diagnostic versus event-disjoint performance.
# Descriptive analysis only; it is not used to tune models or folds.
from collections import Counter

def tokenize_clean(text):
    return [t for t in str(text).split() if len(t)>1]

event_counts={}
global_counts=Counter()
for ev,g in strict_df.groupby('canonical_event'):
    c=Counter()
    for txt in g['clean_text']:
        c.update(tokenize_clean(txt))
    event_counts[ev]=c; global_counts.update(c)

top_vocab=[w for w,_ in global_counts.most_common(20000)]
vocab_idx={w:i for i,w in enumerate(top_vocab)}
shift_rows=[]
for ev,c_ev in event_counts.items():
    c_train=global_counts-c_ev
    p=np.array([c_ev.get(w,0) for w in top_vocab],dtype=float)+1e-12
    q=np.array([c_train.get(w,0) for w in top_vocab],dtype=float)+1e-12
    p/=p.sum(); q/=q.sum()
    js=float(jensenshannon(p,q,base=2.0)**2)
    total=sum(c_ev.values())
    oov=sum(n for w,n in c_ev.items() if c_train.get(w,0)==0)/max(total,1)
    shift_rows.append({'canonical_event':ev,'rows':int((strict_df['canonical_event']==ev).sum()),'token_count':int(total),'lexical_js_divergence':js,'oov_token_mass':float(oov)})
shift=pd.DataFrame(shift_rows)

# Add per-event macro-F1 from saved/new OOF predictions.
base_oof=prior_event[['global_row_id','canonical_event','label_11','pred_11class__tfidf_svc','pred_11class__tfidf_lr','pred_11class__minilm_lr']].copy()
base_oof=base_oof.merge(event_oof[['global_row_id','pred_11class__distilbert_ft']],on='global_row_id',validate='one_to_one')
perf=[]
for ev,g in base_oof.groupby('canonical_event'):
    for model,col in [('tfidf_svc','pred_11class__tfidf_svc'),('tfidf_lr','pred_11class__tfidf_lr'),('minilm_lr','pred_11class__minilm_lr'),('distilbert_ft','pred_11class__distilbert_ft')]:
        perf.append({'canonical_event':ev,'model':model,'event_macro_f1':float(f1_score(g['label_11'],g[col],labels=LABELS,average='macro',zero_division=0)),'event_accuracy':float(accuracy_score(g['label_11'],g[col]))})
perf=pd.DataFrame(perf)
shift_perf=perf.merge(shift,on='canonical_event',validate='many_to_one')
shift_perf.to_csv(TABLE_DIR/'event_lexical_shift_and_performance.csv',index=False)

corr=[]
for model,g in shift_perf.groupby('model'):
    for metric in ['lexical_js_divergence','oov_token_mass']:
        rho,p=spearmanr(g[metric],g['event_macro_f1'])
        corr.append({'model':model,'shift_metric':metric,'spearman_rho_with_event_macro_f1':float(rho),'p_value_descriptive':float(p),'n_events':int(len(g))})
pd.DataFrame(corr).to_csv(TABLE_DIR/'event_shift_performance_spearman.csv',index=False)
print(pd.DataFrame(corr).to_string(index=False))


## Two-rater manual audit

The reviewer specifically questioned the mechanically derived triage proxy. The notebook therefore creates a two-rater version of the existing 200-message audit sample. **This step requires real human annotation.**

Recommended labels for each rater: `request`, `response`, `other`, or `uncertain`.

After both raters complete the CSV and save it as `triage_manual_audit_two_rater_COMPLETED.csv` in `VirconPaper1`, rerun the following analysis cell. It calculates inter-rater agreement and estimates how often the current `other` mapping contains implicit requests/responses.

In [ ]:
# 14. Prepare two-rater audit template; analyze it only if completed by humans.
from sklearn.metrics import cohen_kappa_score

prior_audit=pd.read_csv(PRIOR_TABLE/'triage_manual_audit_template.csv')
audit_cols=['global_row_id','source_split','source_row','event','label_11','label_triage','raw_text']
audit2=prior_audit[audit_cols].copy()
for c in ['rater1_triage','rater2_triage','rater1_implicit_request','rater2_implicit_request','rater1_implicit_response','rater2_implicit_response','rater1_notes','rater2_notes']:
    audit2[c]=''
template_path=WORKDIR/'triage_manual_audit_two_rater_template.csv'
if not template_path.is_file():
    audit2.to_csv(template_path,index=False)
print('Human audit template:',template_path)

completed_path=WORKDIR/'triage_manual_audit_two_rater_COMPLETED.csv'
if completed_path.is_file():
    aud=pd.read_csv(completed_path).fillna('')
    valid_labels={'request','response','other','uncertain'}
    mask=aud['rater1_triage'].isin(valid_labels)&aud['rater2_triage'].isin(valid_labels)
    if mask.sum()<50:
        print('Completed audit file found, but fewer than 50 rows have both rater labels. Finish annotation first.')
    else:
        a=aud.loc[mask,'rater1_triage']; b=aud.loc[mask,'rater2_triage']
        kappa=float(cohen_kappa_score(a,b,labels=['request','response','other','uncertain']))
        agreement=float((a==b).mean())
        consensus=np.where(a==b,a,'disagreement')
        aud.loc[mask,'consensus_triage']=consensus
        summary={
            'n_double_rated':int(mask.sum()),'raw_agreement':agreement,'cohen_kappa':kappa,
            'consensus_request_fraction':float(np.mean(consensus=='request')),
            'consensus_response_fraction':float(np.mean(consensus=='response')),
            'consensus_other_fraction':float(np.mean(consensus=='other')),
            'disagreement_fraction':float(np.mean(consensus=='disagreement')),
        }
        pd.DataFrame([summary]).to_csv(TABLE_DIR/'triage_manual_audit_agreement_summary.csv',index=False)
        aud.to_csv(TABLE_DIR/'triage_manual_audit_completed_with_consensus.csv',index=False)
        # Per source-label contamination estimate among audited items.
        sub=aud.loc[mask].copy(); sub['consensus_triage']=consensus
        per=(sub.groupby('label_11')['consensus_triage'].value_counts(normalize=True).rename('fraction').reset_index())
        per.to_csv(TABLE_DIR/'triage_manual_audit_per_source_label_sensitivity.csv',index=False)
        print(pd.DataFrame([summary]).to_string(index=False))
else:
    print('No completed human audit yet. This is expected. Fill and save:', completed_path.name)


In [ ]:
# 15. Deployment benchmark for fine-tuned DistilBERT on CPU, same text path.
# This is measured in the current runtime; use it alongside a same-runtime re-benchmark if strict hardware matching is required.
def benchmark_transformer_cpu(model_dir, texts, repeats=200, batch_n=1000):
    tok=AutoTokenizer.from_pretrained(model_dir,use_fast=True)
    model=AutoModelForSequenceClassification.from_pretrained(model_dir).to('cpu').eval()
    torch.set_num_threads(1)
    sample=list(map(str,texts))
    # warm-up
    for t in sample[:10]:
        enc=tok(t,truncation=True,padding=True,max_length=MAX_LENGTH,return_tensors='pt')
        with torch.no_grad(): _=model(**enc).logits
    times=[]
    for t in sample[:repeats]:
        enc=tok(t,truncation=True,padding=True,max_length=MAX_LENGTH,return_tensors='pt')
        t0=time.perf_counter()
        with torch.no_grad(): _=model(**enc).logits
        times.append((time.perf_counter()-t0)*1000)
    batch_text=sample[:min(batch_n,len(sample))]
    t0=time.perf_counter()
    for i in range(0,len(batch_text),EVAL_BATCH_SIZE):
        enc=tok(batch_text[i:i+EVAL_BATCH_SIZE],truncation=True,padding=True,max_length=MAX_LENGTH,return_tensors='pt')
        with torch.no_grad(): _=model(**enc).logits
    batch_ms=(time.perf_counter()-t0)*1000/max(len(batch_text),1)
    size_bytes=sum(p.stat().st_size for p in Path(model_dir).rglob('*') if p.is_file())
    return {'model':'distilbert_ft','pipeline_size_mb':size_bytes/(1024**2),'median_single_ms':float(np.median(times)),'p95_single_ms':float(np.quantile(times,.95)),'batch_ms_per_message':float(batch_ms),'device':'cpu','torch_threads':1}

bench=benchmark_transformer_cpu(fixed_model_11_dir,test_df['clean_text'].tolist())
pd.DataFrame([bench]).to_csv(TABLE_DIR/'distilbert_cpu_deployment_benchmark.csv',index=False)
print(bench)


In [ ]:
# 16. Final reviewer-facing summary and reproducibility package.
summary_files=[
'distilbert_fixed_split_summary.csv','distilbert_event_disjoint_summary.csv',
'distilbert_primary_effect_estimates_and_cis.csv','selective_prediction_abstention_curves.csv',
'event_lexical_shift_and_performance.csv','event_shift_performance_spearman.csv','distilbert_cpu_deployment_benchmark.csv']
if RUN_EXTERNAL_HUMAID:
    summary_files += ['humaid_external_transfer_summary.csv','humaid_external_eventwise_metrics.csv','humaid_exact_overlap_by_event.csv']

missing=[f for f in summary_files if not (TABLE_DIR/f).is_file()]
if missing:
    raise RuntimeError('Continuation incomplete. Missing outputs: '+', '.join(missing))

# Environment + source hashes.
env={
    'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,
    'scipy':scipy.__version__,'sklearn':sklearn.__version__,'torch':torch.__version__,
    'cuda_available':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(REPORT_DIR/'environment.json').write_text(json.dumps(env,indent=2),encoding='utf-8')

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as fh:
        for chunk in iter(lambda:fh.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
input_hashes={p.name:sha256(p) for p in (TRAIN_PATH,TEST_PATH,PRIOR_ZIP)}
(REPORT_DIR/'input_hashes.json').write_text(json.dumps(input_hashes,indent=2),encoding='utf-8')

claim_note={
'fixed_split':'Official CrisisBench split; do not describe as unseen-disaster generalization.',
'event_disjoint':'Same-corpus event shift on 50 canonical events; stronger than message-level split but not an independent corpus.',
'external_humaid':'External transfer result on HumAID test events after exact normalized-text overlap removal. Multi-class results use the explicit shared ontology only.',
'abstention':'Selective prediction is a computational confidence analysis, not a measured human workload/queue-reduction study.',
'manual_audit':'Only claim practitioner/manual validation after the two-rater file is actually completed.'}
(REPORT_DIR/'claim_scope_and_limitations_continuation.json').write_text(json.dumps(claim_note,indent=2),encoding='utf-8')

# Artifact checksums.
manifest=[]
for p in sorted(OUT_ROOT.rglob('*')):
    if p.is_file(): manifest.append({'relative_path':str(p.relative_to(OUT_ROOT)),'sha256':sha256(p),'bytes':p.stat().st_size})
pd.DataFrame(manifest).to_csv(OUT_ROOT/'artifact_manifest_sha256.csv',index=False)

zip_path=WORKDIR/'VIRCON_IJIES_Continuation_Strengthening_Artifacts.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for p in OUT_ROOT.rglob('*'):
        if p.is_file(): z.write(p,arcname=str(p.relative_to(OUT_ROOT.parent)))
print('\nCONTINUATION COMPLETED')
print('Artifact directory:',OUT_ROOT)
print('Final continuation ZIP:',zip_path)
print('Two-rater audit template:',template_path)


## What to upload back here

After the full run completes, upload:

1. `VIRCON_IJIES_Continuation_Strengthening_Artifacts.zip`
2. **If completed by two human raters:** `triage_manual_audit_two_rater_COMPLETED.csv`

I can then merge these results with the earlier reviewer-rerun outputs, decide which new analyses belong in the main paper versus supplementary material, update the manuscript with minimal disruption to the humanized wording, and prepare the point-by-point IJIES response.

# VIRCON-SAFE — Final Novel-Model Experiment

**VIRCON-SAFE: Shift-Aware Fusion for Event-Robust Crisis Classification**

This section continues from the completed IJIES reviewer-revision and
strengthening experiments.

The proposed method tests whether event-shift indicators can dynamically
control fusion between:

- a lexical TF-IDF logistic-regression expert; and
- a semantic frozen MiniLM logistic-regression expert.

The proposed SAFE gate is **class-specific** and uses uncertainty,
expert disagreement, lexical OOV behaviour, semantic prototype distance,
prototype margin, and message length.

The principal evaluation is the previously fixed **event-disjoint
CrisisBench protocol**. The official split and external HumAID transfer
are secondary evaluations.

The completed 200-message human audit is used only as **independent
validation of the taxonomy-derived triage proxy**. Human-audit labels
are never used for training, gate fitting, threshold tuning, or model
selection.

The notebook evaluates:

1. TF-IDF + LR lexical expert
2. MiniLM + LR semantic expert
3. static 50/50 fusion
4. globally learned scalar fusion
5. confidence-only learned gate
6. SAFE without hierarchical consistency
7. SAFE without request-sensitive risk
8. full VIRCON-SAFE
9. previously completed fine-tuned DistilBERT as a contemporary comparator

Novelty claims should be made only if the full method is supported by
the event-disjoint ablation results.


In [ ]:
# ================================================================
# SAFE 1. Setup, frozen human audit, output paths and embeddings
# ================================================================

from pathlib import Path
import os
import sys
import gc
import re
import json
import math
import time
import shutil
import random
import hashlib
import zipfile
import platform
import subprocess
import importlib.util
import warnings

import numpy as np
import pandas as pd
import scipy
import sklearn
import joblib
import torch

from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

# These objects are created by the continuation notebook above.
_REQUIRED = [
    "WORKDIR",
    "PRIOR_ROOT",
    "PRIOR_TABLE",
    "TABLE_DIR",
    "combined",
    "strict_df",
    "LABELS",
    "TRIAGE_LABELS",
    "triage_map",
]

_missing = [x for x in _REQUIRED if x not in globals()]
if _missing:
    raise RuntimeError(
        "Run the continuation notebook from the top first. "
        "Missing objects: " + ", ".join(_missing)
    )

# Colab-safe dependency handling.
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "sentence-transformers>=5,<6"
    ])

from sentence_transformers import SentenceTransformer

SAFE_SEED = 20261005
SAFE_INNER_SPLITS = 5
SAFE_GATE_EPOCHS = 80
SAFE_GATE_BATCH = 2048
SAFE_GATE_LR = 1e-3
SAFE_GATE_WEIGHT_DECAY = 1e-3
SAFE_CONSISTENCY_LAMBDA = 0.20
SAFE_REQUEST_RISK = 1.50
SAFE_BOOTSTRAP = 2000

random.seed(SAFE_SEED)
np.random.seed(SAFE_SEED)
torch.manual_seed(SAFE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SAFE_SEED)

SAFE_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SAFE_ROOT = WORKDIR / "artifacts_vircon_safe_final"
SAFE_TABLE = SAFE_ROOT / "tables"
SAFE_REPORT = SAFE_ROOT / "reports"
SAFE_MODEL = SAFE_ROOT / "models"
SAFE_CACHE = SAFE_ROOT / "cache"
SAFE_CHECKPOINT = SAFE_ROOT / "checkpoints"

for d in [
    SAFE_ROOT,
    SAFE_TABLE,
    SAFE_REPORT,
    SAFE_MODEL,
    SAFE_CACHE,
    SAFE_CHECKPOINT,
]:
    d.mkdir(parents=True, exist_ok=True)

print("SAFE device:", SAFE_DEVICE)
print("SAFE output:", SAFE_ROOT)

# ------------------------------------------------
# Frozen human-audit evidence
# ------------------------------------------------

audit_summary_candidates = list(
    WORKDIR.rglob("triage_manual_audit_FINAL_frozen_summary.csv")
)

if not audit_summary_candidates:
    raise FileNotFoundError(
        "triage_manual_audit_FINAL_frozen_summary.csv was not found."
    )

AUDIT_SUMMARY_PATH = max(
    audit_summary_candidates,
    key=lambda p: p.stat().st_mtime
)

audit_summary = pd.read_csv(AUDIT_SUMMARY_PATH)

required_audit_cols = [
    "n_audited",
    "raw_interrater_agreement",
    "cohen_kappa",
    "n_adjudicated",
    "hidden_actionable_count",
    "hidden_actionable_fraction",
]

missing_cols = [
    c for c in required_audit_cols
    if c not in audit_summary.columns
]

if missing_cols:
    raise KeyError(
        "Frozen human-audit summary is missing: "
        + ", ".join(missing_cols)
    )

row = audit_summary.iloc[0]

if int(row["n_audited"]) != 200:
    raise RuntimeError("Expected exactly 200 manually audited records.")

if int(row["n_adjudicated"]) != 7:
    raise RuntimeError("Expected exactly 7 adjudicated disagreements.")

if int(row["hidden_actionable_count"]) != 17:
    warnings.warn(
        "Frozen hidden-actionable count is not 17. "
        "Check whether the audit file changed."
    )

audit_summary.to_csv(
    SAFE_TABLE / "human_audit_frozen_summary.csv",
    index=False
)

audit_integration = {
    "human_audit_used_for_training": False,
    "human_audit_used_for_model_selection": False,
    "human_audit_used_for_threshold_tuning": False,
    "human_audit_role": (
        "independent sensitivity validation of taxonomy-derived "
        "other-to-triage mapping"
    ),
    "n_audited": int(row["n_audited"]),
    "raw_interrater_agreement":
        float(row["raw_interrater_agreement"]),
    "cohen_kappa":
        float(row["cohen_kappa"]),
    "n_adjudicated":
        int(row["n_adjudicated"]),
    "hidden_actionable_count":
        int(row["hidden_actionable_count"]),
    "hidden_actionable_fraction":
        float(row["hidden_actionable_fraction"]),
}

(SAFE_REPORT / "human_audit_integration.json").write_text(
    json.dumps(audit_integration, indent=2),
    encoding="utf-8"
)

print("\nFrozen human audit:")
print(audit_summary.to_string(index=False))

# ------------------------------------------------
# Detailed adjudicated audit: validation only
# ------------------------------------------------

detail_candidates = sorted(
    WORKDIR.glob("triage_manual_audit_two_rater_completed*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

final_audit_detail = None

for candidate in detail_candidates:
    try:
        tmp = pd.read_csv(candidate).fillna("")
        if (
            len(tmp) == 200
            and "adjudicated_triage" in tmp.columns
            and tmp["adjudicated_triage"].astype(str)
                .str.strip().ne("").all()
        ):
            final_audit_detail = tmp
            break
    except Exception:
        pass

if final_audit_detail is not None:
    safe_cols = [
        c for c in [
            "global_row_id",
            "source_split",
            "source_row",
            "event",
            "label_11",
            "label_triage",
            "rater1_triage",
            "rater2_triage",
            "adjudicated_triage",
        ]
        if c in final_audit_detail.columns
    ]

    final_audit_detail[safe_cols].to_csv(
        SAFE_TABLE /
        "human_audit_adjudicated_public_safe.csv",
        index=False
    )

# ------------------------------------------------
# MiniLM semantic representation
# ------------------------------------------------

MINILM_ID = "sentence-transformers/all-MiniLM-L6-v2"

embedding_cache = (
    SAFE_CACHE / "minilm_embeddings_all_prepared_normalized.npy"
)

E_all = None

if embedding_cache.is_file():
    candidate = np.load(
        embedding_cache,
        mmap_mode=None
    )
    if candidate.shape[0] == len(combined):
        E_all = np.asarray(candidate, dtype=np.float32)
        print("Reusing SAFE MiniLM embedding cache.")

# Prefer the exact earlier embedding matrix if it has the correct population.
if E_all is None:
    prior_embedding_candidates = list(
        PRIOR_ROOT.rglob("minilm_embeddings_all_prepared.npy")
    )

    for p in prior_embedding_candidates:
        try:
            candidate = np.load(p, mmap_mode=None)
            if candidate.shape[0] == len(combined):
                E_all = np.asarray(candidate, dtype=np.float32)
                print("Reusing prior reviewer-run MiniLM embeddings:", p)
                break
        except Exception:
            continue

encoder_device = "cuda" if torch.cuda.is_available() else "cpu"

semantic_encoder = SentenceTransformer(
    MINILM_ID,
    device=encoder_device
)

if E_all is None:
    print(
        "Encoding the complete prepared CrisisBench population "
        "with frozen MiniLM..."
    )

    E_all = semantic_encoder.encode(
        combined["clean_text"].astype(str).tolist(),
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

# Guarantee unit-length semantic vectors.
norms = np.linalg.norm(E_all, axis=1, keepdims=True)
norms = np.maximum(norms, 1e-12)
E_all = (E_all / norms).astype(np.float32)

np.save(embedding_cache, E_all)

if E_all.shape[0] != len(combined):
    raise RuntimeError(
        "MiniLM embeddings are not aligned with the 78,520 "
        "prepared CrisisBench rows."
    )

print(
    "MiniLM embeddings:",
    E_all.shape,
    "Prepared records:",
    len(combined)
)

SAFE_CONFIG = {
    "method_name":
        "VIRCON-SAFE",
    "method_expansion":
        "Shift-Aware Fusion for Event-Robust Crisis Classification",
    "lexical_expert":
        "TF-IDF + class-balanced logistic regression",
    "semantic_expert":
        "frozen all-MiniLM-L6-v2 + class-balanced logistic regression",
    "gate":
        "class-specific learned lexical-semantic gate",
    "gate_features": [
        "lexical confidence",
        "semantic confidence",
        "lexical normalized entropy",
        "semantic normalized entropy",
        "expert Jensen-Shannon divergence",
        "lexical probability margin",
        "semantic probability margin",
        "expert top-class disagreement",
        "TF-IDF OOV mass",
        "semantic prototype distance",
        "semantic prototype margin",
        "log message length",
    ],
    "consistency_lambda":
        SAFE_CONSISTENCY_LAMBDA,
    "request_risk_multiplier":
        SAFE_REQUEST_RISK,
    "gate_epochs":
        SAFE_GATE_EPOCHS,
    "gate_batch_size":
        SAFE_GATE_BATCH,
    "gate_learning_rate":
        SAFE_GATE_LR,
    "gate_weight_decay":
        SAFE_GATE_WEIGHT_DECAY,
    "primary_protocol":
        "same-corpus event-disjoint",
    "primary_metric":
        "11-class macro-F1",
    "audit_labels_used_for_training":
        False,
}

(SAFE_REPORT / "vircon_safe_config.json").write_text(
    json.dumps(SAFE_CONFIG, indent=2),
    encoding="utf-8"
)


In [ ]:
# ================================================================
# SAFE 2. Expert models, shift features and joint SAFE gate
# ================================================================

import torch.nn as nn
import torch.nn.functional as F

SAFE_TOKEN_PATTERN = r"(?u)\b[a-z][a-z0-9_]{1,}\b"

SAFE_FEATURE_NAMES = [
    "lex_confidence",
    "sem_confidence",
    "lex_entropy",
    "sem_entropy",
    "expert_js_divergence",
    "lex_margin",
    "sem_margin",
    "expert_disagreement",
    "oov_mass",
    "prototype_distance",
    "prototype_margin",
    "log_token_length",
]

CONFIDENCE_FEATURE_INDEX = list(range(8))
FULL_FEATURE_INDEX = list(range(len(SAFE_FEATURE_NAMES)))


def make_safe_tfidf():
    return TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=50000,
        lowercase=False,
        stop_words="english",
        token_pattern=SAFE_TOKEN_PATTERN,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float64,
    )


def normalize_probabilities(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-9, None)
    p /= p.sum(axis=1, keepdims=True)
    return p.astype(np.float32)


def fit_probability_lr(X, y, seed):
    y = np.asarray(y, dtype=object)

    if np.unique(y).size < 2:
        raise RuntimeError(
            "A training fold contains fewer than two classes."
        )

    clf = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        solver="lbfgs",
        max_iter=5000,
        random_state=int(seed),
    )

    clf.fit(X, y)
    return clf


def predict_probability_aligned(model, X, labels):
    labels = list(labels)

    raw = model.predict_proba(X)

    aligned = np.full(
        (raw.shape[0], len(labels)),
        1e-9,
        dtype=np.float64
    )

    lookup = {
        str(label): i
        for i, label in enumerate(labels)
    }

    for source_col, cls in enumerate(model.classes_):
        if str(cls) in lookup:
            aligned[:, lookup[str(cls)]] = raw[:, source_col]

    return normalize_probabilities(aligned)


def entropy_normalized(p):
    p = normalize_probabilities(p)

    h = -np.sum(
        p * np.log(np.clip(p, 1e-12, 1.0)),
        axis=1
    )

    denominator = max(
        math.log(p.shape[1]),
        1e-12
    )

    return h / denominator


def probability_margin(p):
    p = np.asarray(p)
    if p.shape[1] < 2:
        return np.ones(len(p))

    part = np.partition(
        p,
        kth=p.shape[1] - 2,
        axis=1
    )

    top2 = part[:, -2:]
    top2.sort(axis=1)

    return top2[:, 1] - top2[:, 0]


def js_divergence_rows(p, q):
    p = normalize_probabilities(p).astype(np.float64)
    q = normalize_probabilities(q).astype(np.float64)

    m = 0.5 * (p + q)

    kl_pm = np.sum(
        p * (
            np.log(np.clip(p, 1e-12, 1.0))
            - np.log(np.clip(m, 1e-12, 1.0))
        ),
        axis=1
    )

    kl_qm = np.sum(
        q * (
            np.log(np.clip(q, 1e-12, 1.0))
            - np.log(np.clip(m, 1e-12, 1.0))
        ),
        axis=1
    )

    return 0.5 * (kl_pm + kl_qm)


def semantic_centroids(embeddings, labels_y, class_names):
    E = np.asarray(embeddings, dtype=np.float32)
    y = np.asarray(labels_y, dtype=object)

    centroids = np.zeros(
        (len(class_names), E.shape[1]),
        dtype=np.float32
    )

    present = np.zeros(
        len(class_names),
        dtype=bool
    )

    for i, cls in enumerate(class_names):
        mask = y == cls

        if np.any(mask):
            center = E[mask].mean(axis=0)
            nrm = np.linalg.norm(center)

            if nrm > 0:
                center = center / nrm

            centroids[i] = center
            present[i] = True

    return centroids, present


def semantic_prototype_features(
    eval_embeddings,
    centroids,
    present
):
    E = np.asarray(eval_embeddings, dtype=np.float32)

    sims = E @ centroids.T

    if present is not None:
        absent = ~np.asarray(present, dtype=bool)
        if np.any(absent):
            sims[:, absent] = -1.0

    sorted_sims = np.sort(sims, axis=1)

    max_sim = sorted_sims[:, -1]

    if sims.shape[1] > 1:
        proto_margin = (
            sorted_sims[:, -1]
            - sorted_sims[:, -2]
        )
    else:
        proto_margin = np.ones(len(E))

    proto_distance = 1.0 - max_sim

    return (
        proto_distance.astype(np.float32),
        proto_margin.astype(np.float32)
    )


def oov_mass_from_vectorizer(texts, vectorizer):
    analyzer = vectorizer.build_analyzer()
    vocabulary = vectorizer.vocabulary_

    out = np.zeros(
        len(texts),
        dtype=np.float32
    )

    for i, text in enumerate(texts):
        units = analyzer(str(text))

        if not units:
            out[i] = 0.0
            continue

        unseen = sum(
            1 for token in units
            if token not in vocabulary
        )

        out[i] = unseen / len(units)

    return out


def build_gate_features(
    texts,
    vectorizer,
    p_lex,
    p_sem,
    eval_embeddings,
    centroids,
    centroid_present
):
    p_lex = normalize_probabilities(p_lex)
    p_sem = normalize_probabilities(p_sem)

    proto_distance, proto_margin = (
        semantic_prototype_features(
            eval_embeddings,
            centroids,
            centroid_present
        )
    )

    oov = oov_mass_from_vectorizer(
        list(texts),
        vectorizer
    )

    lengths = np.asarray(
        [
            len(str(x).split())
            for x in texts
        ],
        dtype=np.float32
    )

    features = np.column_stack([
        p_lex.max(axis=1),
        p_sem.max(axis=1),
        entropy_normalized(p_lex),
        entropy_normalized(p_sem),
        js_divergence_rows(p_lex, p_sem),
        probability_margin(p_lex),
        probability_margin(p_sem),
        (
            np.argmax(p_lex, axis=1)
            != np.argmax(p_sem, axis=1)
        ).astype(np.float32),
        oov,
        proto_distance,
        proto_margin,
        np.log1p(lengths),
    ]).astype(np.float32)

    return features


def fit_expert_bundle(
    train_part,
    eval_part,
    seed
):
    train_ids = (
        train_part["global_row_id"]
        .astype(int)
        .to_numpy()
    )

    eval_ids = (
        eval_part["global_row_id"]
        .astype(int)
        .to_numpy()
    )

    train_text = (
        train_part["clean_text"]
        .astype(str)
        .tolist()
    )

    eval_text = (
        eval_part["clean_text"]
        .astype(str)
        .tolist()
    )

    y11_train = (
        train_part["label_11"]
        .astype(str)
        .to_numpy(dtype=object)
    )

    y3_train = (
        train_part["label_triage"]
        .astype(str)
        .to_numpy(dtype=object)
    )

    E_train = E_all[train_ids]
    E_eval = E_all[eval_ids]

    vectorizer = make_safe_tfidf()

    X_train = vectorizer.fit_transform(
        train_text
    )

    X_eval = vectorizer.transform(
        eval_text
    )

    lex11 = fit_probability_lr(
        X_train,
        y11_train,
        seed
    )

    lex3 = fit_probability_lr(
        X_train,
        y3_train,
        seed + 1
    )

    sem11 = fit_probability_lr(
        E_train,
        y11_train,
        seed + 2
    )

    sem3 = fit_probability_lr(
        E_train,
        y3_train,
        seed + 3
    )

    p_lex11 = predict_probability_aligned(
        lex11,
        X_eval,
        LABELS
    )

    p_lex3 = predict_probability_aligned(
        lex3,
        X_eval,
        TRIAGE_LABELS
    )

    p_sem11 = predict_probability_aligned(
        sem11,
        E_eval,
        LABELS
    )

    p_sem3 = predict_probability_aligned(
        sem3,
        E_eval,
        TRIAGE_LABELS
    )

    cent11, present11 = semantic_centroids(
        E_train,
        y11_train,
        LABELS
    )

    cent3, present3 = semantic_centroids(
        E_train,
        y3_train,
        TRIAGE_LABELS
    )

    feat11 = build_gate_features(
        eval_text,
        vectorizer,
        p_lex11,
        p_sem11,
        E_eval,
        cent11,
        present11
    )

    feat3 = build_gate_features(
        eval_text,
        vectorizer,
        p_lex3,
        p_sem3,
        E_eval,
        cent3,
        present3
    )

    return {
        "vectorizer": vectorizer,
        "lex11": lex11,
        "lex3": lex3,
        "sem11": sem11,
        "sem3": sem3,
        "cent11": cent11,
        "present11": present11,
        "cent3": cent3,
        "present3": present3,
        "p_lex11": p_lex11,
        "p_sem11": p_sem11,
        "p_lex3": p_lex3,
        "p_sem3": p_sem3,
        "feat11": feat11,
        "feat3": feat3,
    }


def predict_trained_experts(
    bundle,
    texts,
    embeddings_eval
):
    texts = list(map(str, texts))

    X_eval = bundle["vectorizer"].transform(
        texts
    )

    p_lex11 = predict_probability_aligned(
        bundle["lex11"],
        X_eval,
        LABELS
    )

    p_lex3 = predict_probability_aligned(
        bundle["lex3"],
        X_eval,
        TRIAGE_LABELS
    )

    p_sem11 = predict_probability_aligned(
        bundle["sem11"],
        embeddings_eval,
        LABELS
    )

    p_sem3 = predict_probability_aligned(
        bundle["sem3"],
        embeddings_eval,
        TRIAGE_LABELS
    )

    feat11 = build_gate_features(
        texts,
        bundle["vectorizer"],
        p_lex11,
        p_sem11,
        embeddings_eval,
        bundle["cent11"],
        bundle["present11"]
    )

    feat3 = build_gate_features(
        texts,
        bundle["vectorizer"],
        p_lex3,
        p_sem3,
        embeddings_eval,
        bundle["cent3"],
        bundle["present3"]
    )

    return {
        "p_lex11": p_lex11,
        "p_sem11": p_sem11,
        "p_lex3": p_lex3,
        "p_sem3": p_sem3,
        "feat11": feat11,
        "feat3": feat3,
    }


def save_oof_cache(path, oof):
    np.savez_compressed(
        path,
        p_lex11=oof["p_lex11"],
        p_sem11=oof["p_sem11"],
        p_lex3=oof["p_lex3"],
        p_sem3=oof["p_sem3"],
        feat11=oof["feat11"],
        feat3=oof["feat3"],
    )


def load_oof_cache(path):
    z = np.load(path)

    return {
        "p_lex11": z["p_lex11"],
        "p_sem11": z["p_sem11"],
        "p_lex3": z["p_lex3"],
        "p_sem3": z["p_sem3"],
        "feat11": z["feat11"],
        "feat3": z["feat3"],
    }


def build_gate_oof(
    frame,
    splits,
    seed_base,
    cache_path=None
):
    if (
        cache_path is not None
        and Path(cache_path).is_file()
    ):
        print("Reusing gate-OOF cache:", cache_path)
        return load_oof_cache(cache_path)

    n = len(frame)

    out = {
        "p_lex11":
            np.full(
                (n, len(LABELS)),
                np.nan,
                dtype=np.float32
            ),

        "p_sem11":
            np.full(
                (n, len(LABELS)),
                np.nan,
                dtype=np.float32
            ),

        "p_lex3":
            np.full(
                (n, len(TRIAGE_LABELS)),
                np.nan,
                dtype=np.float32
            ),

        "p_sem3":
            np.full(
                (n, len(TRIAGE_LABELS)),
                np.nan,
                dtype=np.float32
            ),

        "feat11":
            np.full(
                (n, len(SAFE_FEATURE_NAMES)),
                np.nan,
                dtype=np.float32
            ),

        "feat3":
            np.full(
                (n, len(SAFE_FEATURE_NAMES)),
                np.nan,
                dtype=np.float32
            ),
    }

    for split_no, (
        train_idx,
        val_idx
    ) in enumerate(splits, start=1):

        print(
            f"  SAFE inner split "
            f"{split_no}/{len(splits)} "
            f"train={len(train_idx):,} "
            f"validation={len(val_idx):,}"
        )

        bundle = fit_expert_bundle(
            frame.iloc[train_idx].copy(),
            frame.iloc[val_idx].copy(),
            seed_base + split_no * 17
        )

        for key in [
            "p_lex11",
            "p_sem11",
            "p_lex3",
            "p_sem3",
            "feat11",
            "feat3",
        ]:
            out[key][val_idx] = bundle[key]

        del bundle
        gc.collect()

    for key, arr in out.items():
        if np.isnan(arr).any():
            raise RuntimeError(
                f"OOF generation incomplete: {key}"
            )

    if cache_path is not None:
        save_oof_cache(
            cache_path,
            out
        )

    return out


def class_weight_tensor(y, n_classes):
    y = np.asarray(y, dtype=np.int64)

    counts = np.bincount(
        y,
        minlength=n_classes
    ).astype(np.float64)

    weights = np.zeros(
        n_classes,
        dtype=np.float32
    )

    present = counts > 0

    weights[present] = (
        len(y)
        / (
            n_classes
            * counts[present]
        )
    )

    weights[present] = np.clip(
        weights[present],
        0.25,
        10.0
    )

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=SAFE_DEVICE
    )


class JointSafeGate(nn.Module):

    def __init__(
        self,
        input_dim,
        n11,
        n3
    ):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 32),
            nn.ReLU(),
        )

        self.head11 = nn.Linear(
            32,
            n11
        )

        self.head3 = nn.Linear(
            32,
            n3
        )

    def forward(self, x):
        h = self.shared(x)

        return (
            torch.sigmoid(
                self.head11(h)
            ),
            torch.sigmoid(
                self.head3(h)
            ),
        )


TRIAGE_MAP_MATRIX = np.zeros(
    (
        len(LABELS),
        len(TRIAGE_LABELS)
    ),
    dtype=np.float32
)

for i, label in enumerate(LABELS):
    triage_label = triage_map(label)
    TRIAGE_MAP_MATRIX[
        i,
        TRIAGE_LABELS.index(triage_label)
    ] = 1.0


def torch_mix(g, p_sem, p_lex):
    p = (
        g * p_sem
        + (1.0 - g) * p_lex
    )

    p = torch.clamp(
        p,
        min=1e-9
    )

    return (
        p
        / p.sum(
            dim=1,
            keepdim=True
        )
    )


def torch_js(p, q):
    p = torch.clamp(
        p,
        min=1e-9
    )

    q = torch.clamp(
        q,
        min=1e-9
    )

    m = 0.5 * (p + q)

    return (
        0.5
        * torch.sum(
            p * (
                torch.log(p)
                - torch.log(m)
            ),
            dim=1
        )
        +
        0.5
        * torch.sum(
            q * (
                torch.log(q)
                - torch.log(m)
            ),
            dim=1
        )
    ).mean()


def gate_feature_indices(mode):
    if mode == "confidence":
        return CONFIDENCE_FEATURE_INDEX

    if mode == "full":
        return FULL_FEATURE_INDEX

    raise ValueError(
        "Unknown gate feature mode: "
        + str(mode)
    )


def combined_gate_input(
    feat11,
    feat3,
    mode
):
    idx = gate_feature_indices(mode)

    return np.hstack([
        np.asarray(
            feat11[:, idx],
            dtype=np.float32
        ),
        np.asarray(
            feat3[:, idx],
            dtype=np.float32
        )
    ])


def fit_joint_safe_gate(
    oof,
    y11_string,
    y3_string,
    feature_mode,
    consistency_lambda,
    request_risk_multiplier,
    seed
):
    torch.manual_seed(seed)

    y11_lookup = {
        x: i
        for i, x in enumerate(LABELS)
    }

    y3_lookup = {
        x: i
        for i, x in enumerate(TRIAGE_LABELS)
    }

    y11 = np.asarray(
        [
            y11_lookup[str(x)]
            for x in y11_string
        ],
        dtype=np.int64
    )

    y3 = np.asarray(
        [
            y3_lookup[str(x)]
            for x in y3_string
        ],
        dtype=np.int64
    )

    X_raw = combined_gate_input(
        oof["feat11"],
        oof["feat3"],
        feature_mode
    )

    scaler = StandardScaler()

    X = scaler.fit_transform(
        X_raw
    ).astype(np.float32)

    model = JointSafeGate(
        input_dim=X.shape[1],
        n11=len(LABELS),
        n3=len(TRIAGE_LABELS),
    ).to(SAFE_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=SAFE_GATE_LR,
        weight_decay=SAFE_GATE_WEIGHT_DECAY,
    )

    w11 = class_weight_tensor(
        y11,
        len(LABELS)
    )

    w3 = class_weight_tensor(
        y3,
        len(TRIAGE_LABELS)
    )

    request_index = (
        TRIAGE_LABELS.index("request")
    )

    w3 = w3.clone()

    w3[request_index] = (
        w3[request_index]
        * float(request_risk_multiplier)
    )

    A = torch.tensor(
        TRIAGE_MAP_MATRIX,
        dtype=torch.float32,
        device=SAFE_DEVICE
    )

    Xt = torch.tensor(
        X,
        dtype=torch.float32
    )

    pL11 = torch.tensor(
        oof["p_lex11"],
        dtype=torch.float32
    )

    pS11 = torch.tensor(
        oof["p_sem11"],
        dtype=torch.float32
    )

    pL3 = torch.tensor(
        oof["p_lex3"],
        dtype=torch.float32
    )

    pS3 = torch.tensor(
        oof["p_sem3"],
        dtype=torch.float32
    )

    y11t = torch.tensor(
        y11,
        dtype=torch.long
    )

    y3t = torch.tensor(
        y3,
        dtype=torch.long
    )

    n = len(X)

    rng = np.random.default_rng(seed)

    history = []

    for epoch in range(
        1,
        SAFE_GATE_EPOCHS + 1
    ):
        model.train()

        order = rng.permutation(n)

        epoch_losses = []
        epoch_l11 = []
        epoch_l3 = []
        epoch_lc = []

        for start in range(
            0,
            n,
            SAFE_GATE_BATCH
        ):
            idx = order[
                start:
                start + SAFE_GATE_BATCH
            ]

            xb = Xt[idx].to(
                SAFE_DEVICE
            )

            pl11 = pL11[idx].to(
                SAFE_DEVICE
            )

            ps11 = pS11[idx].to(
                SAFE_DEVICE
            )

            pl3 = pL3[idx].to(
                SAFE_DEVICE
            )

            ps3 = pS3[idx].to(
                SAFE_DEVICE
            )

            yy11 = y11t[idx].to(
                SAFE_DEVICE
            )

            yy3 = y3t[idx].to(
                SAFE_DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            g11, g3 = model(xb)

            fused11 = torch_mix(
                g11,
                ps11,
                pl11
            )

            fused3 = torch_mix(
                g3,
                ps3,
                pl3
            )

            mapped3 = torch.matmul(
                fused11,
                A
            )

            mapped3 = (
                mapped3
                / mapped3.sum(
                    dim=1,
                    keepdim=True
                )
            )

            loss11 = F.nll_loss(
                torch.log(
                    fused11
                ),
                yy11,
                weight=w11
            )

            loss3 = F.nll_loss(
                torch.log(
                    fused3
                ),
                yy3,
                weight=w3
            )

            consistency = torch_js(
                fused3,
                mapped3
            )

            loss = (
                loss11
                + loss3
                + float(
                    consistency_lambda
                ) * consistency
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            optimizer.step()

            epoch_losses.append(
                float(
                    loss.detach().cpu()
                )
            )

            epoch_l11.append(
                float(
                    loss11.detach().cpu()
                )
            )

            epoch_l3.append(
                float(
                    loss3.detach().cpu()
                )
            )

            epoch_lc.append(
                float(
                    consistency
                    .detach()
                    .cpu()
                )
            )

        history.append({
            "epoch": epoch,
            "loss":
                float(
                    np.mean(
                        epoch_losses
                    )
                ),
            "loss_11class":
                float(
                    np.mean(
                        epoch_l11
                    )
                ),
            "loss_triage":
                float(
                    np.mean(
                        epoch_l3
                    )
                ),
            "loss_consistency":
                float(
                    np.mean(
                        epoch_lc
                    )
                ),
        })

    return {
        "model": model,
        "scaler": scaler,
        "feature_mode": feature_mode,
        "consistency_lambda":
            float(
                consistency_lambda
            ),
        "request_risk_multiplier":
            float(
                request_risk_multiplier
            ),
        "history":
            pd.DataFrame(history),
    }


def apply_joint_safe_gate(
    gate_state,
    expert_output
):
    X_raw = combined_gate_input(
        expert_output["feat11"],
        expert_output["feat3"],
        gate_state["feature_mode"]
    )

    X = (
        gate_state["scaler"]
        .transform(X_raw)
        .astype(np.float32)
    )

    model = gate_state["model"]

    device = next(
        model.parameters()
    ).device

    model.eval()

    with torch.no_grad():
        xb = torch.tensor(
            X,
            dtype=torch.float32,
            device=device
        )

        g11, g3 = model(xb)

        g11 = (
            g11
            .detach()
            .cpu()
            .numpy()
        )

        g3 = (
            g3
            .detach()
            .cpu()
            .numpy()
        )

    p11 = normalize_probabilities(
        g11
        * expert_output["p_sem11"]
        +
        (1.0 - g11)
        * expert_output["p_lex11"]
    )

    p3 = normalize_probabilities(
        g3
        * expert_output["p_sem3"]
        +
        (1.0 - g3)
        * expert_output["p_lex3"]
    )

    return {
        "p11": p11,
        "p3": p3,
        "gate11": g11,
        "gate3": g3,
    }


def weighted_nll_np(
    y_string,
    probabilities,
    labels
):
    labels = list(labels)

    lookup = {
        x: i
        for i, x in enumerate(labels)
    }

    y = np.asarray(
        [
            lookup[str(x)]
            for x in y_string
        ],
        dtype=np.int64
    )

    counts = np.bincount(
        y,
        minlength=len(labels)
    ).astype(np.float64)

    class_weights = np.zeros(
        len(labels),
        dtype=np.float64
    )

    present = counts > 0

    class_weights[present] = (
        len(y)
        / (
            len(labels)
            * counts[present]
        )
    )

    sample_weight = class_weights[y]

    p = normalize_probabilities(
        probabilities
    )

    losses = -np.log(
        np.clip(
            p[
                np.arange(len(y)),
                y
            ],
            1e-12,
            1.0
        )
    )

    return float(
        np.average(
            losses,
            weights=sample_weight
        )
    )


def choose_global_alpha(
    p_lex,
    p_sem,
    y,
    labels
):
    candidates = np.linspace(
        0.0,
        1.0,
        41
    )

    best_alpha = None
    best_loss = np.inf

    for alpha in candidates:
        p = normalize_probabilities(
            alpha * p_sem
            + (1.0 - alpha) * p_lex
        )

        loss = weighted_nll_np(
            y,
            p,
            labels
        )

        if loss < best_loss:
            best_loss = loss
            best_alpha = float(alpha)

    return best_alpha, best_loss


def pred_from_prob(
    p,
    labels
):
    labels = np.asarray(
        labels,
        dtype=object
    )

    return labels[
        np.argmax(
            p,
            axis=1
        )
    ]


def metric_dict(
    y_true,
    y_pred,
    labels
):
    y_true = np.asarray(
        y_true,
        dtype=object
    )

    y_pred = np.asarray(
        y_pred,
        dtype=object
    )

    return {
        "n":
            int(len(y_true)),

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    labels=list(labels),
                    average="macro",
                    zero_division=0,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    y_pred
                )
            ),
    }


def classwise_frame(
    y_true,
    y_pred,
    labels,
    model,
    task,
    protocol
):
    p, r, f, s = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(labels),
            zero_division=0
        )
    )

    return pd.DataFrame({
        "protocol": protocol,
        "task": task,
        "model": model,
        "label": list(labels),
        "precision": p,
        "recall": r,
        "f1": f,
        "support": s,
    })


GATE_SPECS = {
    "confidence_gate": {
        "feature_mode": "confidence",
        "consistency_lambda":
            SAFE_CONSISTENCY_LAMBDA,
        "request_risk_multiplier":
            SAFE_REQUEST_RISK,
    },

    "safe_no_consistency": {
        "feature_mode": "full",
        "consistency_lambda": 0.0,
        "request_risk_multiplier":
            SAFE_REQUEST_RISK,
    },

    "safe_no_risk": {
        "feature_mode": "full",
        "consistency_lambda":
            SAFE_CONSISTENCY_LAMBDA,
        "request_risk_multiplier": 1.0,
    },

    "vircon_safe": {
        "feature_mode": "full",
        "consistency_lambda":
            SAFE_CONSISTENCY_LAMBDA,
        "request_risk_multiplier":
            SAFE_REQUEST_RISK,
    },
}

SAFE_METHODS = [
    "lexical_lr",
    "semantic_minilm_lr",
    "static_50_50",
    "global_alpha",
    "confidence_gate",
    "safe_no_consistency",
    "safe_no_risk",
    "vircon_safe",
]


In [ ]:
# ================================================================
# SAFE 3. Official fixed-split experiment
# ================================================================

fixed_train = (
    combined[
        combined["source_split"]
        .eq("train")
    ]
    .copy()
    .reset_index(drop=True)
)

fixed_test = (
    combined[
        combined["source_split"]
        .eq("test")
    ]
    .copy()
    .reset_index(drop=True)
)

if (
    len(fixed_train) != 61164
    or len(fixed_test) != 17356
):
    raise RuntimeError(
        "Official prepared train/test populations changed."
    )

duplicate_groups, _ = pd.factorize(
    fixed_train["clean_text"].astype(str),
    sort=True
)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SAFE_SEED
)

fixed_splits = [
    (
        np.asarray(tr, dtype=np.int64),
        np.asarray(va, dtype=np.int64)
    )
    for tr, va in splitter.split(
        np.zeros(len(fixed_train)),
        fixed_train["label_11"],
        groups=duplicate_groups
    )
]

fixed_oof_cache = (
    SAFE_CACHE /
    "fixed_gate_oof_predictions.npz"
)

fixed_oof = build_gate_oof(
    fixed_train,
    fixed_splits,
    seed_base=SAFE_SEED + 100,
    cache_path=fixed_oof_cache
)

fixed_gate_states = {}

for no, (
    name,
    spec
) in enumerate(
    GATE_SPECS.items(),
    start=1
):
    print(
        "\nTraining fixed-split gate:",
        name
    )

    state = fit_joint_safe_gate(
        fixed_oof,
        fixed_train["label_11"],
        fixed_train["label_triage"],
        feature_mode=
            spec["feature_mode"],
        consistency_lambda=
            spec["consistency_lambda"],
        request_risk_multiplier=
            spec["request_risk_multiplier"],
        seed=
            SAFE_SEED
            + 1000
            + no
    )

    fixed_gate_states[name] = state

    state["history"].to_csv(
        SAFE_TABLE /
        f"fixed_gate_history_{name}.csv",
        index=False
    )

fixed_alpha11, fixed_alpha11_loss = (
    choose_global_alpha(
        fixed_oof["p_lex11"],
        fixed_oof["p_sem11"],
        fixed_train["label_11"],
        LABELS
    )
)

fixed_alpha3, fixed_alpha3_loss = (
    choose_global_alpha(
        fixed_oof["p_lex3"],
        fixed_oof["p_sem3"],
        fixed_train["label_triage"],
        TRIAGE_LABELS
    )
)

print(
    "Fixed global alpha 11-class:",
    fixed_alpha11
)

print(
    "Fixed global alpha triage:",
    fixed_alpha3
)

pd.DataFrame([
    {
        "task": "11class",
        "alpha_semantic":
            fixed_alpha11,
        "training_oof_weighted_nll":
            fixed_alpha11_loss,
    },
    {
        "task": "direct_triage",
        "alpha_semantic":
            fixed_alpha3,
        "training_oof_weighted_nll":
            fixed_alpha3_loss,
    },
]).to_csv(
    SAFE_TABLE /
    "fixed_global_fusion_alpha.csv",
    index=False
)

print(
    "\nFitting final source experts "
    "on the official training partition..."
)

fixed_bundle = fit_expert_bundle(
    fixed_train,
    fixed_test,
    seed=SAFE_SEED + 2000
)

fixed_probs = {}

fixed_probs["lexical_lr"] = {
    "p11":
        fixed_bundle["p_lex11"],
    "p3":
        fixed_bundle["p_lex3"],
}

fixed_probs["semantic_minilm_lr"] = {
    "p11":
        fixed_bundle["p_sem11"],
    "p3":
        fixed_bundle["p_sem3"],
}

fixed_probs["static_50_50"] = {
    "p11":
        normalize_probabilities(
            0.5
            * fixed_bundle["p_sem11"]
            +
            0.5
            * fixed_bundle["p_lex11"]
        ),

    "p3":
        normalize_probabilities(
            0.5
            * fixed_bundle["p_sem3"]
            +
            0.5
            * fixed_bundle["p_lex3"]
        ),
}

fixed_probs["global_alpha"] = {
    "p11":
        normalize_probabilities(
            fixed_alpha11
            * fixed_bundle["p_sem11"]
            +
            (1.0 - fixed_alpha11)
            * fixed_bundle["p_lex11"]
        ),

    "p3":
        normalize_probabilities(
            fixed_alpha3
            * fixed_bundle["p_sem3"]
            +
            (1.0 - fixed_alpha3)
            * fixed_bundle["p_lex3"]
        ),
}

for name, state in fixed_gate_states.items():

    gate_result = apply_joint_safe_gate(
        state,
        fixed_bundle
    )

    fixed_probs[name] = {
        "p11": gate_result["p11"],
        "p3": gate_result["p3"],
    }

fixed_pred = fixed_test[
    [
        "global_row_id",
        "source_split",
        "source_row",
        "event",
        "label_11",
        "label_triage",
    ]
].copy()

for method in SAFE_METHODS:

    fixed_pred[
        f"pred_11__{method}"
    ] = pred_from_prob(
        fixed_probs[method]["p11"],
        LABELS
    )

    fixed_pred[
        f"pred_triage__{method}"
    ] = pred_from_prob(
        fixed_probs[method]["p3"],
        TRIAGE_LABELS
    )

# Add the previously completed fine-tuned DistilBERT comparator.
dist_fixed_path = (
    TABLE_DIR /
    "distilbert_fixed_split_predictions.csv"
)

if dist_fixed_path.is_file():

    dfix = pd.read_csv(
        dist_fixed_path
    )

    rename_map = {
        "pred_11class__distilbert_ft":
            "pred_11__distilbert_ft",

        "pred_direct_triage__distilbert_ft":
            "pred_triage__distilbert_ft",
    }

    dfix = dfix.rename(
        columns=rename_map
    )

    keep = [
        c for c in [
            "global_row_id",
            "pred_11__distilbert_ft",
            "pred_triage__distilbert_ft",
        ]
        if c in dfix.columns
    ]

    fixed_pred = fixed_pred.merge(
        dfix[keep],
        on="global_row_id",
        how="left",
        validate="one_to_one"
    )

fixed_pred.to_csv(
    SAFE_TABLE /
    "safe_fixed_split_predictions.csv",
    index=False
)

fixed_summary_rows = []
fixed_per_class = []

fixed_summary_methods = (
    SAFE_METHODS
    + (
        ["distilbert_ft"]
        if (
            "pred_11__distilbert_ft"
            in fixed_pred.columns
        )
        else []
    )
)

for method in fixed_summary_methods:

    c11 = f"pred_11__{method}"
    c3 = f"pred_triage__{method}"

    if c11 in fixed_pred.columns:

        fixed_summary_rows.append({
            "protocol":
                "official_fixed_split",
            "task":
                "11class",
            "model":
                method,
            **metric_dict(
                fixed_pred["label_11"],
                fixed_pred[c11],
                LABELS
            )
        })

        fixed_per_class.append(
            classwise_frame(
                fixed_pred["label_11"],
                fixed_pred[c11],
                LABELS,
                method,
                "11class",
                "official_fixed_split"
            )
        )

    if c3 in fixed_pred.columns:

        fixed_summary_rows.append({
            "protocol":
                "official_fixed_split",
            "task":
                "direct_triage",
            "model":
                method,
            **metric_dict(
                fixed_pred["label_triage"],
                fixed_pred[c3],
                TRIAGE_LABELS
            )
        })

        fixed_per_class.append(
            classwise_frame(
                fixed_pred["label_triage"],
                fixed_pred[c3],
                TRIAGE_LABELS,
                method,
                "direct_triage",
                "official_fixed_split"
            )
        )

fixed_summary = pd.DataFrame(
    fixed_summary_rows
)

fixed_summary.to_csv(
    SAFE_TABLE /
    "safe_fixed_split_summary.csv",
    index=False
)

pd.concat(
    fixed_per_class,
    ignore_index=True
).to_csv(
    SAFE_TABLE /
    "safe_fixed_split_per_class.csv",
    index=False
)

print("\nSAFE FIXED-SPLIT SUMMARY")
display(fixed_summary)

# ------------------------------------------------
# Save source-model components needed for external transfer/deployment.
# ------------------------------------------------

SOURCE_MODEL_DIR = (
    SAFE_MODEL /
    "official_source_model"
)

SOURCE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    fixed_bundle["vectorizer"],
    SOURCE_MODEL_DIR /
    "tfidf_vectorizer.joblib"
)

joblib.dump(
    fixed_bundle["lex11"],
    SOURCE_MODEL_DIR /
    "lexical_11class_lr.joblib"
)

joblib.dump(
    fixed_bundle["lex3"],
    SOURCE_MODEL_DIR /
    "lexical_triage_lr.joblib"
)

joblib.dump(
    fixed_bundle["sem11"],
    SOURCE_MODEL_DIR /
    "semantic_11class_lr.joblib"
)

joblib.dump(
    fixed_bundle["sem3"],
    SOURCE_MODEL_DIR /
    "semantic_triage_lr.joblib"
)

np.savez_compressed(
    SOURCE_MODEL_DIR /
    "semantic_prototypes.npz",
    cent11=fixed_bundle["cent11"],
    present11=fixed_bundle["present11"],
    cent3=fixed_bundle["cent3"],
    present3=fixed_bundle["present3"],
)

for name, state in fixed_gate_states.items():

    torch.save(
        {
            "state_dict":
                state["model"]
                .state_dict(),

            "input_dim":
                next(
                    state["model"]
                    .parameters()
                ).shape[-1]
                if False
                else combined_gate_input(
                    fixed_oof["feat11"],
                    fixed_oof["feat3"],
                    state["feature_mode"]
                ).shape[1],

            "feature_mode":
                state["feature_mode"],

            "consistency_lambda":
                state["consistency_lambda"],

            "request_risk_multiplier":
                state["request_risk_multiplier"],
        },
        SOURCE_MODEL_DIR /
        f"{name}_gate.pt"
    )

    joblib.dump(
        state["scaler"],
        SOURCE_MODEL_DIR /
        f"{name}_scaler.joblib"
    )


In [ ]:
# ================================================================
# SAFE 4. Nested event-disjoint experiment — PRIMARY EVALUATION
# ================================================================

event_fold_outputs = []

outer_folds = sorted(
    strict_df["fold"]
    .astype(int)
    .unique()
)

print(
    "Outer event folds:",
    outer_folds
)

for outer_fold in outer_folds:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "SAFE OUTER EVENT FOLD",
        outer_fold
    )

    print(
        "=" * 70
    )

    test_mask = (
        strict_df["fold"]
        .astype(int)
        .eq(int(outer_fold))
    )

    outer_train = (
        strict_df.loc[
            ~test_mask
        ]
        .copy()
        .reset_index(drop=True)
    )

    outer_test = (
        strict_df.loc[
            test_mask
        ]
        .copy()
        .reset_index(drop=True)
    )

    checkpoint = (
        SAFE_CHECKPOINT /
        f"safe_event_outer_fold_{outer_fold}.csv"
    )

    required_pred_cols = [
        f"pred_11__{m}"
        for m in SAFE_METHODS
    ] + [
        f"pred_triage__{m}"
        for m in SAFE_METHODS
    ]

    if checkpoint.is_file():

        ck = pd.read_csv(
            checkpoint
        )

        if (
            len(ck) == len(outer_test)
            and all(
                c in ck.columns
                for c in required_pred_cols
            )
        ):
            print(
                "Reusing completed outer-fold checkpoint."
            )

            event_fold_outputs.append(
                ck
            )
            continue

    # ------------------------------------------------
    # Inner event-held-out OOF predictions.
    # The outer test event groups are never used here.
    # ------------------------------------------------

    inner_folds = sorted(
        outer_train["fold"]
        .astype(int)
        .unique()
    )

    inner_splits = []

    for inner_fold in inner_folds:

        va = np.flatnonzero(
            outer_train["fold"]
            .astype(int)
            .eq(int(inner_fold))
            .to_numpy()
        )

        tr = np.flatnonzero(
            ~outer_train["fold"]
            .astype(int)
            .eq(int(inner_fold))
            .to_numpy()
        )

        if len(tr) and len(va):
            inner_splits.append(
                (
                    tr.astype(np.int64),
                    va.astype(np.int64)
                )
            )

    if len(inner_splits) < 2:
        raise RuntimeError(
            "Insufficient event-aware inner splits."
        )

    oof_cache = (
        SAFE_CACHE /
        f"event_outer_{outer_fold}_gate_oof.npz"
    )

    event_oof = build_gate_oof(
        outer_train,
        inner_splits,
        seed_base=
            SAFE_SEED
            + outer_fold * 1000,
        cache_path=oof_cache
    )

    gate_states = {}

    for no, (
        name,
        spec
    ) in enumerate(
        GATE_SPECS.items(),
        start=1
    ):

        print(
            "Training",
            name,
            "for outer fold",
            outer_fold
        )

        state = fit_joint_safe_gate(
            event_oof,
            outer_train["label_11"],
            outer_train["label_triage"],
            feature_mode=
                spec["feature_mode"],
            consistency_lambda=
                spec["consistency_lambda"],
            request_risk_multiplier=
                spec[
                    "request_risk_multiplier"
                ],
            seed=
                SAFE_SEED
                + outer_fold * 100
                + no
        )

        gate_states[name] = state

        state["history"].to_csv(
            SAFE_TABLE /
            (
                f"event_outer_{outer_fold}"
                f"_gate_history_{name}.csv"
            ),
            index=False
        )

    alpha11, alpha11_loss = (
        choose_global_alpha(
            event_oof["p_lex11"],
            event_oof["p_sem11"],
            outer_train["label_11"],
            LABELS
        )
    )

    alpha3, alpha3_loss = (
        choose_global_alpha(
            event_oof["p_lex3"],
            event_oof["p_sem3"],
            outer_train["label_triage"],
            TRIAGE_LABELS
        )
    )

    pd.DataFrame([
        {
            "outer_fold":
                outer_fold,
            "task":
                "11class",
            "alpha_semantic":
                alpha11,
            "training_oof_weighted_nll":
                alpha11_loss,
        },
        {
            "outer_fold":
                outer_fold,
            "task":
                "direct_triage",
            "alpha_semantic":
                alpha3,
            "training_oof_weighted_nll":
                alpha3_loss,
        },
    ]).to_csv(
        SAFE_TABLE /
        (
            f"event_outer_{outer_fold}"
            "_global_alpha.csv"
        ),
        index=False
    )

    print(
        "Fitting source experts "
        "on all outer-training events..."
    )

    outer_bundle = fit_expert_bundle(
        outer_train,
        outer_test,
        seed=
            SAFE_SEED
            + 5000
            + outer_fold
    )

    fold_probs = {}

    fold_probs["lexical_lr"] = {
        "p11":
            outer_bundle["p_lex11"],
        "p3":
            outer_bundle["p_lex3"],
    }

    fold_probs["semantic_minilm_lr"] = {
        "p11":
            outer_bundle["p_sem11"],
        "p3":
            outer_bundle["p_sem3"],
    }

    fold_probs["static_50_50"] = {
        "p11":
            normalize_probabilities(
                0.5
                * outer_bundle["p_sem11"]
                +
                0.5
                * outer_bundle["p_lex11"]
            ),

        "p3":
            normalize_probabilities(
                0.5
                * outer_bundle["p_sem3"]
                +
                0.5
                * outer_bundle["p_lex3"]
            ),
    }

    fold_probs["global_alpha"] = {
        "p11":
            normalize_probabilities(
                alpha11
                * outer_bundle["p_sem11"]
                +
                (1.0 - alpha11)
                * outer_bundle["p_lex11"]
            ),

        "p3":
            normalize_probabilities(
                alpha3
                * outer_bundle["p_sem3"]
                +
                (1.0 - alpha3)
                * outer_bundle["p_lex3"]
            ),
    }

    for name, state in gate_states.items():

        result = apply_joint_safe_gate(
            state,
            outer_bundle
        )

        fold_probs[name] = {
            "p11": result["p11"],
            "p3": result["p3"],
        }

    fold_pred = outer_test[
        [
            "global_row_id",
            "source_split",
            "source_row",
            "canonical_event",
            "fold",
            "label_11",
            "label_triage",
        ]
    ].copy()

    for method in SAFE_METHODS:

        fold_pred[
            f"pred_11__{method}"
        ] = pred_from_prob(
            fold_probs[method]["p11"],
            LABELS
        )

        fold_pred[
            f"pred_triage__{method}"
        ] = pred_from_prob(
            fold_probs[method]["p3"],
            TRIAGE_LABELS
        )

    fold_pred.to_csv(
        checkpoint,
        index=False
    )

    event_fold_outputs.append(
        fold_pred
    )

    del (
        event_oof,
        gate_states,
        outer_bundle,
        fold_probs
    )

    gc.collect()

event_pred = (
    pd.concat(
        event_fold_outputs,
        ignore_index=True
    )
    .sort_values(
        "global_row_id"
    )
    .reset_index(drop=True)
)

if len(event_pred) != len(strict_df):
    raise RuntimeError(
        "Event-disjoint SAFE OOF population is incomplete."
    )

if event_pred["global_row_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate SAFE event-disjoint predictions detected."
    )

# Add the pre-computed fine-tuned DistilBERT comparator.
dist_event_path = (
    TABLE_DIR /
    "distilbert_event_disjoint_oof_predictions.csv"
)

if dist_event_path.is_file():

    dev = pd.read_csv(
        dist_event_path
    )

    dev = dev.rename(
        columns={
            "pred_11class__distilbert_ft":
                "pred_11__distilbert_ft",

            "pred_direct_triage__distilbert_ft":
                "pred_triage__distilbert_ft",
        }
    )

    keep = [
        c for c in [
            "global_row_id",
            "pred_11__distilbert_ft",
            "pred_triage__distilbert_ft",
        ]
        if c in dev.columns
    ]

    event_pred = event_pred.merge(
        dev[keep],
        on="global_row_id",
        how="left",
        validate="one_to_one"
    )

event_pred.to_csv(
    SAFE_TABLE /
    "safe_event_disjoint_oof_predictions.csv",
    index=False
)

event_methods = (
    SAFE_METHODS
    + (
        ["distilbert_ft"]
        if (
            "pred_11__distilbert_ft"
            in event_pred.columns
        )
        else []
    )
)

event_summary_rows = []
event_per_class_frames = []

for method in event_methods:

    c11 = f"pred_11__{method}"
    c3 = f"pred_triage__{method}"

    if c11 in event_pred.columns:

        event_summary_rows.append({
            "protocol":
                "event_disjoint_same_corpus",
            "task":
                "11class",
            "model":
                method,
            **metric_dict(
                event_pred["label_11"],
                event_pred[c11],
                LABELS
            )
        })

        event_per_class_frames.append(
            classwise_frame(
                event_pred["label_11"],
                event_pred[c11],
                LABELS,
                method,
                "11class",
                "event_disjoint_same_corpus"
            )
        )

    if c3 in event_pred.columns:

        event_summary_rows.append({
            "protocol":
                "event_disjoint_same_corpus",
            "task":
                "direct_triage",
            "model":
                method,
            **metric_dict(
                event_pred["label_triage"],
                event_pred[c3],
                TRIAGE_LABELS
            )
        })

        event_per_class_frames.append(
            classwise_frame(
                event_pred["label_triage"],
                event_pred[c3],
                TRIAGE_LABELS,
                method,
                "direct_triage",
                "event_disjoint_same_corpus"
            )
        )

event_summary = pd.DataFrame(
    event_summary_rows
)

event_per_class = pd.concat(
    event_per_class_frames,
    ignore_index=True
)

event_summary.to_csv(
    SAFE_TABLE /
    "safe_event_disjoint_summary.csv",
    index=False
)

event_per_class.to_csv(
    SAFE_TABLE /
    "safe_event_disjoint_per_class.csv",
    index=False
)

print(
    "\nPRIMARY EVENT-DISJOINT RESULTS"
)

display(
    event_summary.sort_values(
        ["task", "macro_f1"],
        ascending=[True, False]
    )
)

print(
    "\nEVENT-DISJOINT REQUEST CLASS"
)

display(
    event_per_class[
        (
            event_per_class["task"]
            == "direct_triage"
        )
        &
        (
            event_per_class["label"]
            == "request"
        )
    ].sort_values(
        "recall",
        ascending=False
    )
)


In [ ]:
# ================================================================
# SAFE 5. Independent HumAID transfer evaluation
# ================================================================

humaid_path = (
    TABLE_DIR /
    "humaid_external_predictions.csv"
)

external_summary = pd.DataFrame()
external_eventwise = pd.DataFrame()

if not humaid_path.is_file():

    print(
        "HumAID external file was not found. "
        "External SAFE evaluation is skipped."
    )

else:

    huma = pd.read_csv(
        humaid_path
    )

    if "clean_text" not in huma.columns:
        if "raw_text" not in huma.columns:
            raise KeyError(
                "HumAID predictions file has no clean_text/raw_text."
            )

        huma["clean_text"] = (
            huma["raw_text"]
            .map(normalize_text)
        )

    huma_text = (
        huma["clean_text"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    external_embedding_cache = (
        SAFE_CACHE /
        "humaid_minilm_embeddings_normalized.npy"
    )

    if external_embedding_cache.is_file():

        E_external = np.load(
            external_embedding_cache
        ).astype(np.float32)

    else:

        E_external = (
            semantic_encoder.encode(
                huma_text,
                batch_size=256,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
            )
            .astype(np.float32)
        )

        np.save(
            external_embedding_cache,
            E_external
        )

    if len(E_external) != len(huma):
        raise RuntimeError(
            "HumAID embedding count mismatch."
        )

    ext_base = predict_trained_experts(
        fixed_bundle,
        huma_text,
        E_external
    )

    ext_probs = {}

    ext_probs["lexical_lr"] = {
        "p11":
            ext_base["p_lex11"],
        "p3":
            ext_base["p_lex3"],
    }

    ext_probs["semantic_minilm_lr"] = {
        "p11":
            ext_base["p_sem11"],
        "p3":
            ext_base["p_sem3"],
    }

    ext_probs["static_50_50"] = {
        "p11":
            normalize_probabilities(
                0.5
                * ext_base["p_sem11"]
                +
                0.5
                * ext_base["p_lex11"]
            ),

        "p3":
            normalize_probabilities(
                0.5
                * ext_base["p_sem3"]
                +
                0.5
                * ext_base["p_lex3"]
            ),
    }

    ext_probs["global_alpha"] = {
        "p11":
            normalize_probabilities(
                fixed_alpha11
                * ext_base["p_sem11"]
                +
                (1.0 - fixed_alpha11)
                * ext_base["p_lex11"]
            ),

        "p3":
            normalize_probabilities(
                fixed_alpha3
                * ext_base["p_sem3"]
                +
                (1.0 - fixed_alpha3)
                * ext_base["p_lex3"]
            ),
    }

    for name, state in fixed_gate_states.items():

        gate_out = apply_joint_safe_gate(
            state,
            ext_base
        )

        ext_probs[name] = {
            "p11": gate_out["p11"],
            "p3": gate_out["p3"],
        }

    ext_pred = huma[
        [
            c for c in [
                "external_event",
                "external_label_raw",
                "label_harmonized",
                "label_triage",
            ]
            if c in huma.columns
        ]
    ].copy()

    for method in SAFE_METHODS:

        ext_pred[
            f"pred_11__{method}"
        ] = pred_from_prob(
            ext_probs[method]["p11"],
            LABELS
        )

        ext_pred[
            f"pred_triage__{method}"
        ] = pred_from_prob(
            ext_probs[method]["p3"],
            TRIAGE_LABELS
        )

    # Retain the already-completed fine-tuned DistilBERT external comparator.
    if (
        "pred_11class__distilbert_ft"
        in huma.columns
    ):
        ext_pred[
            "pred_11__distilbert_ft"
        ] = huma[
            "pred_11class__distilbert_ft"
        ].to_numpy()

    if (
        "pred_triage__distilbert_ft"
        in huma.columns
    ):
        ext_pred[
            "pred_triage__distilbert_ft"
        ] = huma[
            "pred_triage__distilbert_ft"
        ].to_numpy()

    ext_pred.to_csv(
        SAFE_TABLE /
        "safe_humaid_external_predictions.csv",
        index=False
    )

    shared_labels = [
        "caution_and_advice",
        "displaced_and_evacuations",
        "infrastructure_and_utilities_damage",
        "injured_or_dead_people",
        "missing_and_found_people",
        "not_humanitarian",
        "requests_or_needs",
        "donation_and_volunteering",
        "sympathy_and_support",
    ]

    external_methods = (
        SAFE_METHODS
        + (
            ["distilbert_ft"]
            if (
                "pred_triage__distilbert_ft"
                in ext_pred.columns
            )
            else []
        )
    )

    ext_rows = []

    for method in external_methods:

        triage_col = (
            f"pred_triage__{method}"
        )

        if triage_col in ext_pred.columns:

            ext_rows.append({
                "external_task":
                    "triage_3class",
                "model":
                    method,
                **metric_dict(
                    ext_pred["label_triage"],
                    ext_pred[triage_col],
                    TRIAGE_LABELS
                )
            })

        class_col = (
            f"pred_11__{method}"
        )

        if (
            class_col in ext_pred.columns
            and "label_harmonized"
            in ext_pred.columns
        ):

            shared = ext_pred[
                ext_pred[
                    "label_harmonized"
                ].isin(shared_labels)
            ].copy()

            if len(shared):

                metrics = metric_dict(
                    shared[
                        "label_harmonized"
                    ],
                    shared[class_col],
                    shared_labels
                )

                metrics[
                    "unsupported_prediction_rate"
                ] = float(
                    (
                        ~shared[class_col]
                        .isin(shared_labels)
                    ).mean()
                )

                ext_rows.append({
                    "external_task":
                        "shared_9class",
                    "model":
                        method,
                    **metrics
                })

    external_summary = pd.DataFrame(
        ext_rows
    )

    external_summary.to_csv(
        SAFE_TABLE /
        "safe_humaid_external_summary.csv",
        index=False
    )

    event_rows = []

    if "external_event" in ext_pred.columns:

        for event_name, g in (
            ext_pred
            .groupby("external_event")
        ):

            for method in external_methods:

                c = (
                    f"pred_triage__{method}"
                )

                if c not in g.columns:
                    continue

                event_rows.append({
                    "external_event":
                        event_name,
                    "task":
                        "triage_3class",
                    "model":
                        method,
                    **metric_dict(
                        g["label_triage"],
                        g[c],
                        TRIAGE_LABELS
                    )
                })

    external_eventwise = pd.DataFrame(
        event_rows
    )

    external_eventwise.to_csv(
        SAFE_TABLE /
        "safe_humaid_external_eventwise.csv",
        index=False
    )

    print(
        "\nHUMAID EXTERNAL TRANSFER"
    )

    display(
        external_summary.sort_values(
            [
                "external_task",
                "macro_f1"
            ],
            ascending=[
                True,
                False
            ]
        )
    )


In [ ]:
# ================================================================
# SAFE 6. Statistical inference, novelty evidence,
# deployment benchmark and final package
# ================================================================

def macro_f1_fixed(
    y_true,
    y_pred,
    labels
):
    return float(
        f1_score(
            y_true,
            y_pred,
            labels=list(labels),
            average="macro",
            zero_division=0,
        )
    )


def paired_stratified_bootstrap(
    y_true,
    pred_a,
    pred_b,
    labels,
    n_boot,
    seed
):
    y_true = np.asarray(
        y_true,
        dtype=object
    )

    pred_a = np.asarray(
        pred_a,
        dtype=object
    )

    pred_b = np.asarray(
        pred_b,
        dtype=object
    )

    rng = np.random.default_rng(
        seed
    )

    strata = {
        label:
            np.flatnonzero(
                y_true == label
            )
        for label in labels
    }

    diffs = np.empty(
        n_boot,
        dtype=np.float64
    )

    for b in range(n_boot):

        sampled = []

        for label in labels:

            idx = strata[label]

            if len(idx) == 0:
                continue

            sampled.append(
                rng.choice(
                    idx,
                    size=len(idx),
                    replace=True
                )
            )

        sampled = np.concatenate(
            sampled
        )

        diffs[b] = (
            macro_f1_fixed(
                y_true[sampled],
                pred_a[sampled],
                labels
            )
            -
            macro_f1_fixed(
                y_true[sampled],
                pred_b[sampled],
                labels
            )
        )

    point = (
        macro_f1_fixed(
            y_true,
            pred_a,
            labels
        )
        -
        macro_f1_fixed(
            y_true,
            pred_b,
            labels
        )
    )

    low, high = np.percentile(
        diffs,
        [2.5, 97.5]
    )

    left = (
        np.sum(diffs <= 0.0) + 1
    ) / (
        n_boot + 1
    )

    right = (
        np.sum(diffs >= 0.0) + 1
    ) / (
        n_boot + 1
    )

    p = min(
        1.0,
        2.0 * min(left, right)
    )

    return (
        float(point),
        float(low),
        float(high),
        float(p)
    )


def event_cluster_bootstrap(
    frame,
    true_col,
    pred_a_col,
    pred_b_col,
    labels,
    n_boot,
    seed
):
    rng = np.random.default_rng(
        seed
    )

    events = (
        frame["canonical_event"]
        .astype(str)
        .unique()
    )

    cms_a = []
    cms_b = []

    for event in events:

        g = frame[
            frame[
                "canonical_event"
            ].astype(str).eq(
                str(event)
            )
        ]

        cms_a.append(
            confusion_matrix(
                g[true_col],
                g[pred_a_col],
                labels=list(labels)
            )
        )

        cms_b.append(
            confusion_matrix(
                g[true_col],
                g[pred_b_col],
                labels=list(labels)
            )
        )

    cms_a = np.asarray(
        cms_a,
        dtype=np.float64
    )

    cms_b = np.asarray(
        cms_b,
        dtype=np.float64
    )

    def f1_from_cm(cm):

        tp = np.diag(cm)
        fp = cm.sum(axis=0) - tp
        fn = cm.sum(axis=1) - tp

        denom = (
            2 * tp + fp + fn
        )

        f1 = np.divide(
            2 * tp,
            denom,
            out=np.zeros_like(
                tp,
                dtype=float
            ),
            where=denom > 0
        )

        return float(
            np.mean(f1)
        )

    point = (
        f1_from_cm(
            cms_a.sum(axis=0)
        )
        -
        f1_from_cm(
            cms_b.sum(axis=0)
        )
    )

    diffs = np.empty(
        n_boot,
        dtype=np.float64
    )

    n_events = len(events)

    for b in range(n_boot):

        idx = rng.integers(
            0,
            n_events,
            n_events
        )

        diffs[b] = (
            f1_from_cm(
                cms_a[idx]
                .sum(axis=0)
            )
            -
            f1_from_cm(
                cms_b[idx]
                .sum(axis=0)
            )
        )

    low, high = np.percentile(
        diffs,
        [2.5, 97.5]
    )

    left = (
        np.sum(diffs <= 0.0) + 1
    ) / (
        n_boot + 1
    )

    right = (
        np.sum(diffs >= 0.0) + 1
    ) / (
        n_boot + 1
    )

    p = min(
        1.0,
        2.0 * min(left, right)
    )

    return (
        float(point),
        float(low),
        float(high),
        float(p)
    )


def holm_adjust(values):
    p = np.asarray(
        values,
        dtype=float
    )

    n = len(p)

    order = np.argsort(p)

    out = np.empty(
        n,
        dtype=float
    )

    running = 0.0

    for rank, idx in enumerate(order):

        candidate = min(
            1.0,
            (
                n - rank
            ) * p[idx]
        )

        running = max(
            running,
            candidate
        )

        out[idx] = running

    return out


comparison_models = [
    "lexical_lr",
    "semantic_minilm_lr",
    "static_50_50",
    "global_alpha",
    "confidence_gate",
]

if (
    "pred_11__distilbert_ft"
    in event_pred.columns
):
    comparison_models.append(
        "distilbert_ft"
    )

comparison_rows = []

# ------------------------------------------------
# Official fixed-split paired message bootstrap
# ------------------------------------------------

for task, true_col, prefix, labels in [
    (
        "11class",
        "label_11",
        "pred_11__",
        LABELS
    ),
    (
        "direct_triage",
        "label_triage",
        "pred_triage__",
        TRIAGE_LABELS
    ),
]:

    safe_col = (
        prefix
        + "vircon_safe"
    )

    if safe_col not in fixed_pred.columns:
        continue

    for model in comparison_models:

        base_col = prefix + model

        if base_col not in fixed_pred.columns:
            continue

        eff, lo, hi, p = (
            paired_stratified_bootstrap(
                fixed_pred[true_col],
                fixed_pred[safe_col],
                fixed_pred[base_col],
                labels,
                SAFE_BOOTSTRAP,
                SAFE_SEED
                + len(comparison_rows)
            )
        )

        comparison_rows.append({
            "protocol":
                "official_fixed_split",
            "task":
                task,
            "model_a":
                "vircon_safe",
            "model_b":
                model,
            "macro_f1_difference_a_minus_b":
                eff,
            "ci95_low":
                lo,
            "ci95_high":
                hi,
            "p_two_sided":
                p,
            "resampling_unit":
                "message_within_true_class",
            "n_bootstrap":
                SAFE_BOOTSTRAP,
        })

# ------------------------------------------------
# Primary event-cluster bootstrap
# ------------------------------------------------

for task, true_col, prefix, labels in [
    (
        "11class",
        "label_11",
        "pred_11__",
        LABELS
    ),
    (
        "direct_triage",
        "label_triage",
        "pred_triage__",
        TRIAGE_LABELS
    ),
]:

    safe_col = (
        prefix
        + "vircon_safe"
    )

    if safe_col not in event_pred.columns:
        continue

    for model in comparison_models:

        base_col = prefix + model

        if base_col not in event_pred.columns:
            continue

        eff, lo, hi, p = (
            event_cluster_bootstrap(
                event_pred,
                true_col,
                safe_col,
                base_col,
                labels,
                SAFE_BOOTSTRAP,
                SAFE_SEED
                + 1000
                + len(comparison_rows)
            )
        )

        comparison_rows.append({
            "protocol":
                "event_disjoint_same_corpus",
            "task":
                task,
            "model_a":
                "vircon_safe",
            "model_b":
                model,
            "macro_f1_difference_a_minus_b":
                eff,
            "ci95_low":
                lo,
            "ci95_high":
                hi,
            "p_two_sided":
                p,
            "resampling_unit":
                "canonical_event_cluster",
            "n_bootstrap":
                SAFE_BOOTSTRAP,
        })

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df[
    "p_holm_within_protocol_task"
] = np.nan

for (
    protocol,
    task
), idx in (
    comparison_df
    .groupby(
        ["protocol", "task"]
    )
    .groups
    .items()
):

    idx = list(idx)

    comparison_df.loc[
        idx,
        "p_holm_within_protocol_task"
    ] = holm_adjust(
        comparison_df.loc[
            idx,
            "p_two_sided"
        ].to_numpy()
    )

comparison_df.to_csv(
    SAFE_TABLE /
    "safe_primary_effect_estimates_and_cis.csv",
    index=False
)

print(
    "\nPRIMARY SAFE EFFECT ESTIMATES"
)

display(comparison_df)

# ------------------------------------------------
# Novelty evidence checks — descriptive, not a guarantee.
# ------------------------------------------------

def summary_macro(
    frame,
    task,
    model
):
    x = frame[
        (
            frame["task"] == task
        )
        &
        (
            frame["model"] == model
        )
    ]

    if x.empty:
        return np.nan

    return float(
        x.iloc[0]["macro_f1"]
    )


safe_event_11 = summary_macro(
    event_summary,
    "11class",
    "vircon_safe"
)

safe_event_3 = summary_macro(
    event_summary,
    "direct_triage",
    "vircon_safe"
)

criteria = []

for comparator in [
    "lexical_lr",
    "semantic_minilm_lr",
    "static_50_50",
    "global_alpha",
    "confidence_gate",
    "safe_no_consistency",
    "safe_no_risk",
]:

    baseline = summary_macro(
        event_summary,
        "11class",
        comparator
    )

    criteria.append({
        "criterion":
            (
                "Event-disjoint 11-class macro-F1: "
                f"VIRCON-SAFE > {comparator}"
            ),
        "vircon_safe_value":
            safe_event_11,
        "comparator_value":
            baseline,
        "pass":
            bool(
                np.isfinite(baseline)
                and safe_event_11 > baseline
            ),
    })

request_rows = event_per_class[
    (
        event_per_class["task"]
        == "direct_triage"
    )
    &
    (
        event_per_class["label"]
        == "request"
    )
]

safe_request = request_rows[
    request_rows["model"]
    == "vircon_safe"
]

safe_request_recall = (
    float(
        safe_request.iloc[0]["recall"]
    )
    if len(safe_request)
    else np.nan
)

criteria.append({
    "criterion":
        (
            "Event-disjoint request recall "
            "for VIRCON-SAFE is greater than zero"
        ),
    "vircon_safe_value":
        safe_request_recall,
    "comparator_value":
        0.0,
    "pass":
        bool(
            np.isfinite(
                safe_request_recall
            )
            and safe_request_recall > 0
        ),
})

criteria.append({
    "criterion":
        "Frozen two-rater human audit completed",
    "vircon_safe_value":
        int(
            audit_summary.iloc[0][
                "n_audited"
            ]
        ),
    "comparator_value":
        200,
    "pass":
        bool(
            int(
                audit_summary.iloc[0][
                    "n_audited"
                ]
            )
            == 200
        ),
})

novelty_df = pd.DataFrame(
    criteria
)

novelty_df.to_csv(
    SAFE_TABLE /
    "novelty_evidence_check.csv",
    index=False
)

print(
    "\nNOVELTY EVIDENCE CHECK"
)

display(novelty_df)

# ------------------------------------------------
# Deployment benchmark on the same current CPU runtime.
# ------------------------------------------------

DEPLOY_DIR = (
    SAFE_MODEL /
    "vircon_safe_deployment"
)

if DEPLOY_DIR.exists():
    shutil.rmtree(
        DEPLOY_DIR
    )

DEPLOY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "\nSaving frozen MiniLM encoder "
    "for pipeline-size accounting..."
)

semantic_encoder.save(
    str(
        DEPLOY_DIR /
        "minilm_encoder"
    )
)

joblib.dump(
    fixed_bundle["vectorizer"],
    DEPLOY_DIR /
    "tfidf_vectorizer.joblib"
)

joblib.dump(
    fixed_bundle["lex11"],
    DEPLOY_DIR /
    "lexical_11_lr.joblib"
)

joblib.dump(
    fixed_bundle["lex3"],
    DEPLOY_DIR /
    "lexical_triage_lr.joblib"
)

joblib.dump(
    fixed_bundle["sem11"],
    DEPLOY_DIR /
    "semantic_11_lr.joblib"
)

joblib.dump(
    fixed_bundle["sem3"],
    DEPLOY_DIR /
    "semantic_triage_lr.joblib"
)

joblib.dump(
    fixed_gate_states[
        "vircon_safe"
    ]["scaler"],
    DEPLOY_DIR /
    "safe_scaler.joblib"
)

torch.save(
    fixed_gate_states[
        "vircon_safe"
    ]["model"].state_dict(),
    DEPLOY_DIR /
    "safe_gate_state.pt"
)

np.savez_compressed(
    DEPLOY_DIR /
    "prototypes.npz",
    cent11=fixed_bundle["cent11"],
    present11=fixed_bundle["present11"],
    cent3=fixed_bundle["cent3"],
    present3=fixed_bundle["present3"],
)

deployment_config = {
    "encoder":
        MINILM_ID,
    "safe_feature_names":
        SAFE_FEATURE_NAMES,
    "labels_11":
        list(LABELS),
    "labels_triage":
        list(TRIAGE_LABELS),
    "consistency_lambda":
        SAFE_CONSISTENCY_LAMBDA,
    "request_risk_multiplier":
        SAFE_REQUEST_RISK,
}

(DEPLOY_DIR / "config.json").write_text(
    json.dumps(
        deployment_config,
        indent=2
    ),
    encoding="utf-8"
)

pipeline_bytes = sum(
    p.stat().st_size
    for p in DEPLOY_DIR.rglob("*")
    if p.is_file()
)

pipeline_mb = (
    pipeline_bytes
    / (1024 ** 2)
)

print(
    "VIRCON-SAFE deployment bundle:",
    round(
        pipeline_mb,
        3
    ),
    "MB"
)

# CPU latency benchmark.
benchmark_rows = []

try:

    cpu_encoder = SentenceTransformer(
        MINILM_ID,
        device="cpu"
    )

    torch.set_num_threads(1)

    gate_cpu = fixed_gate_states[
        "vircon_safe"
    ]["model"].to("cpu")

    sample_text = (
        fixed_test[
            "clean_text"
        ]
        .astype(str)
        .iloc[:100]
        .tolist()
    )

    # warm-up
    for text in sample_text[:5]:

        _ = fixed_bundle[
            "vectorizer"
        ].transform([text])

        _ = cpu_encoder.encode(
            [text],
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

    # Lexical expert
    lexical_times = []

    for text in sample_text:

        t0 = time.perf_counter()

        X = fixed_bundle[
            "vectorizer"
        ].transform([text])

        _ = predict_probability_aligned(
            fixed_bundle["lex11"],
            X,
            LABELS
        )

        lexical_times.append(
            (
                time.perf_counter()
                - t0
            ) * 1000
        )

    benchmark_rows.append({
        "model":
            "lexical_lr",
        "device":
            "cpu",
        "threads":
            1,
        "median_single_ms":
            float(
                np.median(
                    lexical_times
                )
            ),
        "p95_single_ms":
            float(
                np.quantile(
                    lexical_times,
                    0.95
                )
            ),
        "pipeline_size_mb":
            np.nan,
    })

    # Semantic expert
    semantic_times = []

    for text in sample_text:

        t0 = time.perf_counter()

        E = cpu_encoder.encode(
            [text],
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32)

        _ = predict_probability_aligned(
            fixed_bundle["sem11"],
            E,
            LABELS
        )

        semantic_times.append(
            (
                time.perf_counter()
                - t0
            ) * 1000
        )

    benchmark_rows.append({
        "model":
            "semantic_minilm_lr",
        "device":
            "cpu",
        "threads":
            1,
        "median_single_ms":
            float(
                np.median(
                    semantic_times
                )
            ),
        "p95_single_ms":
            float(
                np.quantile(
                    semantic_times,
                    0.95
                )
            ),
        "pipeline_size_mb":
            np.nan,
    })

    # Full VIRCON-SAFE
    safe_times = []

    gate_state_cpu = dict(
        fixed_gate_states[
            "vircon_safe"
        ]
    )

    gate_state_cpu[
        "model"
    ] = gate_cpu

    for text in sample_text:

        t0 = time.perf_counter()

        X = fixed_bundle[
            "vectorizer"
        ].transform([text])

        E = cpu_encoder.encode(
            [text],
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32)

        pL11 = predict_probability_aligned(
            fixed_bundle["lex11"],
            X,
            LABELS
        )

        pL3 = predict_probability_aligned(
            fixed_bundle["lex3"],
            X,
            TRIAGE_LABELS
        )

        pS11 = predict_probability_aligned(
            fixed_bundle["sem11"],
            E,
            LABELS
        )

        pS3 = predict_probability_aligned(
            fixed_bundle["sem3"],
            E,
            TRIAGE_LABELS
        )

        f11 = build_gate_features(
            [text],
            fixed_bundle[
                "vectorizer"
            ],
            pL11,
            pS11,
            E,
            fixed_bundle["cent11"],
            fixed_bundle[
                "present11"
            ]
        )

        f3 = build_gate_features(
            [text],
            fixed_bundle[
                "vectorizer"
            ],
            pL3,
            pS3,
            E,
            fixed_bundle["cent3"],
            fixed_bundle[
                "present3"
            ]
        )

        _ = apply_joint_safe_gate(
            gate_state_cpu,
            {
                "p_lex11": pL11,
                "p_sem11": pS11,
                "p_lex3": pL3,
                "p_sem3": pS3,
                "feat11": f11,
                "feat3": f3,
            }
        )

        safe_times.append(
            (
                time.perf_counter()
                - t0
            ) * 1000
        )

    benchmark_rows.append({
        "model":
            "vircon_safe",
        "device":
            "cpu",
        "threads":
            1,
        "median_single_ms":
            float(
                np.median(
                    safe_times
                )
            ),
        "p95_single_ms":
            float(
                np.quantile(
                    safe_times,
                    0.95
                )
            ),
        "pipeline_size_mb":
            float(
                pipeline_mb
            ),
    })

except Exception as exc:

    warnings.warn(
        "CPU deployment benchmark failed: "
        + repr(exc)
    )

deployment_df = pd.DataFrame(
    benchmark_rows
)

deployment_df.to_csv(
    SAFE_TABLE /
    "safe_cpu_deployment_benchmark.csv",
    index=False
)

if len(deployment_df):
    print(
        "\nCURRENT-RUNTIME CPU BENCHMARK"
    )
    display(deployment_df)

# ------------------------------------------------
# Reproducibility metadata and final ZIP.
# ------------------------------------------------

environment = {
    "python":
        sys.version,
    "platform":
        platform.platform(),
    "numpy":
        np.__version__,
    "pandas":
        pd.__version__,
    "scipy":
        scipy.__version__,
    "sklearn":
        sklearn.__version__,
    "torch":
        torch.__version__,
    "cuda_available":
        torch.cuda.is_available(),
    "gpu":
        (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
}

(SAFE_REPORT / "environment.json").write_text(
    json.dumps(
        environment,
        indent=2
    ),
    encoding="utf-8"
)

claim_scope = {
    "proposed_method":
        "VIRCON-SAFE",
    "primary_supported_test":
        "same-corpus event-disjoint CrisisBench evaluation",
    "external_test":
        (
            "HumAID external transfer after the "
            "continuation notebook's overlap control"
        ),
    "human_audit":
        (
            "independent validation of the taxonomy-derived "
            "other category; not model-training data"
        ),
    "novelty_claim_rule":
        (
            "Do not claim that TF-IDF, MiniLM, fusion, "
            "consistency loss or uncertainty are individually novel. "
            "The methodological hypothesis is the event-aware, "
            "class-specific shift-gated combination and must be "
            "supported by the reported ablations."
        ),
}

(SAFE_REPORT / "claim_scope.json").write_text(
    json.dumps(
        claim_scope,
        indent=2
    ),
    encoding="utf-8"
)

README = """
# VIRCON-SAFE final experiment

This directory contains the final proposed-model evaluation.

Primary endpoint:
same-corpus event-disjoint 11-class macro-F1.

Proposed mechanism:
class-specific shift-aware lexical-semantic fusion.

The human audit is independent validation only and was not used to
train, tune or select the classifier.

Important:
Novelty conclusions must follow the ablation results. A new model name
does not by itself establish novelty.
""".strip()

(SAFE_ROOT / "README_VIRCON_SAFE.md").write_text(
    README,
    encoding="utf-8"
)


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as fh:

        for chunk in iter(
            lambda:
                fh.read(
                    1024 * 1024
                ),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


manifest = []

for p in sorted(
    SAFE_ROOT.rglob("*")
):

    if not p.is_file():
        continue

    # Caches are reproducible intermediates and are deliberately
    # excluded from the public/final ZIP to keep it manageable.
    if SAFE_CACHE in p.parents:
        continue

    manifest.append({
        "relative_path":
            str(
                p.relative_to(
                    SAFE_ROOT
                )
            ),
        "sha256":
            sha256_file(p),
        "bytes":
            p.stat().st_size,
    })

manifest_df = pd.DataFrame(
    manifest
)

manifest_df.to_csv(
    SAFE_ROOT /
    "artifact_manifest_sha256.csv",
    index=False
)

FINAL_ZIP = (
    WORKDIR /
    "VIRCON_SAFE_Final_Artifacts.zip"
)

with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    allowZip64=True
) as z:

    for p in sorted(
        SAFE_ROOT.rglob("*")
    ):

        if not p.is_file():
            continue

        if SAFE_CACHE in p.parents:
            continue

        z.write(
            p,
            arcname=str(
                Path(
                    SAFE_ROOT.name
                )
                /
                p.relative_to(
                    SAFE_ROOT
                )
            )
        )

    notebook_path = (
        WORKDIR /
        "VIRCON_SAFE_Novel_Model_Final.ipynb"
    )

    if notebook_path.is_file():

        z.write(
            notebook_path,
            arcname=
                notebook_path.name
        )

print(
    "\n"
    + "=" * 70
)

print(
    "VIRCON-SAFE FINAL RUN COMPLETED"
)

print(
    "=" * 70
)

print(
    "Results directory:",
    SAFE_ROOT
)

print(
    "Final ZIP:",
    FINAL_ZIP
)

print(
    "\nPrimary table:"
)

print(
    SAFE_TABLE /
    "safe_event_disjoint_summary.csv"
)

print(
    "\nNovelty evidence:"
)

print(
    SAFE_TABLE /
    "novelty_evidence_check.csv"
)

print(
    "\nStatistics:"
)

print(
    SAFE_TABLE /
    "safe_primary_effect_estimates_and_cis.csv"
)

print(
    "\nHuman audit:"
)

print(
    SAFE_TABLE /
    "human_audit_frozen_summary.csv"
)


# VIRCON-ERHT Final Proposed-Model Experiment

## Event-Robust Hierarchical Triage Transformer

This section evaluates a second-stage proposed architecture developed
after the VIRCON-SAFE ablation showed that post-hoc shift-aware fusion
did not outperform either static lexical-semantic fusion or the
fine-tuned DistilBERT comparator on the primary event-disjoint endpoint.

**VIRCON-ERHT** therefore moves event-robust learning inside the
transformer representation itself.

The architecture combines:

1. Fine-tuned DistilBERT humanitarian representation.
2. Eleven-class humanitarian classification head.
3. Three-way taxonomy-derived triage head.
4. Dedicated binary request-recovery head.
5. Hierarchical consistency between eleven-class and triage outputs.
6. Cross-event supervised contrastive representation learning.
7. Event-adversarial representation learning through gradient reversal.

The primary endpoint remains **event-disjoint eleven-class macro-F1**.
Request recall is a prespecified secondary endpoint because previous
event-disjoint experiments produced zero request recall.

The 200-message manual audit remains frozen and is used only as
independent validation of the taxonomy-derived triage construction.
Human-audit labels are not used to train, tune, threshold, or select
VIRCON-ERHT.


In [ ]:
# =====================================================================
# ERHT 1. Configuration, validation and reproducibility
# =====================================================================

from pathlib import Path
import os
import sys
import gc
import json
import math
import time
import random
import shutil
import hashlib
import zipfile
import platform
import warnings
import subprocess
import importlib.util

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

# ------------------------------------------------------------------
# Validate prerequisite objects from the previous VIRCON notebook
# ------------------------------------------------------------------

ERHT_REQUIRED_GLOBALS = [
    "WORKDIR",
    "TABLE_DIR",
    "combined",
    "strict_df",
    "LABELS",
    "TRIAGE_LABELS",
    "triage_map",
]

ERHT_MISSING = [
    name
    for name in ERHT_REQUIRED_GLOBALS
    if name not in globals()
]

if ERHT_MISSING:
    raise RuntimeError(
        "Run the notebook from the beginning before starting ERHT. "
        "Missing objects: "
        + ", ".join(ERHT_MISSING)
    )

# ------------------------------------------------------------------
# Install only missing packages.
# Do NOT broadly upgrade numpy/pandas/torch in Colab.
# ------------------------------------------------------------------

if importlib.util.find_spec(
    "transformers"
) is None:

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "transformers>=4.45,<5",
            "accelerate>=1,<2",
        ]
    )

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

# ------------------------------------------------------------------
# Main settings
# ------------------------------------------------------------------

ERHT_SEED = 20261008

ERHT_BASE_MODEL = (
    "distilbert-base-uncased"
)

ERHT_MAX_LENGTH = 128

# T4-safe defaults.
ERHT_TRAIN_BATCH = 24
ERHT_EVAL_BATCH = 128

# Full manuscript run.
ERHT_EPOCHS = 2

ERHT_LEARNING_RATE = 2e-5
ERHT_WEIGHT_DECAY = 0.01
ERHT_WARMUP_RATIO = 0.10

# Prespecified joint-objective coefficients.
ERHT_LAMBDA_TRIAGE = 0.60
ERHT_LAMBDA_REQUEST = 0.50
ERHT_LAMBDA_HIERARCHY = 0.20
ERHT_LAMBDA_CONTRASTIVE = 0.10
ERHT_LAMBDA_EVENT_ADV = 0.05

ERHT_CONTRASTIVE_TEMP = 0.10

# Fixed before outer-event evaluation.
ERHT_REQUEST_THRESHOLD = 0.50

# Set False only for a quick diagnostic.
# For manuscript evidence this must remain True.
ERHT_RUN_ABLATIONS = True

ERHT_RUN_FIXED_SPLIT = True
ERHT_RUN_HUMAID = True

ERHT_N_BOOTSTRAP = 2000

ERHT_DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

if ERHT_DEVICE.type != "cuda":
    warnings.warn(
        "ERHT is computationally expensive. "
        "A GPU runtime is strongly recommended."
    )

random.seed(ERHT_SEED)
np.random.seed(ERHT_SEED)
torch.manual_seed(ERHT_SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        ERHT_SEED
    )

    # Safe speed optimisation on recent GPUs.
    torch.backends.cuda.matmul.allow_tf32 = True

print(
    "ERHT device:",
    ERHT_DEVICE
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# ------------------------------------------------------------------
# Output folders
# ------------------------------------------------------------------

ERHT_ROOT = (
    Path(WORKDIR)
    /
    "artifacts_vircon_erht_final"
)

ERHT_TABLE = (
    ERHT_ROOT /
    "tables"
)

ERHT_REPORT = (
    ERHT_ROOT /
    "reports"
)

ERHT_MODEL_DIR = (
    ERHT_ROOT /
    "models"
)

ERHT_CHECKPOINT = (
    ERHT_ROOT /
    "checkpoints"
)

for directory in [
    ERHT_ROOT,
    ERHT_TABLE,
    ERHT_REPORT,
    ERHT_MODEL_DIR,
    ERHT_CHECKPOINT,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------------
# Label definitions
# ------------------------------------------------------------------

LABELS = list(LABELS)
TRIAGE_LABELS = list(
    TRIAGE_LABELS
)

ERHT_LABEL_TO_ID = {
    label: index
    for index, label
    in enumerate(LABELS)
}

ERHT_TRIAGE_TO_ID = {
    label: index
    for index, label
    in enumerate(TRIAGE_LABELS)
}

if "request" not in ERHT_TRIAGE_TO_ID:
    raise RuntimeError(
        "The triage labels do not contain 'request'."
    )

if (
    "requests_or_needs"
    not in ERHT_LABEL_TO_ID
):
    raise RuntimeError(
        "The 11-class labels do not contain requests_or_needs."
    )

ERHT_REQUEST_TRIAGE_ID = (
    ERHT_TRIAGE_TO_ID[
        "request"
    ]
)

ERHT_REQUEST_11_ID = (
    ERHT_LABEL_TO_ID[
        "requests_or_needs"
    ]
)


def erht_triage_mapping(
    label
):

    if callable(
        triage_map
    ):
        return str(
            triage_map(label)
        )

    if isinstance(
        triage_map,
        dict
    ):
        return str(
            triage_map[
                str(label)
            ]
        )

    raise TypeError(
        "triage_map must be a function or dictionary."
    )


# Mapping matrix from 11 classes to 3 triage classes.
ERHT_A = np.zeros(
    (
        len(LABELS),
        len(TRIAGE_LABELS)
    ),
    dtype=np.float32
)

for index, label in enumerate(
    LABELS
):

    triage_label = (
        erht_triage_mapping(
            label
        )
    )

    ERHT_A[
        index,
        ERHT_TRIAGE_TO_ID[
            triage_label
        ]
    ] = 1.0

# ------------------------------------------------------------------
# Freeze and copy the human-audit summary
# ------------------------------------------------------------------

audit_candidates = list(
    Path(WORKDIR).rglob(
        "triage_manual_audit_FINAL_frozen_summary.csv"
    )
)

if not audit_candidates:
    raise FileNotFoundError(
        "triage_manual_audit_FINAL_frozen_summary.csv "
        "was not found in VirconPaper1."
    )

ERHT_AUDIT_PATH = max(
    audit_candidates,
    key=lambda p:
        p.stat().st_mtime
)

erht_audit = pd.read_csv(
    ERHT_AUDIT_PATH
)

required_audit_cols = [
    "n_audited",
    "original_rater_agreement_n",
    "original_rater_disagreement_n",
    "raw_interrater_agreement",
    "cohen_kappa",
    "n_adjudicated",
    "hidden_actionable_count",
    "hidden_actionable_fraction",
]

missing_audit = [
    c
    for c in required_audit_cols
    if c not in erht_audit.columns
]

if missing_audit:
    raise RuntimeError(
        "Frozen audit summary is incomplete: "
        + ", ".join(
            missing_audit
        )
    )

audit_row = erht_audit.iloc[0]

if int(
    audit_row[
        "n_audited"
    ]
) != 200:
    raise RuntimeError(
        "Frozen audit should contain 200 records."
    )

if int(
    audit_row[
        "n_adjudicated"
    ]
) != 7:
    raise RuntimeError(
        "Expected 7 adjudicated disagreements."
    )

erht_audit.to_csv(
    ERHT_TABLE /
    "human_audit_frozen_summary.csv",
    index=False
)

human_audit_role = {
    "used_for_training":
        False,

    "used_for_hyperparameter_selection":
        False,

    "used_for_threshold_selection":
        False,

    "used_for_model_selection":
        False,

    "role":
        (
            "Independent sensitivity validation "
            "of taxonomy-derived triage."
        )
}

(
    ERHT_REPORT /
    "human_audit_role.json"
).write_text(
    json.dumps(
        human_audit_role,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "\nFrozen human audit:"
)

print(
    erht_audit.to_string(
        index=False
    )
)

# ------------------------------------------------------------------
# Save configuration
# ------------------------------------------------------------------

ERHT_CONFIG = {
    "model":
        "VIRCON-ERHT",

    "expansion":
        "Event-Robust Hierarchical Triage Transformer",

    "base_encoder":
        ERHT_BASE_MODEL,

    "max_length":
        ERHT_MAX_LENGTH,

    "epochs":
        ERHT_EPOCHS,

    "train_batch":
        ERHT_TRAIN_BATCH,

    "learning_rate":
        ERHT_LEARNING_RATE,

    "weight_decay":
        ERHT_WEIGHT_DECAY,

    "lambda_triage":
        ERHT_LAMBDA_TRIAGE,

    "lambda_request":
        ERHT_LAMBDA_REQUEST,

    "lambda_hierarchy":
        ERHT_LAMBDA_HIERARCHY,

    "lambda_cross_event_contrastive":
        ERHT_LAMBDA_CONTRASTIVE,

    "lambda_event_adversarial":
        ERHT_LAMBDA_EVENT_ADV,

    "request_threshold":
        ERHT_REQUEST_THRESHOLD,

    "primary_endpoint":
        (
            "same-corpus event-disjoint "
            "11-class macro-F1"
        ),

    "human_audit_used_for_training":
        False,
}

(
    ERHT_REPORT /
    "erht_configuration.json"
).write_text(
    json.dumps(
        ERHT_CONFIG,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "\nERHT configuration ready."
)


In [ ]:
# =====================================================================
# ERHT 2. Dataset, transformer architecture and loss functions
# =====================================================================

erht_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        ERHT_BASE_MODEL
    )
)


class ERHTDataset(
    Dataset
):

    def __init__(
        self,
        frame,
        event_mapping=None,
        require_labels=True,
    ):

        self.frame = (
            frame
            .reset_index(
                drop=True
            )
            .copy()
        )

        texts = (
            self.frame[
                "clean_text"
            ]
            .fillna("")
            .astype(str)
            .tolist()
        )

        encoded = (
            erht_tokenizer(
                texts,
                truncation=True,
                padding="max_length",
                max_length=
                    ERHT_MAX_LENGTH,
                return_tensors="pt",
            )
        )

        self.input_ids = (
            encoded[
                "input_ids"
            ]
        )

        self.attention_mask = (
            encoded[
                "attention_mask"
            ]
        )

        self.global_ids = (
            self.frame[
                "global_row_id"
            ]
            .astype(int)
            .to_numpy()
        )

        if require_labels:

            self.y11 = (
                torch.tensor(
                    [
                        ERHT_LABEL_TO_ID[
                            str(x)
                        ]
                        for x
                        in self.frame[
                            "label_11"
                        ]
                    ],
                    dtype=torch.long
                )
            )

            self.y3 = (
                torch.tensor(
                    [
                        ERHT_TRIAGE_TO_ID[
                            str(x)
                        ]
                        for x
                        in self.frame[
                            "label_triage"
                        ]
                    ],
                    dtype=torch.long
                )
            )

        else:

            self.y11 = torch.zeros(
                len(self.frame),
                dtype=torch.long
            )

            self.y3 = torch.zeros(
                len(self.frame),
                dtype=torch.long
            )

        self.y_request = (
            self.y3
            ==
            ERHT_REQUEST_TRIAGE_ID
        ).float()

        if event_mapping is not None:

            event_col = (
                "canonical_event"
                if "canonical_event"
                   in self.frame.columns
                else "event"
            )

            self.event_ids = (
                torch.tensor(
                    [
                        event_mapping[
                            str(x)
                        ]
                        for x
                        in self.frame[
                            event_col
                        ]
                    ],
                    dtype=torch.long
                )
            )

        else:

            self.event_ids = (
                torch.full(
                    (
                        len(
                            self.frame
                        ),
                    ),
                    -1,
                    dtype=torch.long
                )
            )

    def __len__(self):

        return len(
            self.frame
        )

    def __getitem__(
        self,
        index
    ):

        return {
            "input_ids":
                self.input_ids[
                    index
                ],

            "attention_mask":
                self.attention_mask[
                    index
                ],

            "y11":
                self.y11[
                    index
                ],

            "y3":
                self.y3[
                    index
                ],

            "y_request":
                self.y_request[
                    index
                ],

            "event_id":
                self.event_ids[
                    index
                ],

            "global_row_id":
                int(
                    self.global_ids[
                        index
                    ]
                ),
        }


# ------------------------------------------------------------------
# Gradient reversal
# ------------------------------------------------------------------

class ERHTGradientReversal(
    torch.autograd.Function
):

    @staticmethod
    def forward(
        ctx,
        x,
        strength
    ):

        ctx.strength = (
            float(strength)
        )

        return x.view_as(x)

    @staticmethod
    def backward(
        ctx,
        gradient
    ):

        return (
            -ctx.strength
            * gradient,
            None
        )


def erht_gradient_reverse(
    x,
    strength
):

    return (
        ERHTGradientReversal
        .apply(
            x,
            strength
        )
    )


# ------------------------------------------------------------------
# ERHT network
# ------------------------------------------------------------------

class VIRCON_ERHT(
    nn.Module
):

    def __init__(
        self,
        base_model,
        n_11,
        n_triage,
        n_source_events,
    ):

        super().__init__()

        self.encoder = (
            AutoModel
            .from_pretrained(
                base_model
            )
        )

        hidden_size = (
            self.encoder
            .config
            .hidden_size
        )

        self.dropout = (
            nn.Dropout(
                0.20
            )
        )

        self.humanitarian_head = (
            nn.Linear(
                hidden_size,
                n_11
            )
        )

        self.triage_head = (
            nn.Linear(
                hidden_size,
                n_triage
            )
        )

        self.request_head = (
            nn.Linear(
                hidden_size,
                1
            )
        )

        # Event discriminator is training-only.
        self.event_discriminator = (
            nn.Sequential(
                nn.Linear(
                    hidden_size,
                    128
                ),
                nn.ReLU(),
                nn.Dropout(
                    0.10
                ),
                nn.Linear(
                    128,
                    n_source_events
                ),
            )
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        grl_strength=0.0,
    ):

        encoder_output = (
            self.encoder(
                input_ids=
                    input_ids,
                attention_mask=
                    attention_mask
            )
        )

        representation = (
            encoder_output
            .last_hidden_state[
                :,
                0,
                :
            ]
        )

        dropped = (
            self.dropout(
                representation
            )
        )

        logits_11 = (
            self.humanitarian_head(
                dropped
            )
        )

        logits_triage = (
            self.triage_head(
                dropped
            )
        )

        logits_request = (
            self.request_head(
                dropped
            )
            .squeeze(-1)
        )

        event_representation = (
            erht_gradient_reverse(
                representation,
                grl_strength
            )
        )

        logits_event = (
            self.event_discriminator(
                event_representation
            )
        )

        return {
            "representation":
                representation,

            "logits_11":
                logits_11,

            "logits_triage":
                logits_triage,

            "logits_request":
                logits_request,

            "logits_event":
                logits_event,
        }


# ------------------------------------------------------------------
# Weighted losses
# ------------------------------------------------------------------

def erht_class_weights(
    values,
    n_classes,
    maximum=8.0,
):

    values = np.asarray(
        values,
        dtype=np.int64
    )

    counts = np.bincount(
        values,
        minlength=n_classes
    ).astype(float)

    weights = np.zeros(
        n_classes,
        dtype=np.float32
    )

    present = (
        counts > 0
    )

    weights[
        present
    ] = (
        len(values)
        /
        (
            n_classes
            * counts[
                present
            ]
        )
    )

    weights[
        present
    ] = np.clip(
        weights[
            present
        ],
        0.25,
        maximum
    )

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=ERHT_DEVICE
    )


def erht_request_focal_loss(
    logits,
    targets,
    pos_weight,
    gamma=2.0,
):

    targets = (
        targets.float()
    )

    bce = (
        F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=
                pos_weight,
            reduction="none",
        )
    )

    probabilities = (
        torch.sigmoid(
            logits
        )
    )

    pt = torch.where(
        targets > 0.5,
        probabilities,
        1.0
        - probabilities
    )

    focal_weight = (
        1.0 - pt
    ).pow(
        gamma
    )

    return (
        focal_weight
        * bce
    ).mean()


ERHT_A_TENSOR = (
    torch.tensor(
        ERHT_A,
        dtype=torch.float32,
        device=ERHT_DEVICE
    )
)


def erht_js_divergence(
    p,
    q
):

    p = torch.clamp(
        p,
        min=1e-8
    )

    q = torch.clamp(
        q,
        min=1e-8
    )

    middle = (
        0.5
        *
        (
            p + q
        )
    )

    return (
        0.5
        *
        (
            p
            *
            (
                torch.log(p)
                -
                torch.log(
                    middle
                )
            )
        ).sum(
            dim=1
        )
        +
        0.5
        *
        (
            q
            *
            (
                torch.log(q)
                -
                torch.log(
                    middle
                )
            )
        ).sum(
            dim=1
        )
    ).mean()


def erht_hierarchy_loss(
    logits_11,
    logits_triage,
):

    p11 = (
        F.softmax(
            logits_11,
            dim=1
        )
    )

    p3_direct = (
        F.softmax(
            logits_triage,
            dim=1
        )
    )

    p3_from_11 = (
        torch.matmul(
            p11,
            ERHT_A_TENSOR
        )
    )

    p3_from_11 = (
        p3_from_11
        /
        p3_from_11.sum(
            dim=1,
            keepdim=True
        )
        .clamp_min(
            1e-8
        )
    )

    return (
        erht_js_divergence(
            p3_direct,
            p3_from_11
        )
    )


# ------------------------------------------------------------------
# Cross-event supervised contrastive loss
#
# Positive:
#     same humanitarian class
#     but different disaster event
# ------------------------------------------------------------------

def erht_cross_event_contrastive(
    representation,
    humanitarian_labels,
    event_labels,
    temperature=ERHT_CONTRASTIVE_TEMP,
):
    """
    Cross-event supervised contrastive loss.

    Positive pair:
        same humanitarian class
        AND different disaster event.

    IMPORTANT:
    Similarity computation is forced to float32 because the model
    itself is trained under mixed precision. This prevents FP16
    overflow when masking self-similarity.
    """

    # Disable AMP only for this numerically sensitive loss.
    with torch.amp.autocast(
        device_type=representation.device.type,
        enabled=False
    ):

        # Force float32.
        z = F.normalize(
            representation.float(),
            p=2,
            dim=1
        )

        humanitarian_labels = (
            humanitarian_labels.long()
        )

        event_labels = (
            event_labels.long()
        )

        batch_size = z.size(0)

        # [B, B] cosine similarity matrix
        similarity = (
            torch.matmul(
                z,
                z.T
            )
            /
            float(temperature)
        )

        # Self-comparison mask
        eye = torch.eye(
            batch_size,
            device=z.device,
            dtype=torch.bool
        )

        # Same humanitarian class
        same_class = (
            humanitarian_labels[:, None]
            ==
            humanitarian_labels[None, :]
        )

        # Different disaster event
        different_event = (
            event_labels[:, None]
            !=
            event_labels[None, :]
        )

        # Positive pairs:
        # same class, different event, not self
        positive_mask = (
            same_class
            &
            different_event
            &
            (~eye)
        )

        positive_count = (
            positive_mask
            .sum(dim=1)
        )

        valid_anchor = (
            positive_count > 0
        )

        # Some mini-batches may contain no valid cross-event
        # positive pair. Return differentiable zero safely.
        if not valid_anchor.any():
            return (
                representation.float().sum()
                * 0.0
            )

        # Mask self-similarity.
        # -inf is safe here because this entire block is FP32.
        denominator_logits = (
            similarity.masked_fill(
                eye,
                float("-inf")
            )
        )

        log_denominator = (
            torch.logsumexp(
                denominator_logits,
                dim=1,
                keepdim=True
            )
        )

        log_probability = (
            similarity
            -
            log_denominator
        )

        # Only positive-pair log probabilities
        positive_log_probability = (
            (
                log_probability
                *
                positive_mask.float()
            )
            .sum(dim=1)
            /
            positive_count
            .clamp_min(1)
            .float()
        )

        loss = (
            -positive_log_probability[
                valid_anchor
            ]
            .mean()
        )

    return loss


In [ ]:
# =====================================================================
# ERHT 3. Training, prediction and evaluation utilities
# =====================================================================

ERHT_VARIANTS = {

    "erht_full": {
        "lambda_triage":
            ERHT_LAMBDA_TRIAGE,

        "lambda_request":
            ERHT_LAMBDA_REQUEST,

        "lambda_hierarchy":
            ERHT_LAMBDA_HIERARCHY,

        "lambda_contrastive":
            ERHT_LAMBDA_CONTRASTIVE,

        "lambda_event":
            ERHT_LAMBDA_EVENT_ADV,

        "request_override":
            True,
    },

    # Removes the two explicitly event-robust representation objectives.
    "erht_no_event": {
        "lambda_triage":
            ERHT_LAMBDA_TRIAGE,

        "lambda_request":
            ERHT_LAMBDA_REQUEST,

        "lambda_hierarchy":
            ERHT_LAMBDA_HIERARCHY,

        "lambda_contrastive":
            0.0,

        "lambda_event":
            0.0,

        "request_override":
            True,
    },

    "erht_no_hierarchy": {
        "lambda_triage":
            ERHT_LAMBDA_TRIAGE,

        "lambda_request":
            ERHT_LAMBDA_REQUEST,

        "lambda_hierarchy":
            0.0,

        "lambda_contrastive":
            ERHT_LAMBDA_CONTRASTIVE,

        "lambda_event":
            ERHT_LAMBDA_EVENT_ADV,

        "request_override":
            True,
    },

    "erht_no_request": {
        "lambda_triage":
            ERHT_LAMBDA_TRIAGE,

        "lambda_request":
            0.0,

        "lambda_hierarchy":
            ERHT_LAMBDA_HIERARCHY,

        "lambda_contrastive":
            ERHT_LAMBDA_CONTRASTIVE,

        "lambda_event":
            ERHT_LAMBDA_EVENT_ADV,

        "request_override":
            False,
    },
}

if not ERHT_RUN_ABLATIONS:

    ERHT_VARIANTS = {
        "erht_full":
            ERHT_VARIANTS[
                "erht_full"
            ]
    }


def erht_event_mapping(
    frame
):

    event_col = (
        "canonical_event"
        if "canonical_event"
           in frame.columns
        else "event"
    )

    events = sorted(
        frame[
            event_col
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    return {
        event:
            index
        for index, event
        in enumerate(events)
    }


def erht_loader(
    dataset,
    batch_size,
    shuffle,
):

    return DataLoader(
        dataset,
        batch_size=
            batch_size,
        shuffle=
            shuffle,

        # More reliable in notebook environments.
        num_workers=0,

        pin_memory=(
            ERHT_DEVICE.type
            == "cuda"
        ),
    )


def erht_train(
    train_frame,
    variant_name,
    seed,
):

    spec = (
        ERHT_VARIANTS[
            variant_name
        ]
    )

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )

    event_map = (
        erht_event_mapping(
            train_frame
        )
    )

    train_dataset = (
        ERHTDataset(
            train_frame,
            event_mapping=
                event_map,
            require_labels=True,
        )
    )

    train_loader = (
        erht_loader(
            train_dataset,
            ERHT_TRAIN_BATCH,
            True
        )
    )

    class_weight_11 = (
        erht_class_weights(
            train_dataset
            .y11
            .numpy(),

            len(
                LABELS
            )
        )
    )

    class_weight_3 = (
        erht_class_weights(
            train_dataset
            .y3
            .numpy(),

            len(
                TRIAGE_LABELS
            )
        )
    )

    request_positive = float(
        train_dataset
        .y_request
        .sum()
        .item()
    )

    request_negative = float(
        len(
            train_dataset
        )
        -
        request_positive
    )

    # Prevent an extreme rare-class gradient.
    request_pos_weight = (
        min(
            20.0,
            request_negative
            /
            max(
                request_positive,
                1.0
            )
        )
    )

    request_pos_weight = (
        torch.tensor(
            request_pos_weight,
            dtype=torch.float32,
            device=ERHT_DEVICE
        )
    )

    print(
        "Source request positives:",
        int(
            request_positive
        ),
        "| request pos_weight:",
        float(
            request_pos_weight
            .detach()
            .cpu()
        )
    )

    model = (
        VIRCON_ERHT(
            ERHT_BASE_MODEL,
            len(LABELS),
            len(TRIAGE_LABELS),
            len(event_map),
        )
        .to(
            ERHT_DEVICE
        )
    )

    optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=
                ERHT_LEARNING_RATE,
            weight_decay=
                ERHT_WEIGHT_DECAY,
        )
    )

    total_steps = (
        len(
            train_loader
        )
        *
        ERHT_EPOCHS
    )

    warmup_steps = int(
        total_steps
        *
        ERHT_WARMUP_RATIO
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=
                warmup_steps,
            num_training_steps=
                total_steps,
        )
    )

    use_amp = (
        ERHT_DEVICE.type
        == "cuda"
    )

    # Compatibility with multiple Colab/PyTorch versions.
    scaler = (
        torch.cuda.amp
        .GradScaler(
            enabled=
                use_amp
        )
    )

    history = []

    global_step = 0

    for epoch in range(
        1,
        ERHT_EPOCHS + 1
    ):

        model.train()

        losses = {
            "total": [],
            "11class": [],
            "triage": [],
            "request": [],
            "hierarchy": [],
            "contrastive": [],
            "event": [],
        }

        for batch in (
            train_loader
        ):

            global_step += 1

            input_ids = (
                batch[
                    "input_ids"
                ]
                .to(
                    ERHT_DEVICE,
                    non_blocking=True
                )
            )

            attention_mask = (
                batch[
                    "attention_mask"
                ]
                .to(
                    ERHT_DEVICE,
                    non_blocking=True
                )
            )

            y11 = (
                batch[
                    "y11"
                ]
                .to(
                    ERHT_DEVICE
                )
            )

            y3 = (
                batch[
                    "y3"
                ]
                .to(
                    ERHT_DEVICE
                )
            )

            y_request = (
                batch[
                    "y_request"
                ]
                .to(
                    ERHT_DEVICE
                )
            )

            event_id = (
                batch[
                    "event_id"
                ]
                .to(
                    ERHT_DEVICE
                )
            )

            progress = (
                global_step
                /
                max(
                    total_steps,
                    1
                )
            )

            # Standard gradual GRL schedule.
            grl_strength = (
                2.0
                /
                (
                    1.0
                    +
                    math.exp(
                        -10.0
                        * progress
                    )
                )
                -
                1.0
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with (
                torch.cuda.amp
                .autocast(
                    enabled=
                        use_amp
                )
            ):

                output = (
                    model(
                        input_ids=
                            input_ids,

                        attention_mask=
                            attention_mask,

                        grl_strength=
                            grl_strength,
                    )
                )

                loss_11 = (
                    F.cross_entropy(
                        output[
                            "logits_11"
                        ],
                        y11,
                        weight=
                            class_weight_11,
                    )
                )

                loss_triage = (
                    F.cross_entropy(
                        output[
                            "logits_triage"
                        ],
                        y3,
                        weight=
                            class_weight_3,
                    )
                )

                loss_request = (
                    erht_request_focal_loss(
                        output[
                            "logits_request"
                        ],
                        y_request,
                        request_pos_weight,
                    )
                )

                loss_hierarchy = (
                    erht_hierarchy_loss(
                        output[
                            "logits_11"
                        ],
                        output[
                            "logits_triage"
                        ],
                    )
                )

                loss_contrastive = (
                    erht_cross_event_contrastive(
                        output[
                            "representation"
                        ],
                        y11,
                        event_id,
                    )
                )

                loss_event = (
                    F.cross_entropy(
                        output[
                            "logits_event"
                        ],
                        event_id,
                    )
                )

                total_loss = (
                    loss_11

                    +
                    spec[
                        "lambda_triage"
                    ]
                    *
                    loss_triage

                    +
                    spec[
                        "lambda_request"
                    ]
                    *
                    loss_request

                    +
                    spec[
                        "lambda_hierarchy"
                    ]
                    *
                    loss_hierarchy

                    +
                    spec[
                        "lambda_contrastive"
                    ]
                    *
                    loss_contrastive

                    +
                    spec[
                        "lambda_event"
                    ]
                    *
                    loss_event
                )

            scaler.scale(
                total_loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            scheduler.step()

            losses[
                "total"
            ].append(
                float(
                    total_loss
                    .detach()
                    .cpu()
                )
            )

            losses[
                "11class"
            ].append(
                float(
                    loss_11
                    .detach()
                    .cpu()
                )
            )

            losses[
                "triage"
            ].append(
                float(
                    loss_triage
                    .detach()
                    .cpu()
                )
            )

            losses[
                "request"
            ].append(
                float(
                    loss_request
                    .detach()
                    .cpu()
                )
            )

            losses[
                "hierarchy"
            ].append(
                float(
                    loss_hierarchy
                    .detach()
                    .cpu()
                )
            )

            losses[
                "contrastive"
            ].append(
                float(
                    loss_contrastive
                    .detach()
                    .cpu()
                )
            )

            losses[
                "event"
            ].append(
                float(
                    loss_event
                    .detach()
                    .cpu()
                )
            )

        epoch_row = {
            "epoch":
                epoch
        }

        for name, values in (
            losses.items()
        ):

            epoch_row[
                f"loss_{name}"
            ] = float(
                np.mean(
                    values
                )
            )

        history.append(
            epoch_row
        )

        print(
            variant_name,
            "| epoch",
            epoch,
            "| total loss =",
            round(
                epoch_row[
                    "loss_total"
                ],
                4
            )
        )

    return (
        model,
        pd.DataFrame(
            history
        ),
        event_map,
    )


@torch.no_grad()
def erht_predict(
    model,
    frame,
    request_override=True,
):

    model.eval()

    dataset = (
        ERHTDataset(
            frame,
            event_mapping=None,
            require_labels=False,
        )
    )

    loader = (
        erht_loader(
            dataset,
            ERHT_EVAL_BATCH,
            False
        )
    )

    probability_11 = []
    probability_3 = []
    request_probability = []
    ids = []

    for batch in loader:

        output = (
            model(
                input_ids=
                    batch[
                        "input_ids"
                    ].to(
                        ERHT_DEVICE
                    ),

                attention_mask=
                    batch[
                        "attention_mask"
                    ].to(
                        ERHT_DEVICE
                    ),

                grl_strength=
                    0.0,
            )
        )

        probability_11.append(
            F.softmax(
                output[
                    "logits_11"
                ],
                dim=1
            )
            .cpu()
            .numpy()
        )

        probability_3.append(
            F.softmax(
                output[
                    "logits_triage"
                ],
                dim=1
            )
            .cpu()
            .numpy()
        )

        request_probability.append(
            torch.sigmoid(
                output[
                    "logits_request"
                ]
            )
            .cpu()
            .numpy()
        )

        ids.extend(
            batch[
                "global_row_id"
            ]
            .numpy()
            .tolist()
        )

    probability_11 = (
        np.vstack(
            probability_11
        )
    )

    probability_3 = (
        np.vstack(
            probability_3
        )
    )

    request_probability = (
        np.concatenate(
            request_probability
        )
    )

    pred_11_raw = (
        np.argmax(
            probability_11,
            axis=1
        )
    )

    pred_3_raw = (
        np.argmax(
            probability_3,
            axis=1
        )
    )

    pred_11 = (
        pred_11_raw.copy()
    )

    pred_3 = (
        pred_3_raw.copy()
    )

    if request_override:

        request_mask = (
            request_probability
            >=
            ERHT_REQUEST_THRESHOLD
        )

        pred_11[
            request_mask
        ] = (
            ERHT_REQUEST_11_ID
        )

        pred_3[
            request_mask
        ] = (
            ERHT_REQUEST_TRIAGE_ID
        )

    return {
        "global_row_id":
            np.asarray(
                ids,
                dtype=np.int64
            ),

        "probability_11":
            probability_11,

        "probability_triage":
            probability_3,

        "request_probability":
            request_probability,

        "pred_11_raw":
            pred_11_raw,

        "pred_triage_raw":
            pred_3_raw,

        "pred_11":
            pred_11,

        "pred_triage":
            pred_3,
    }


def erht_decode(
    ids,
    labels,
):

    labels = np.asarray(
        labels,
        dtype=object
    )

    return labels[
        np.asarray(
            ids,
            dtype=int
        )
    ]


def erht_metrics(
    y_true,
    y_pred,
    labels,
):

    return {
        "n":
            int(
                len(
                    y_true
                )
            ),

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    labels=
                        list(
                            labels
                        ),
                    average=
                        "macro",
                    zero_division=
                        0,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    y_pred
                )
            ),
    }


def erht_per_class(
    y_true,
    y_pred,
    labels,
    model_name,
    task,
    protocol,
):

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=
                list(
                    labels
                ),
            zero_division=
                0,
        )
    )

    return pd.DataFrame(
        {
            "protocol":
                protocol,

            "task":
                task,

            "model":
                model_name,

            "label":
                list(
                    labels
                ),

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1,

            "support":
                support,
        }
    )


print(
    "ERHT training utilities ready."
)


In [ ]:
# =====================================================================
# ERHT 4. PRIMARY event-disjoint evaluation
# =====================================================================

erht_event_outputs = []

erht_outer_folds = sorted(
    strict_df[
        "fold"
    ]
    .astype(int)
    .unique()
    .tolist()
)

print(
    "Event-disjoint folds:",
    erht_outer_folds
)

variant_names = list(
    ERHT_VARIANTS.keys()
)

for outer_fold in (
    erht_outer_folds
):

    outer_test_mask = (
        strict_df[
            "fold"
        ]
        .astype(int)
        .eq(
            int(
                outer_fold
            )
        )
    )

    outer_train = (
        strict_df.loc[
            ~outer_test_mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    outer_test = (
        strict_df.loc[
            outer_test_mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    held_events = sorted(
        outer_test[
            "canonical_event"
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    print(
        "\n"
        + "=" * 72
    )

    print(
        "ERHT EVENT FOLD",
        outer_fold,
        "| train =",
        len(
            outer_train
        ),
        "| test =",
        len(
            outer_test
        ),
    )

    print(
        "Held-out events:",
        len(
            held_events
        )
    )

    print(
        "=" * 72
    )

    for variant_index, variant_name in enumerate(
        variant_names,
        start=1
    ):

        checkpoint = (
            ERHT_CHECKPOINT
            /
            (
                f"event_fold_{outer_fold}_"
                f"{variant_name}.csv"
            )
        )

        # --------------------------------------------------------------
        # Resume support
        # --------------------------------------------------------------

        if checkpoint.is_file():

            saved = (
                pd.read_csv(
                    checkpoint
                )
            )

            required_columns = {
                "global_row_id",
                "canonical_event",
                "fold",
                "label_11",
                "label_triage",
                "model",
                "pred_11",
                "pred_triage",
                "request_probability",
            }

            expected_ids = set(
                outer_test[
                    "global_row_id"
                ]
                .astype(int)
                .tolist()
            )

            observed_ids = set(
                saved[
                    "global_row_id"
                ]
                .astype(int)
                .tolist()
            )

            if (
                required_columns
                .issubset(
                    saved.columns
                )
                and
                len(saved)
                ==
                len(
                    outer_test
                )
                and
                expected_ids
                ==
                observed_ids
            ):

                print(
                    "Reusing checkpoint:",
                    checkpoint.name
                )

                erht_event_outputs.append(
                    saved
                )

                continue

        print(
            "\nTraining",
            variant_name,
            "on outer fold",
            outer_fold
        )

        model_seed = (
            ERHT_SEED
            +
            outer_fold * 100
            +
            variant_index
        )

        (
            erht_model,
            training_history,
            source_event_map,
        ) = (
            erht_train(
                outer_train,
                variant_name,
                model_seed,
            )
        )

        training_history.to_csv(
            ERHT_TABLE
            /
            (
                f"training_history_"
                f"event_fold_{outer_fold}_"
                f"{variant_name}.csv"
            ),
            index=False
        )

        prediction = (
            erht_predict(
                erht_model,
                outer_test,
                request_override=
                    ERHT_VARIANTS[
                        variant_name
                    ][
                        "request_override"
                    ],
            )
        )

        result = (
            outer_test[
                [
                    "global_row_id",
                    "canonical_event",
                    "fold",
                    "label_11",
                    "label_triage",
                ]
            ]
            .copy()
        )

        result[
            "model"
        ] = (
            variant_name
        )

        result[
            "pred_11"
        ] = (
            erht_decode(
                prediction[
                    "pred_11"
                ],
                LABELS
            )
        )

        result[
            "pred_11_raw"
        ] = (
            erht_decode(
                prediction[
                    "pred_11_raw"
                ],
                LABELS
            )
        )

        result[
            "pred_triage"
        ] = (
            erht_decode(
                prediction[
                    "pred_triage"
                ],
                TRIAGE_LABELS
            )
        )

        result[
            "pred_triage_raw"
        ] = (
            erht_decode(
                prediction[
                    "pred_triage_raw"
                ],
                TRIAGE_LABELS
            )
        )

        result[
            "request_probability"
        ] = (
            prediction[
                "request_probability"
            ]
        )

        result.to_csv(
            checkpoint,
            index=False
        )

        erht_event_outputs.append(
            result
        )

        del erht_model
        del prediction

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


# ------------------------------------------------------------------
# Combine all folds
# ------------------------------------------------------------------

erht_event_predictions = (
    pd.concat(
        erht_event_outputs,
        ignore_index=True
    )
)

# Validate every variant separately.
for model_name in (
    ERHT_VARIANTS
    .keys()
):

    model_rows = (
        erht_event_predictions[
            erht_event_predictions[
                "model"
            ]
            .eq(
                model_name
            )
        ]
    )

    if (
        len(
            model_rows
        )
        !=
        len(
            strict_df
        )
    ):

        raise RuntimeError(
            f"{model_name}: expected "
            f"{len(strict_df)} OOF rows, "
            f"found {len(model_rows)}."
        )

    if (
        model_rows[
            "global_row_id"
        ]
        .duplicated()
        .any()
    ):

        raise RuntimeError(
            f"{model_name}: duplicate OOF rows found."
        )


erht_event_predictions.to_csv(
    ERHT_TABLE /
    "erht_event_disjoint_predictions.csv",
    index=False
)

# ------------------------------------------------------------------
# Overall and class-wise metrics
# ------------------------------------------------------------------

summary_rows = []
class_frames = []

for model_name, group in (
    erht_event_predictions
    .groupby(
        "model"
    )
):

    summary_rows.append(
        {
            "protocol":
                "event_disjoint_same_corpus",

            "task":
                "11class",

            "model":
                model_name,

            **erht_metrics(
                group[
                    "label_11"
                ],
                group[
                    "pred_11"
                ],
                LABELS
            ),
        }
    )

    class_frames.append(
        erht_per_class(
            group[
                "label_11"
            ],
            group[
                "pred_11"
            ],
            LABELS,
            model_name,
            "11class",
            "event_disjoint_same_corpus",
        )
    )

    summary_rows.append(
        {
            "protocol":
                "event_disjoint_same_corpus",

            "task":
                "direct_triage",

            "model":
                model_name,

            **erht_metrics(
                group[
                    "label_triage"
                ],
                group[
                    "pred_triage"
                ],
                TRIAGE_LABELS
            ),
        }
    )

    class_frames.append(
        erht_per_class(
            group[
                "label_triage"
            ],
            group[
                "pred_triage"
            ],
            TRIAGE_LABELS,
            model_name,
            "direct_triage",
            "event_disjoint_same_corpus",
        )
    )

    # Dedicated request-head evaluation.
    true_request = (
        group[
            "label_triage"
        ]
        .eq(
            "request"
        )
        .astype(int)
        .to_numpy()
    )

    predicted_request = (
        group[
            "request_probability"
        ]
        .to_numpy()
        >=
        ERHT_REQUEST_THRESHOLD
    ).astype(int)

    summary_rows.append(
        {
            "protocol":
                "event_disjoint_same_corpus",

            "task":
                "binary_request_head",

            "model":
                model_name,

            "n":
                int(
                    len(
                        group
                    )
                ),

            "accuracy":
                float(
                    accuracy_score(
                        true_request,
                        predicted_request
                    )
                ),

            "macro_f1":
                float(
                    f1_score(
                        true_request,
                        predicted_request,
                        average=
                            "macro",
                        zero_division=
                            0
                    )
                ),

            "balanced_accuracy":
                float(
                    balanced_accuracy_score(
                        true_request,
                        predicted_request
                    )
                ),

            "request_precision":
                float(
                    precision_score(
                        true_request,
                        predicted_request,
                        zero_division=
                            0
                    )
                ),

            "request_recall":
                float(
                    recall_score(
                        true_request,
                        predicted_request,
                        zero_division=
                            0
                    )
                ),

            "request_f1":
                float(
                    f1_score(
                        true_request,
                        predicted_request,
                        zero_division=
                            0
                    )
                ),
        }
    )


erht_event_summary = (
    pd.DataFrame(
        summary_rows
    )
)

erht_event_per_class = (
    pd.concat(
        class_frames,
        ignore_index=True
    )
)

erht_event_summary.to_csv(
    ERHT_TABLE /
    "erht_event_disjoint_summary.csv",
    index=False
)

erht_event_per_class.to_csv(
    ERHT_TABLE /
    "erht_event_disjoint_per_class.csv",
    index=False
)

print(
    "\nPRIMARY EVENT-DISJOINT RESULTS"
)

display(
    erht_event_summary.sort_values(
        [
            "task",
            "macro_f1"
        ],
        ascending=[
            True,
            False
        ]
    )
)

print(
    "\nEVENT-DISJOINT REQUEST CLASS"
)

display(
    erht_event_per_class[
        (
            erht_event_per_class[
                "task"
            ]
            ==
            "direct_triage"
        )
        &
        (
            erht_event_per_class[
                "label"
            ]
            ==
            "request"
        )
    ]
)


In [ ]:
# =====================================================================
# ERHT 5. Compare full ERHT with existing fine-tuned DistilBERT
# =====================================================================

distilbert_event_path = (
    Path(TABLE_DIR)
    /
    "distilbert_event_disjoint_oof_predictions.csv"
)

if not distilbert_event_path.is_file():

    candidates = list(
        Path(WORKDIR).rglob(
            "distilbert_event_disjoint_oof_predictions.csv"
        )
    )

    if not candidates:

        raise FileNotFoundError(
            "Could not find previous DistilBERT "
            "event-disjoint predictions."
        )

    distilbert_event_path = max(
        candidates,
        key=lambda p:
            p.stat().st_mtime
    )

distil = pd.read_csv(
    distilbert_event_path
)

distil = distil.rename(
    columns={
        "pred_11class__distilbert_ft":
            "distilbert_pred_11",

        "pred_direct_triage__distilbert_ft":
            "distilbert_pred_triage",
    }
)

full_erht = (
    erht_event_predictions[
        erht_event_predictions[
            "model"
        ]
        .eq(
            "erht_full"
        )
    ]
    .copy()
)

keep_distil = [
    c
    for c in [
        "global_row_id",
        "distilbert_pred_11",
        "distilbert_pred_triage",
    ]
    if c in distil.columns
]

comparison = (
    full_erht.merge(
        distil[
            keep_distil
        ],
        on=
            "global_row_id",
        how=
            "inner",
        validate=
            "one_to_one",
    )
)

if (
    len(
        comparison
    )
    !=
    len(
        full_erht
    )
):

    raise RuntimeError(
        "ERHT/DistilBERT comparison population mismatch."
    )


def erht_macro_f1_cm(
    cm
):

    cm = np.asarray(
        cm,
        dtype=float
    )

    tp = np.diag(
        cm
    )

    fp = (
        cm.sum(
            axis=0
        )
        -
        tp
    )

    fn = (
        cm.sum(
            axis=1
        )
        -
        tp
    )

    denominator = (
        2.0 * tp
        +
        fp
        +
        fn
    )

    class_f1 = np.divide(
        2.0 * tp,
        denominator,
        out=
            np.zeros_like(
                tp,
                dtype=float
            ),
        where=
            denominator > 0,
    )

    return float(
        class_f1.mean()
    )


def erht_event_bootstrap(
    frame,
    true_column,
    erht_column,
    comparator_column,
    labels,
    n_bootstrap,
    seed,
):

    events = (
        frame[
            "canonical_event"
        ]
        .astype(str)
        .unique()
    )

    erht_matrices = []
    comparator_matrices = []

    for event in events:

        subset = (
            frame[
                frame[
                    "canonical_event"
                ]
                .astype(str)
                .eq(
                    str(
                        event
                    )
                )
            ]
        )

        erht_matrices.append(
            confusion_matrix(
                subset[
                    true_column
                ],
                subset[
                    erht_column
                ],
                labels=
                    list(
                        labels
                    ),
            )
        )

        comparator_matrices.append(
            confusion_matrix(
                subset[
                    true_column
                ],
                subset[
                    comparator_column
                ],
                labels=
                    list(
                        labels
                    ),
            )
        )

    erht_matrices = np.asarray(
        erht_matrices,
        dtype=float
    )

    comparator_matrices = np.asarray(
        comparator_matrices,
        dtype=float
    )

    point = (
        erht_macro_f1_cm(
            erht_matrices.sum(
                axis=0
            )
        )
        -
        erht_macro_f1_cm(
            comparator_matrices.sum(
                axis=0
            )
        )
    )

    rng = (
        np.random.default_rng(
            seed
        )
    )

    differences = np.empty(
        n_bootstrap,
        dtype=float
    )

    n_events = len(
        events
    )

    for index in range(
        n_bootstrap
    ):

        selected = rng.integers(
            0,
            n_events,
            size=
                n_events
        )

        differences[
            index
        ] = (
            erht_macro_f1_cm(
                erht_matrices[
                    selected
                ]
                .sum(
                    axis=0
                )
            )
            -
            erht_macro_f1_cm(
                comparator_matrices[
                    selected
                ]
                .sum(
                    axis=0
                )
            )
        )

    ci_low, ci_high = np.percentile(
        differences,
        [
            2.5,
            97.5
        ]
    )

    left = (
        np.sum(
            differences <= 0
        )
        + 1.0
    ) / (
        n_bootstrap
        + 1.0
    )

    right = (
        np.sum(
            differences >= 0
        )
        + 1.0
    ) / (
        n_bootstrap
        + 1.0
    )

    p_two_sided = min(
        1.0,
        2.0
        *
        min(
            left,
            right
        )
    )

    return {
        "effect_erht_minus_distilbert":
            float(
                point
            ),

        "ci95_low":
            float(
                ci_low
            ),

        "ci95_high":
            float(
                ci_high
            ),

        "p_two_sided":
            float(
                p_two_sided
            ),

        "n_event_clusters":
            int(
                n_events
            ),

        "n_bootstrap":
            int(
                n_bootstrap
            ),
    }


effect_rows = []

if (
    "distilbert_pred_11"
    in comparison.columns
):

    effect_rows.append(
        {
            "protocol":
                "event_disjoint_same_corpus",

            "task":
                "11class",

            "model_a":
                "erht_full",

            "model_b":
                "distilbert_ft",

            **erht_event_bootstrap(
                comparison,
                "label_11",
                "pred_11",
                "distilbert_pred_11",
                LABELS,
                ERHT_N_BOOTSTRAP,
                ERHT_SEED + 901,
            ),
        }
    )


if (
    "distilbert_pred_triage"
    in comparison.columns
):

    effect_rows.append(
        {
            "protocol":
                "event_disjoint_same_corpus",

            "task":
                "direct_triage",

            "model_a":
                "erht_full",

            "model_b":
                "distilbert_ft",

            **erht_event_bootstrap(
                comparison,
                "label_triage",
                "pred_triage",
                "distilbert_pred_triage",
                TRIAGE_LABELS,
                ERHT_N_BOOTSTRAP,
                ERHT_SEED + 902,
            ),
        }
    )


erht_effects = (
    pd.DataFrame(
        effect_rows
    )
)

erht_effects.to_csv(
    ERHT_TABLE /
    "erht_vs_distilbert_event_cluster_bootstrap.csv",
    index=False
)

print(
    "\nERHT VS PREVIOUS DISTILBERT"
)

display(
    erht_effects
)

# ------------------------------------------------------------------
# Calculate comparator point estimates explicitly
# ------------------------------------------------------------------

distilbert_event_rows = []

if (
    "distilbert_pred_11"
    in comparison.columns
):

    distilbert_event_rows.append(
        {
            "task":
                "11class",

            "model":
                "distilbert_ft",

            **erht_metrics(
                comparison[
                    "label_11"
                ],
                comparison[
                    "distilbert_pred_11"
                ],
                LABELS
            ),
        }
    )

if (
    "distilbert_pred_triage"
    in comparison.columns
):

    distilbert_event_rows.append(
        {
            "task":
                "direct_triage",

            "model":
                "distilbert_ft",

            **erht_metrics(
                comparison[
                    "label_triage"
                ],
                comparison[
                    "distilbert_pred_triage"
                ],
                TRIAGE_LABELS
            ),
        }
    )

pd.DataFrame(
    distilbert_event_rows
).to_csv(
    ERHT_TABLE /
    "distilbert_comparator_recomputed.csv",
    index=False
)


In [ ]:
# =====================================================================
# ERHT 6. Secondary evaluation: official split and HumAID transfer
# =====================================================================

erht_fixed_summary = pd.DataFrame()
erht_humaid_summary = pd.DataFrame()

erht_source_model = None


# ------------------------------------------------------------------
# Official fixed split
# ------------------------------------------------------------------

if ERHT_RUN_FIXED_SPLIT:

    fixed_train = (
        combined[
            combined[
                "source_split"
            ]
            .astype(str)
            .eq(
                "train"
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    fixed_test = (
        combined[
            combined[
                "source_split"
            ]
            .astype(str)
            .eq(
                "test"
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    if (
        len(
            fixed_train
        )
        !=
        61164
        or
        len(
            fixed_test
        )
        !=
        17356
    ):

        warnings.warn(
            "Official split row counts differ from the "
            "previous prepared CrisisBench population."
        )

    # Event-discriminator labels for this secondary fit.
    for frame in [
        fixed_train,
        fixed_test
    ]:

        if (
            "canonical_event"
            not in frame.columns
        ):

            frame[
                "canonical_event"
            ] = (
                frame[
                    "event"
                ]
                .fillna(
                    "unknown"
                )
                .astype(str)
            )

    (
        erht_source_model,
        fixed_history,
        fixed_event_map,
    ) = (
        erht_train(
            fixed_train,
            "erht_full",
            ERHT_SEED
            + 5000,
        )
    )

    fixed_history.to_csv(
        ERHT_TABLE /
        "erht_fixed_training_history.csv",
        index=False
    )

    fixed_prediction = (
        erht_predict(
            erht_source_model,
            fixed_test,
            request_override=True,
        )
    )

    fixed_results = (
        fixed_test[
            [
                "global_row_id",
                "label_11",
                "label_triage",
            ]
        ]
        .copy()
    )

    fixed_results[
        "pred_11"
    ] = (
        erht_decode(
            fixed_prediction[
                "pred_11"
            ],
            LABELS
        )
    )

    fixed_results[
        "pred_triage"
    ] = (
        erht_decode(
            fixed_prediction[
                "pred_triage"
            ],
            TRIAGE_LABELS
        )
    )

    fixed_results[
        "request_probability"
    ] = (
        fixed_prediction[
            "request_probability"
        ]
    )

    fixed_results.to_csv(
        ERHT_TABLE /
        "erht_fixed_predictions.csv",
        index=False
    )

    erht_fixed_summary = (
        pd.DataFrame(
            [
                {
                    "protocol":
                        "official_fixed_split",

                    "task":
                        "11class",

                    "model":
                        "erht_full",

                    **erht_metrics(
                        fixed_results[
                            "label_11"
                        ],
                        fixed_results[
                            "pred_11"
                        ],
                        LABELS,
                    ),
                },

                {
                    "protocol":
                        "official_fixed_split",

                    "task":
                        "direct_triage",

                    "model":
                        "erht_full",

                    **erht_metrics(
                        fixed_results[
                            "label_triage"
                        ],
                        fixed_results[
                            "pred_triage"
                        ],
                        TRIAGE_LABELS,
                    ),
                },
            ]
        )
    )

    erht_fixed_summary.to_csv(
        ERHT_TABLE /
        "erht_fixed_summary.csv",
        index=False
    )

    # Save the final source-trained ERHT model.
    torch.save(
        {
            "state_dict":
                erht_source_model
                .state_dict(),

            "source_event_count":
                len(
                    fixed_event_map
                ),

            "config":
                ERHT_CONFIG,
        },
        ERHT_MODEL_DIR /
        "erht_full_fixed_state.pt",
    )

    erht_tokenizer.save_pretrained(
        ERHT_MODEL_DIR /
        "tokenizer"
    )

    print(
        "\nERHT FIXED-SPLIT RESULTS"
    )

    display(
        erht_fixed_summary
    )


# ------------------------------------------------------------------
# HumAID external transfer
# ------------------------------------------------------------------

if (
    ERHT_RUN_HUMAID
    and
    erht_source_model
    is not None
):

    humaid_candidates = [
        Path(TABLE_DIR)
        /
        "humaid_external_predictions.csv"
    ]

    humaid_candidates.extend(
        list(
            Path(WORKDIR).rglob(
                "humaid_external_predictions.csv"
            )
        )
    )

    HUM = next(
        (
            p
            for p
            in humaid_candidates
            if p.is_file()
        ),
        None
    )

    if HUM is None:

        print(
            "HumAID prepared prediction table was not found; "
            "external ERHT evaluation skipped."
        )

    else:

        print(
            "Using HumAID table:",
            HUM
        )

        huma = (
            pd.read_csv(
                HUM
            )
        )

        if (
            "clean_text"
            not in huma.columns
        ):

            if (
                "raw_text"
                in huma.columns
            ):

                # No new aggressive preprocessing.
                huma[
                    "clean_text"
                ] = (
                    huma[
                        "raw_text"
                    ]
                    .fillna("")
                    .astype(str)
                )

            else:

                raise KeyError(
                    "HumAID table contains neither clean_text nor raw_text."
                )

        if (
            "global_row_id"
            not in huma.columns
        ):

            huma[
                "global_row_id"
            ] = np.arange(
                len(
                    huma
                ),
                dtype=np.int64
            )

        hum_prediction = (
            erht_predict(
                erht_source_model,
                huma,
                request_override=True,
            )
        )

        huma[
            "pred_11__erht_full"
        ] = (
            erht_decode(
                hum_prediction[
                    "pred_11"
                ],
                LABELS
            )
        )

        huma[
            "pred_triage__erht_full"
        ] = (
            erht_decode(
                hum_prediction[
                    "pred_triage"
                ],
                TRIAGE_LABELS
            )
        )

        huma[
            "request_probability__erht_full"
        ] = (
            hum_prediction[
                "request_probability"
            ]
        )

        external_rows = []

        if (
            "label_triage"
            in huma.columns
        ):

            external_rows.append(
                {
                    "dataset":
                        "HumAID",

                    "task":
                        "triage_3class",

                    "model":
                        "erht_full",

                    **erht_metrics(
                        huma[
                            "label_triage"
                        ],
                        huma[
                            "pred_triage__erht_full"
                        ],
                        TRIAGE_LABELS,
                    ),
                }
            )

        if (
            "label_harmonized"
            in huma.columns
        ):

            shared_labels = [
                label
                for label in LABELS
                if label in set(
                    huma[
                        "label_harmonized"
                    ]
                    .astype(str)
                )
            ]

            shared = (
                huma[
                    huma[
                        "label_harmonized"
                    ]
                    .astype(str)
                    .isin(
                        shared_labels
                    )
                ]
                .copy()
            )

            if (
                len(
                    shared_labels
                )
                >= 2
                and
                len(
                    shared
                )
                > 0
            ):

                shared_metrics = (
                    erht_metrics(
                        shared[
                            "label_harmonized"
                        ],
                        shared[
                            "pred_11__erht_full"
                        ],
                        shared_labels,
                    )
                )

                shared_metrics[
                    "unsupported_prediction_rate"
                ] = float(
                    (
                        ~shared[
                            "pred_11__erht_full"
                        ]
                        .isin(
                            shared_labels
                        )
                    ).mean()
                )

                external_rows.append(
                    {
                        "dataset":
                            "HumAID",

                        "task":
                            (
                                "shared_"
                                f"{len(shared_labels)}class"
                            ),

                        "model":
                            "erht_full",

                        **shared_metrics,
                    }
                )

        erht_humaid_summary = (
            pd.DataFrame(
                external_rows
            )
        )

        erht_humaid_summary.to_csv(
            ERHT_TABLE /
            "erht_humaid_external_summary.csv",
            index=False
        )

        huma.to_csv(
            ERHT_TABLE /
            "erht_humaid_external_predictions.csv",
            index=False
        )

        print(
            "\nERHT HUMAID RESULTS"
        )

        display(
            erht_humaid_summary
        )


In [ ]:
# =====================================================================
# ERHT 7. Final evidence check and reproducibility package
# =====================================================================

def erht_summary_value(
    task,
    model,
    column,
):

    subset = (
        erht_event_summary[
            (
                erht_event_summary[
                    "task"
                ]
                ==
                task
            )
            &
            (
                erht_event_summary[
                    "model"
                ]
                ==
                model
            )
        ]
    )

    if subset.empty:
        return np.nan

    return float(
        subset.iloc[0][
            column
        ]
    )


# ------------------------------------------------------------------
# Full ERHT primary metric
# ------------------------------------------------------------------

full_event_f1 = (
    erht_summary_value(
        "11class",
        "erht_full",
        "macro_f1",
    )
)

# ------------------------------------------------------------------
# Previous DistilBERT comparator
# ------------------------------------------------------------------

if (
    "distilbert_pred_11"
    in comparison.columns
):

    distil_event_f1 = float(
        f1_score(
            comparison[
                "label_11"
            ],
            comparison[
                "distilbert_pred_11"
            ],
            labels=
                LABELS,
            average=
                "macro",
            zero_division=
                0,
        )
    )

else:

    distil_event_f1 = np.nan


# ------------------------------------------------------------------
# Request recall
# ------------------------------------------------------------------

request_result = (
    erht_event_per_class[
        (
            erht_event_per_class[
                "task"
            ]
            ==
            "direct_triage"
        )
        &
        (
            erht_event_per_class[
                "model"
            ]
            ==
            "erht_full"
        )
        &
        (
            erht_event_per_class[
                "label"
            ]
            ==
            "request"
        )
    ]
)

if request_result.empty:

    full_request_recall = (
        np.nan
    )

else:

    full_request_recall = (
        float(
            request_result
            .iloc[0][
                "recall"
            ]
        )
    )


# ------------------------------------------------------------------
# Evidence table
# ------------------------------------------------------------------

evidence_rows = [
    {
        "criterion":
            (
                "ERHT event-disjoint 11-class "
                "macro-F1 > DistilBERT"
            ),

        "erht_value":
            full_event_f1,

        "comparator_value":
            distil_event_f1,

        "pass":
            bool(
                np.isfinite(
                    full_event_f1
                )
                and
                np.isfinite(
                    distil_event_f1
                )
                and
                full_event_f1
                >
                distil_event_f1
            ),
    },

    {
        "criterion":
            (
                "ERHT event-disjoint "
                "request recall > 0"
            ),

        "erht_value":
            full_request_recall,

        "comparator_value":
            0.0,

        "pass":
            bool(
                np.isfinite(
                    full_request_recall
                )
                and
                full_request_recall
                >
                0.0
            ),
    },
]


for ablation in [
    "erht_no_event",
    "erht_no_hierarchy",
    "erht_no_request",
]:

    if ablation in set(
        erht_event_summary[
            "model"
        ]
    ):

        ablation_f1 = (
            erht_summary_value(
                "11class",
                ablation,
                "macro_f1"
            )
        )

        evidence_rows.append(
            {
                "criterion":
                    (
                        "ERHT full event-disjoint "
                        "11-class macro-F1 > "
                        + ablation
                    ),

                "erht_value":
                    full_event_f1,

                "comparator_value":
                    ablation_f1,

                "pass":
                    bool(
                        np.isfinite(
                            full_event_f1
                        )
                        and
                        np.isfinite(
                            ablation_f1
                        )
                        and
                        full_event_f1
                        >
                        ablation_f1
                    ),
            }
        )


evidence_rows.append(
    {
        "criterion":
            (
                "Frozen two-rater "
                "human audit retained"
            ),

        "erht_value":
            int(
                erht_audit
                .iloc[0][
                    "n_audited"
                ]
            ),

        "comparator_value":
            200,

        "pass":
            bool(
                int(
                    erht_audit
                    .iloc[0][
                        "n_audited"
                    ]
                )
                ==
                200
            ),
    }
)


erht_evidence = (
    pd.DataFrame(
        evidence_rows
    )
)

erht_evidence.to_csv(
    ERHT_TABLE /
    "erht_novelty_evidence_check.csv",
    index=False
)

print(
    "\nFINAL ERHT EVIDENCE CHECK"
)

display(
    erht_evidence
)


# ------------------------------------------------------------------
# Important interpretation file
# ------------------------------------------------------------------

claim_scope = {
    "model_name":
        "VIRCON-ERHT",

    "primary_endpoint":
        (
            "Event-disjoint "
            "11-class macro-F1"
        ),

    "secondary_request_endpoint":
        (
            "Event-disjoint "
            "request recall/F1"
        ),

    "components_not_claimed_individually_novel": [
        "DistilBERT",
        "multi-task learning",
        "hierarchical consistency",
        "focal loss",
        "supervised contrastive learning",
        "domain-adversarial gradient reversal",
    ],

    "candidate_combined_contribution":
        (
            "A joint crisis-text architecture combining "
            "cross-event representation alignment, "
            "event-adversarial learning, hierarchical "
            "humanitarian-triage consistency, and a dedicated "
            "rare-request recovery objective under a strictly "
            "event-held-out evaluation protocol."
        ),

    "human_audit_used_for_training":
        False,

    "claim_rule":
        (
            "Do not state that ERHT is superior unless the "
            "actual event-disjoint results and confidence "
            "intervals support that statement."
        ),
}

(
    ERHT_REPORT /
    "claim_scope.json"
).write_text(
    json.dumps(
        claim_scope,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------------
# Environment
# ------------------------------------------------------------------

environment = {
    "python":
        sys.version,

    "platform":
        platform.platform(),

    "numpy":
        np.__version__,

    "pandas":
        pd.__version__,

    "torch":
        torch.__version__,

    "cuda_available":
        torch.cuda.is_available(),

    "gpu":
        (
            torch.cuda.get_device_name(
                0
            )
            if
            torch.cuda.is_available()
            else
            None
        ),
}

(
    ERHT_REPORT /
    "environment.json"
).write_text(
    json.dumps(
        environment,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------------
# README
# ------------------------------------------------------------------

README = """
# VIRCON-ERHT

Primary evaluation:
same-corpus event-disjoint 11-class macro-F1.

Secondary endpoint:
request recall under event-disjoint direct triage.

Human audit:
The frozen 200-message human audit is independent validation only.
It is not used for model fitting, hyperparameter selection, threshold
selection, or model selection.

Interpretation:
The model should be presented as a candidate methodological contribution
only when the full ERHT ablation and event-cluster uncertainty analysis
support the associated claims.
""".strip()

(
    ERHT_ROOT /
    "README_ERHT.md"
).write_text(
    README,
    encoding="utf-8"
)


# ------------------------------------------------------------------
# Manifest
# ------------------------------------------------------------------

def erht_sha256(
    path
):

    digest = (
        hashlib.sha256()
    )

    with open(
        path,
        "rb"
    ) as file_handle:

        for chunk in iter(
            lambda:
                file_handle.read(
                    1024 * 1024
                ),
            b""
        ):

            digest.update(
                chunk
            )

    return (
        digest.hexdigest()
    )


manifest_rows = []

for path in sorted(
    ERHT_ROOT.rglob("*")
):

    if (
        not path.is_file()
    ):
        continue

    manifest_rows.append(
        {
            "relative_path":
                str(
                    path.relative_to(
                        ERHT_ROOT
                    )
                ),

            "bytes":
                int(
                    path.stat()
                    .st_size
                ),

            "sha256":
                erht_sha256(
                    path
                ),
        }
    )

manifest = pd.DataFrame(
    manifest_rows
)

manifest.to_csv(
    ERHT_ROOT /
    "artifact_manifest_sha256.csv",
    index=False
)


# ------------------------------------------------------------------
# ZIP package
# ------------------------------------------------------------------

ERHT_FINAL_ZIP = (
    Path(WORKDIR)
    /
    "VIRCON_ERHT_Final_Artifacts.zip"
)

with zipfile.ZipFile(
    ERHT_FINAL_ZIP,
    "w",
    compression=
        zipfile.ZIP_DEFLATED,
    allowZip64=True,
) as archive:

    for path in sorted(
        ERHT_ROOT.rglob("*")
    ):

        if (
            not path.is_file()
        ):
            continue

        archive.write(
            path,
            arcname=
                str(
                    Path(
                        ERHT_ROOT.name
                    )
                    /
                    path.relative_to(
                        ERHT_ROOT
                    )
                ),
        )

    notebook_path = (
        Path(WORKDIR)
        /
        "VIRCON_ERHT_Final_Model.ipynb"
    )

    if (
        notebook_path
        .is_file()
    ):

        archive.write(
            notebook_path,
            arcname=
                notebook_path.name,
        )


print(
    "\n"
    + "=" * 72
)

print(
    "VIRCON-ERHT EXPERIMENT COMPLETE"
)

print(
    "=" * 72
)

print(
    "\nResults folder:"
)

print(
    ERHT_ROOT
)

print(
    "\nFinal ZIP:"
)

print(
    ERHT_FINAL_ZIP
)

print(
    "\nUPLOAD THESE FILES TO CHAT:"
)

print(
    ERHT_TABLE /
    "erht_event_disjoint_summary.csv"
)

print(
    ERHT_TABLE /
    "erht_event_disjoint_per_class.csv"
)

print(
    ERHT_TABLE /
    "erht_vs_distilbert_event_cluster_bootstrap.csv"
)

print(
    ERHT_TABLE /
    "erht_novelty_evidence_check.csv"
)

if (
    ERHT_TABLE /
    "erht_humaid_external_summary.csv"
).is_file():

    print(
        ERHT_TABLE /
        "erht_humaid_external_summary.csv"
    )
